# Redrob Ranker — Colab Sandbox

**Intelligent Candidate Discovery & Ranking Challenge.** This notebook runs the combined
**Option A (structured) + Option B (semantic)** ranker end-to-end on a small candidate sample
and produces a ranked `submission.csv`.

- **No setup, no internet, no GPU.** The default semantic layer is a built-in TF-IDF scorer (pure Python standard library).
- **How to use:** `Runtime -> Run all`. It writes the code, loads the 50-candidate sample, ranks it, and shows the result.
- Satisfies submission_spec Section 10.5 (accepts a small sample, runs end-to-end, produces a ranked CSV, well under the 5-min CPU budget).



## Step 1 - Write the ranker source files
Each cell below saves one module to disk (identical to the GitHub repo).


In [ ]:
%%writefile config.py
"""
config.py — All tunable constants, keyword lists, and scoring weights.
Approach A: Structured Feature Ranker for Redrob Hackathon.

Modify weights here to tune ranking without touching feature logic.
"""

import re

# ═══════════════════════════════════════════════════════════════════════════════
# 1. JD-RELEVANT SKILL KEYWORDS (tiered by relevance to the Senior AI Engineer JD)
# ═══════════════════════════════════════════════════════════════════════════════

# Tier 1 — Core skills the JD explicitly asks for (highest weight)
TIER1_SKILLS = {
    "embeddings", "embedding", "sentence-transformers", "sentence transformers",
    "retrieval", "information retrieval", "hybrid search", "hybrid retrieval",
    "vector database", "vector db", "vector search",
    "ranking", "learning to rank", "learning-to-rank",
    "recommendation", "recommendation system", "recommender system", "recsys",
    "search", "search engine", "search system",
    "faiss", "pinecone", "weaviate", "qdrant", "milvus", "opensearch",
    "elasticsearch", "elastic search",
    "ndcg", "mrr", "map", "evaluation framework", "a/b testing", "ab testing",
    "bge", "e5", "openai embeddings",
    "ranking system", "retrieval system", "matching system",
}

# Tier 2 — Strong supporting skills (good signal)
TIER2_SKILLS = {
    "nlp", "natural language processing",
    "llm", "large language model", "large language models",
    "fine-tuning", "fine tuning", "finetuning", "lora", "qlora", "peft",
    "transformer", "transformers", "bert", "gpt",
    "pytorch", "tensorflow", "keras",
    "rag", "retrieval augmented generation", "retrieval-augmented generation",
    "langchain", "llamaindex", "llama index",
    "huggingface", "hugging face",
    "text classification", "text mining", "named entity recognition", "ner",
    "sentiment analysis", "topic modeling",
    "xgboost", "lightgbm", "catboost",
    "feature engineering", "feature store",
}

# Tier 3 — Supporting / general engineering (weak signal alone)
TIER3_SKILLS = {
    "python", "sql", "java", "scala", "go",
    "machine learning", "ml", "deep learning", "dl",
    "data pipeline", "data pipelines", "etl", "data engineering",
    "spark", "pyspark", "kafka", "airflow", "apache beam",
    "docker", "kubernetes", "k8s",
    "aws", "gcp", "azure", "cloud",
    "mlflow", "weights & biases", "wandb", "mlops",
    "ci/cd", "git", "linux",
    "distributed systems", "microservices",
    "api", "rest api", "grpc", "fastapi", "flask",
    "statistical modeling", "statistics",
    "data science", "data analysis",
    "bentoml", "triton", "model serving",
}

# Combined set for quick lookup
ALL_JD_SKILLS = TIER1_SKILLS | TIER2_SKILLS | TIER3_SKILLS

# Tier weights for trust-weighted skill scoring
SKILL_TIER_WEIGHT = {
    "tier1": 3.0,
    "tier2": 2.0,
    "tier3": 1.0,
    "none": 0.0,   # skill not relevant to JD
}

# Proficiency weights
PROFICIENCY_WEIGHT = {
    "expert": 1.0,
    "advanced": 0.8,
    "intermediate": 0.5,
    "beginner": 0.2,
}

# ═══════════════════════════════════════════════════════════════════════════════
# 2. TITLE PATTERNS
# ═══════════════════════════════════════════════════════════════════════════════

# Tech / ML / AI / Data titles (regex patterns, case-insensitive)
TECH_TITLE_PATTERNS = [
    r"\b(ml|machine\s*learning)\s*(engineer|developer|scientist|lead|architect|specialist)\b",
    r"\b(ai|artificial\s*intelligence)\s*(engineer|developer|scientist|lead|architect|specialist)\b",
    r"\bdata\s*scientist\b",
    r"\bnlp\s*(engineer|scientist|developer|specialist|researcher)\b",
    r"\b(senior|staff|principal|lead)?\s*(software|backend|fullstack|full[\s-]?stack)\s*(engineer|developer)\b",
    r"\bdata\s*engineer\b",
    r"\bdata\s*analyst\b",
    r"\bresearch\s*(engineer|scientist)\b",
    r"\bplatform\s*engineer\b",
    r"\bdevops\s*engineer\b",
    r"\bsite\s*reliability\s*engineer\b",
    r"\bsoftware\s*(engineer|developer|architect)\b",
    r"\bbackend\s*(engineer|developer)\b",
    r"\bfrontend\s*(engineer|developer)\b",
    r"\bfull[\s-]?stack\s*(engineer|developer)\b",
    r"\bcloud\s*(engineer|architect)\b",
    r"\binfrastructure\s*engineer\b",
    r"\b(deep\s*learning|dl)\s*(engineer|scientist|researcher)\b",
    r"\bcomputer\s*vision\s*(engineer|scientist|researcher)\b",
    r"\bspeech\s*(engineer|scientist|researcher)\b",
    r"\brobotics\s*(engineer|scientist|researcher)\b",
    r"\bapplied\s*scientist\b",
    r"\bquantitative\s*(analyst|developer|researcher)\b",
    r"\bcto\b",
    r"\bvp\s*(of\s*)?(engineering|technology)\b",
    r"\btech\s*lead\b",
    r"\bengineering\s*(manager|lead|director)\b",
]

# Compiled for speed (done once at import)
TECH_TITLE_RE = [re.compile(p, re.IGNORECASE) for p in TECH_TITLE_PATTERNS]

# High-value ML/AI title patterns (subset for extra boosting)
ML_AI_TITLE_PATTERNS = [
    r"\b(ml|machine\s*learning)\s*(engineer|developer|scientist|lead|architect|specialist)\b",
    r"\b(ai|artificial\s*intelligence)\s*(engineer|developer|scientist|lead|architect|specialist)\b",
    r"\bdata\s*scientist\b",
    r"\bnlp\s*(engineer|scientist|developer|specialist|researcher)\b",
    r"\b(deep\s*learning|dl)\s*(engineer|scientist|researcher)\b",
    r"\bapplied\s*scientist\b",
]

ML_AI_TITLE_RE = [re.compile(p, re.IGNORECASE) for p in ML_AI_TITLE_PATTERNS]

# Non-tech titles (keyword-stuffer signals)
NON_TECH_TITLES = {
    "hr manager", "human resources manager", "hr executive", "hr specialist",
    "content writer", "copywriter", "technical writer",
    "marketing manager", "digital marketing", "marketing executive", "marketing specialist",
    "accountant", "accounting manager", "finance manager", "financial analyst",
    "sales executive", "sales manager", "business development",
    "graphic designer", "ui designer", "visual designer",
    "mechanical engineer", "civil engineer", "electrical engineer", "chemical engineer",
    "customer support", "customer service", "support executive", "support specialist",
    "operations manager", "operations executive", "operations analyst",
    "project manager", "program manager", "scrum master",
    "teacher", "professor", "lecturer",
    "lawyer", "legal", "advocate",
    "doctor", "physician", "nurse",
    "architect",  # building architect, not software
    "supply chain", "logistics",
    "quality analyst", "quality manager",  # non-software QA
    "recruiter", "talent acquisition",
    "admin", "executive assistant", "office manager",
}

# ═══════════════════════════════════════════════════════════════════════════════
# 3. CAREER EVIDENCE PATTERNS
# ═══════════════════════════════════════════════════════════════════════════════

# Patterns that indicate the candidate actually BUILT ranking/retrieval/recsys
CAREER_EVIDENCE_PATTERNS = [
    r"\b(built|shipped|designed|implemented|developed|created|architected|deployed|owned|led)\b"
    r"[^.]{0,80}"
    r"\b(ranking|recommendation|retrieval|search|matching|recsys|recommender|re-ranking|re ranking|"
    r"candidate[\s-]?scoring|relevance|information[\s-]?retrieval|query[\s-]?understanding|"
    r"vector[\s-]?search|hybrid[\s-]?search|embedding[\s-]?based|semantic[\s-]?search)\b",

    r"\b(ranking|recommendation|retrieval|search|matching|recsys|recommender)\s*(system|engine|pipeline|model|service|platform|infrastructure)\b",

    r"\b(embeddings?|vector[\s-]?db|vector[\s-]?database|faiss|pinecone|weaviate|qdrant|milvus|elasticsearch|opensearch)\b"
    r"[^.]{0,60}"
    r"\b(production|deployed|served|real[\s-]?users|scale|million|billion)\b",

    r"\b(ndcg|mrr|map|precision@|recall@|a/b\s*test|offline[\s-]?eval|online[\s-]?eval)\b",

    r"\bend[\s-]?to[\s-]?end\b[^.]{0,60}\b(ml|ai|machine\s*learning|model|pipeline)\b",
]

CAREER_EVIDENCE_RE = [re.compile(p, re.IGNORECASE) for p in CAREER_EVIDENCE_PATTERNS]

# Production / deployment language in career descriptions
PRODUCTION_PATTERNS = [
    r"\b(production|deployed|shipped|launched|served|real[\s-]?users|real[\s-]?traffic|live|scale|scaled|million|billion)\b",
    r"\b(end[\s-]?to[\s-]?end|full[\s-]?stack|infra|infrastructure|pipeline)\b",
    r"\b(monitoring|alerting|on[\s-]?call|sla|latency|throughput|reliability)\b",
]

PRODUCTION_RE = [re.compile(p, re.IGNORECASE) for p in PRODUCTION_PATTERNS]

# ═══════════════════════════════════════════════════════════════════════════════
# 4. COMPANY CLASSIFICATION
# ═══════════════════════════════════════════════════════════════════════════════

# Major consulting / services firms (JD explicitly flags these)
CONSULTING_FIRMS = {
    "tcs", "tata consultancy", "tata consultancy services",
    "infosys",
    "wipro",
    "accenture",
    "cognizant", "cognizant technology",
    "capgemini",
    "hcl", "hcl technologies",
    "mindtree",
    "tech mahindra",
    "lti", "l&t infotech", "ltimindtree",
    "mphasis",
    "hexaware",
    "persistent systems",
    "zensar",
    "cyient",
    "niit", "niit technologies",
    "virtusa",
    "birlasoft",
    "coforge",
    "sonata software",
    "mastek",
    "happiest minds",
    "kpit",
    "deloitte",
    "ey", "ernst & young", "ernst and young",
    "kpmg",
    "pwc", "pricewaterhousecoopers",
    "ibm", "ibm consulting",
    "dxc technology",
    "atos",
    "ntt data",
}

# Industries that suggest product companies
PRODUCT_INDUSTRIES = {
    "internet", "software", "saas", "technology", "fintech",
    "e-commerce", "ecommerce", "marketplace",
    "gaming", "media", "entertainment",
    "healthtech", "edtech", "agritech",
    "automotive", "electric vehicles",
    "telecommunications",
    "financial services", "banking",
    "cloud computing",
    "cybersecurity",
    "artificial intelligence",
}

# ═══════════════════════════════════════════════════════════════════════════════
# 5. LOCATION DATA
# ═══════════════════════════════════════════════════════════════════════════════

# Priority locations from JD
TIER1_LOCATIONS = {"pune", "noida"}
TIER2_LOCATIONS = {"hyderabad", "mumbai", "delhi", "delhi ncr", "new delhi",
                   "gurgaon", "gurugram", "bangalore", "bengaluru", "chennai",
                   "kolkata"}
INDIA_KEYWORDS = {"india"}

# ═══════════════════════════════════════════════════════════════════════════════
# 6. CV / SPEECH / ROBOTICS — Specialist Penalty
# ═══════════════════════════════════════════════════════════════════════════════

CV_SPEECH_ROBOTICS_SKILLS = {
    "computer vision", "image classification", "object detection",
    "image segmentation", "yolo", "opencv", "image processing",
    "speech recognition", "speech synthesis", "tts", "text to speech",
    "asr", "automatic speech recognition", "voice",
    "robotics", "ros", "robot operating system", "slam",
    "gans", "generative adversarial", "image generation",
    "3d reconstruction", "point cloud", "lidar",
}

NLP_IR_SKILLS = {
    "nlp", "natural language processing", "information retrieval",
    "text classification", "named entity recognition", "ner",
    "sentiment analysis", "topic modeling", "text mining",
    "retrieval", "search", "ranking", "recommendation",
    "embeddings", "embedding", "sentence-transformers",
    "llm", "large language model", "transformer", "transformers",
    "bert", "gpt", "rag", "langchain",
    "elasticsearch", "opensearch", "faiss", "pinecone",
}

# ═══════════════════════════════════════════════════════════════════════════════
# 7. SCORING WEIGHTS (the main tuning knobs)
# ═══════════════════════════════════════════════════════════════════════════════

WEIGHTS = {
    # ── Positive features (sum = 1.00) ───────────────────────────────────────
    # Option A structured signals + the Option B semantic signal, fused.
    "title_career_fit":        0.26,   # A: decisive anti-stuffer (title + career evidence)
    "semantic_fit":            0.22,   # B: dense (embedding) + sparse (BM25) JD similarity  <- NEW
    "skill_trust":             0.15,   # A: trust-weighted skill overlap
    "experience_fit":          0.12,   # A: 6-8 yr band fit
    "product_company":         0.09,   # A: product vs services
    "career_evidence_bonus":   0.08,   # A: built ranking/retrieval/recsys
    "location_fit":            0.05,   # A: Pune/Noida > metros > relocatable
    "education":               0.03,   # A: light tiebreaker

    # ── Penalties (subtracted; can drive a stuffer's fit_score to 0) ─────────
    "consulting_only":         0.15,
    "research_only":           0.10,
    "cv_speech_only":          0.10,
    "keyword_stuffer":         0.25,
    "job_hopper":              0.05,
}

# ═══════════════════════════════════════════════════════════════════════════════
# 7b. SEMANTIC FUSION (Option B) — how the embedding layer blends in
# ═══════════════════════════════════════════════════════════════════════════════
# semantic_fit = DENSE_WEIGHT * norm(cosine(JD, candidate)) + SPARSE_WEIGHT * norm(BM25)
# Both components are min-max normalized across the full pool at rank time. The
# blended value is ONE additive feature inside the structured score above — never
# the ranker itself. Structured penalties + the honeypot gate still dominate, so
# semantics cannot pull keyword-stuffers or honeypots into the top ranks.
SEMANTIC_DENSE_WEIGHT = 0.65    # embedding cosine similarity
SEMANTIC_SPARSE_WEIGHT = 0.35   # BM25 lexical overlap
# If artifacts are missing, rank.py falls back to pure Option A (semantic_fit = 0).

# ═══════════════════════════════════════════════════════════════════════════════
# 8. EXPERIENCE BAND SCORING
# ═══════════════════════════════════════════════════════════════════════════════

def experience_score(years: float) -> float:
    """
    Score experience fit for the JD (ideal: 6-8 yrs).
    Returns float in [0, 1].
    """
    if 6.0 <= years <= 8.0:
        return 1.0
    elif 5.0 <= years < 6.0 or 8.0 < years <= 9.0:
        return 0.85
    elif 4.0 <= years < 5.0 or 9.0 < years <= 12.0:
        return 0.60
    elif 3.0 <= years < 4.0 or 12.0 < years <= 15.0:
        return 0.35
    elif years < 3.0:
        return 0.15
    else:  # > 15
        return 0.20

# ═══════════════════════════════════════════════════════════════════════════════
# 9. OUTPUT CONFIGURATION
# ═══════════════════════════════════════════════════════════════════════════════

TOP_K = 100  # Number of candidates to output
CSV_COLUMNS = ["candidate_id", "rank", "score", "reasoning"]


In [ ]:
%%writefile build_text.py
"""
Step 1 — build_text.py
======================
Converts a raw candidate JSON record into a single text string
for embedding. The text blob is what BAAI/bge-base-en-v1.5 will
encode, so what you put in here directly determines ranking quality.

Key design decisions (grounded in the actual JD):
  - Career history descriptions are weighted first and heaviest.
    The JD explicitly says "find Tier-5 candidates who don't use the
    keyword RAG but whose history shows they built a ranking system."
    That signal lives in descriptions, not in the skills list.

  - Skills list comes second, but only with proficiency + duration.
    A skill with 0 months duration is meaningless (and a honeypot signal).
    We skip skills whose duration_months is 0 — they pollute the embedding.

  - Summary and headline come last. They tend to be self-promotional
    and keyword-heavy, exactly the "trap" the JD warns about.

  - We do NOT include: location, salary, notice period, company size,
    or redrob_signals — those are structured features for the scoring
    formula in step 3, not semantic text for embedding.

Run this file directly to test on the sample candidate:
    python build_text.py
"""

import json
import re
from datetime import datetime, date


# --------------------------------------------------------------------------
# Constants — pulled from the actual JD
# --------------------------------------------------------------------------

# The JD says: "People who have only worked at consulting firms
# in their entire career" are disqualifiers.
CONSULTING_FIRMS = {
    "tcs", "infosys", "wipro", "accenture", "cognizant",
    "capgemini", "hcl", "tech mahindra", "mphasis", "hexaware",
    "ltimindtree", "mindtree",  # note: mindtree merged into LTIMindtree
}

# JD explicitly wants: embeddings, retrieval, ranking, LLMs, vector DBs,
# eval frameworks. We use these to weight skill mentions in the text.
HIGH_VALUE_SKILLS = {
    # Core must-haves from JD
    "embeddings", "vector search", "semantic search", "hybrid search",
    "dense retrieval", "sparse retrieval", "bm25",
    "sentence transformers", "faiss", "milvus", "qdrant", "weaviate",
    "pinecone", "opensearch", "elasticsearch",
    "ranking", "reranking", "learning to rank", "ltr",
    "ndcg", "mrr", "map", "information retrieval", "ir",
    "rag", "retrieval augmented generation",
    # Nice-to-haves from JD
    "lora", "qlora", "peft", "fine-tuning", "fine tuning",
    "llm", "transformer", "bert", "nlp",
    "recommendation system", "recommender",
    "xgboost", "lightgbm",
    # Strong signal skills from schema sample
    "pyspark", "spark", "airflow", "kafka", "dbt", "snowflake",
}


# --------------------------------------------------------------------------
# Skill matching helper
# --------------------------------------------------------------------------

# Pre-compile word-boundary patterns for each high-value skill
_HV_PATTERNS = {hv: re.compile(rf'\b{re.escape(hv)}\b') for hv in HIGH_VALUE_SKILLS}


def _is_high_value_skill(name_lower: str) -> bool:
    """
    Check if a skill name matches any HIGH_VALUE_SKILLS entry
    using word-boundary matching instead of substring `in`.
    Avoids false positives like 'ir' matching 'Airflow'.
    """
    if name_lower in HIGH_VALUE_SKILLS:
        return True
    return any(pat.search(name_lower) for pat in _HV_PATTERNS.values())


# --------------------------------------------------------------------------
# Main text builder
# --------------------------------------------------------------------------

def build_text(candidate: dict) -> str:
    """
    Convert a candidate record to a single embedding-ready text string.

    Returns a string roughly structured as:
        [career history, weighted by recency]
        [skills, filtered and weighted]
        [summary]
        [headline]

    The string uses a pipe '|' separator between logical sections
    so sentence-transformers sees clear boundaries.
    """
    parts = []

    profile = candidate.get("profile", {})
    career  = candidate.get("career_history", [])
    skills  = candidate.get("skills", [])

    # ------------------------------------------------------------------
    # 1. CAREER HISTORY — highest signal, put first
    # ------------------------------------------------------------------
    # Sort by is_current first, then by start_date descending (most recent first).
    # Recency matters: a 2-year-old role at a product company > a 10-year-old one.

    # Two-pass stable sort: first by start_date descending (most recent first),
    # then by is_current (current roles always on top).
    # Python's sort is stable, so the second sort preserves the date order.
    sorted_career = sorted(
        career,
        key=lambda j: j.get("start_date", "1970-01-01"),
        reverse=True,
    )
    sorted_career = sorted(
        sorted_career,
        key=lambda j: 0 if j.get("is_current") else 1,
    )

    for job in sorted_career:
        company    = job.get("company", "")
        title      = job.get("title", "")
        duration_m = job.get("duration_months", 0)
        duration_y = round(duration_m / 12, 1)
        description = job.get("description", "").strip()
        industry   = job.get("industry", "")
        is_current = job.get("is_current", False)

        # Skip entirely empty descriptions — they add noise
        if not description and not title:
            continue

        current_marker = " [current]" if is_current else ""
        role_line = (
            f"{title} at {company}{current_marker} "
            f"({duration_y}y, {industry}): {description}"
        )
        parts.append(role_line)

    # ------------------------------------------------------------------
    # 2. SKILLS — filtered and weighted
    # ------------------------------------------------------------------
    # Rules:
    #   - Skip skills with duration_months == 0 (honeypot signal + noise)
    #   - Skip beginner skills with < 6 months duration (low signal)
    #   - Repeat high-value skills to give them more weight in embedding
    #   - Include proficiency level — "advanced NLP" ≠ "beginner NLP"

    skill_parts_normal = []
    skill_parts_highval = []

    for sk in skills:
        name       = sk.get("name", "").strip()
        proficiency = sk.get("proficiency", "")
        duration_m  = sk.get("duration_months", 0)

        # Hard filter: duration=0 means never used → skip entirely
        if duration_m == 0:
            continue

        # Soft filter: beginner + very short exposure → skip
        if proficiency == "beginner" and duration_m < 6:
            continue

        duration_y = round(duration_m / 12, 1)
        skill_str  = f"{name} ({proficiency}, {duration_y}y)"

        # Check if this is a high-value skill for this JD
        # Uses word-boundary matching to avoid false positives
        name_lower = name.lower()
        is_highval = _is_high_value_skill(name_lower)

        if is_highval:
            skill_parts_highval.append(skill_str)
        else:
            skill_parts_normal.append(skill_str)

    # High-value skills go first and are repeated to boost their signal weight
    if skill_parts_highval:
        parts.append("Key skills: " + "; ".join(skill_parts_highval))
        # Repeat once to increase embedding weight (a simple but effective trick)
        parts.append("Core expertise: " + "; ".join(skill_parts_highval))

    if skill_parts_normal:
        parts.append("Also: " + "; ".join(skill_parts_normal))

    # ------------------------------------------------------------------
    # 3. SUMMARY — good signal but keyword-stuffable, so goes after career
    # ------------------------------------------------------------------
    summary = profile.get("summary", "").strip()
    if summary:
        parts.append(summary)

    # ------------------------------------------------------------------
    # 4. HEADLINE — weakest signal (one-liner, often keyword stuffed)
    # ------------------------------------------------------------------
    headline = profile.get("headline", "").strip()
    if headline:
        parts.append(headline)

    # ------------------------------------------------------------------
    # 5. Education field of study — subtle semantic signal
    # ------------------------------------------------------------------
    education = candidate.get("education", [])
    for edu in education:
        field = edu.get("field_of_study", "")
        degree = edu.get("degree", "")
        if field or degree:
            parts.append(f"{degree} in {field}")

    # Join with pipe separator — clean boundary between sections
    return " | ".join(filter(None, parts))


# --------------------------------------------------------------------------
# Consulting-only detector (used in penalty step, but computed here)
# --------------------------------------------------------------------------

def is_consulting_only(candidate: dict) -> bool:
    """
    Returns True if ALL career history is at known consulting firms.
    The JD says: "People who have only worked at consulting firms
    in their entire career" are disqualifiers.
    A candidate currently at a consulting firm but with prior
    product-company experience is explicitly still OK.
    """
    career = candidate.get("career_history", [])
    if not career:
        return False

    companies = {job.get("company", "").lower() for job in career}
    return all(
        any(firm in company for firm in CONSULTING_FIRMS)
        for company in companies
    )


# --------------------------------------------------------------------------
# Honeypot detection
# --------------------------------------------------------------------------

def detect_honeypot(candidate: dict) -> bool:
    """
    Flag candidates with subtly impossible profiles.
    The dataset contains ~80 honeypots. >10% in top-100 = disqualification.

    Red-flag heuristics:
      - Advanced/expert proficiency with near-zero duration
      - Career duration_months exceeds actual calendar span
      - Total years_of_experience exceeds career timeline
      - High endorsements on zero-duration skills
    """
    skills  = candidate.get("skills", [])
    career  = candidate.get("career_history", [])
    profile = candidate.get("profile", {})

    red_flags = 0

    # Check 1: Advanced/expert proficiency with near-zero duration (<= 2 months)
    expert_zero = sum(
        1 for s in skills
        if s.get("proficiency") in ("advanced", "expert")
        and s.get("duration_months", 0) <= 2
    )
    if expert_zero >= 2:
        red_flags += 2
    elif expert_zero >= 1:
        red_flags += 1

    # Check 2: Career duration_months significantly exceeds actual calendar time
    for job in career:
        start   = job.get("start_date")
        end     = job.get("end_date")
        claimed = job.get("duration_months", 0)
        if start and end and claimed > 0:
            try:
                d_start = datetime.fromisoformat(start)
                d_end   = datetime.fromisoformat(end)
                actual_months = (d_end - d_start).days / 30.44
                if actual_months > 0 and claimed > actual_months * 1.5:
                    red_flags += 2
            except (ValueError, TypeError):
                pass

    # Check 3: Total years_of_experience vs. career timeline
    claimed_years = profile.get("years_of_experience", 0)
    if career and claimed_years and claimed_years > 0:
        earliest = min(
            (j.get("start_date", "9999") for j in career),
            default="9999",
        )
        if earliest != "9999":
            try:
                d_earliest = datetime.fromisoformat(earliest).date()
                actual_years = (date.today() - d_earliest).days / 365.25
                if actual_years > 0 and claimed_years > actual_years * 1.5:
                    red_flags += 2
            except (ValueError, TypeError):
                pass

    # Check 4: Many skills with high endorsements but near-zero duration
    suspicious_skills = sum(
        1 for s in skills
        if s.get("endorsements", 0) > 20
        and s.get("duration_months", 0) <= 1
    )
    if suspicious_skills >= 2:
        red_flags += 1

    # Threshold: 2+ red flags = likely honeypot
    return red_flags >= 2


# --------------------------------------------------------------------------
# Disqualifier features (from JD explicit rules)
# --------------------------------------------------------------------------

def compute_disqualifiers(candidate: dict) -> dict:
    """
    Compute binary disqualifier flags derived from the JD's explicit rules.
    Returns dict of flag_name -> bool.
    """
    career  = candidate.get("career_history", [])
    skills  = candidate.get("skills", [])
    profile = candidate.get("profile", {})

    flags = {}

    # 1. Consulting-only career
    flags["consulting_only"] = is_consulting_only(candidate)

    # 2. Job-hopping pattern (avg tenure < 18 months)
    if career:
        tenures = [j.get("duration_months", 0) for j in career]
        avg_tenure = sum(tenures) / len(tenures)
        flags["job_hopper"] = avg_tenure < 18
    else:
        flags["job_hopper"] = False

    # 3. Pure research — academic/research signal with NO production-deployment evidence.
    #
    #    BUGFIX: the original production_kw list included generic engineering nouns
    #    ("system", "platform", "api", "pipeline", "backend", "infrastructure", "scale")
    #    that show up in almost every career description, academic or not. That made
    #    `not has_production` collapse to False for nearly every candidate, so this
    #    flag could never fire at all (verified empirically: 0 / 100,000 on the real
    #    candidate pool). research_kw had the mirror-image problem — bare "research",
    #    "scientist", and "lab" match common industry titles like "AI Research Engineer"
    #    or "Data Scientist" (and "lab" even matches inside "collaborate"), none of which
    #    are academic-only. Both keyword sets are tightened below to fix this.
    research_kw = {"researcher", "phd", "postdoc", "postdoctoral", "professor",
                   "academia", "doctoral", "dissertation", "thesis",
                   "research lab", "research fellow", "research scientist"}
    production_kw = {"production", "deploy", "deployed", "shipped", "launched",
                     "rolled out", "in production", "live users", "real users",
                     "end users", "users", "customers", "product", "service"}
    all_career_text = " ".join(
        (j.get("title", "") + " " + j.get("description", "")).lower()
        for j in career
    )
    has_research   = any(kw in all_career_text for kw in research_kw)
    has_production = any(kw in all_career_text for kw in production_kw)
    flags["pure_research"] = has_research and not has_production

    # 4. CV / Speech / Robotics without NLP/IR exposure
    cv_speech_kw = {"computer vision", "image classification", "object detection",
                    "speech recognition", "speech", "tts", "text to speech",
                    "robotics", "robot", "autonomous driving"}
    nlp_ir_kw = {"nlp", "natural language", "information retrieval", "search",
                 "ranking", "embeddings", "transformer", "bert", "llm",
                 "text mining", "text classification", "ner",
                 "sentiment", "rag", "retrieval"}
    skill_text = " ".join(s.get("name", "").lower() for s in skills)
    has_cv_speech = any(kw in skill_text for kw in cv_speech_kw)
    has_nlp_ir    = any(kw in skill_text for kw in nlp_ir_kw)
    # Also check career descriptions for NLP/IR work
    has_nlp_ir    = has_nlp_ir or any(kw in all_career_text for kw in nlp_ir_kw)
    flags["cv_speech_no_nlp"] = has_cv_speech and not has_nlp_ir

    # 5. AI experience = LangChain demos under 12 months
    langchain_kw = {"langchain", "llamaindex", "llama index"}
    deep_ai_kw   = {"machine learning", "deep learning", "tensorflow", "pytorch",
                    "scikit-learn", "xgboost", "lightgbm", "embeddings",
                    "transformers", "ranking", "recommendation", "faiss"}
    has_langchain = any(
        any(lc in s.get("name", "").lower() for lc in langchain_kw)
        and s.get("duration_months", 0) < 12
        for s in skills
    )
    has_real_ai = any(
        any(ai in s.get("name", "").lower() for ai in deep_ai_kw)
        and s.get("duration_months", 0) >= 12
        for s in skills
    )
    flags["langchain_only"] = has_langchain and not has_real_ai

    return flags


# --------------------------------------------------------------------------
# Engagement multiplier (from redrob_signals)
# --------------------------------------------------------------------------

def compute_engagement_score(candidate: dict) -> float:
    """
    Compute a 0–1 engagement multiplier from redrob_signals.
    Higher = more likely to be reachable and responsive.

    Components (equal weight):
      - Recency of last_active_date   (180-day decay)
      - recruiter_response_rate       (direct 0–1)
      - interview_completion_rate     (direct 0–1)
      - offer_acceptance_rate         (if available)
      - open_to_work_flag             (binary boost)
    """
    signals = candidate.get("redrob_signals", {})
    components = []

    # 1. Recency of last_active_date (0–1, recent = higher)
    last_active = signals.get("last_active_date")
    if last_active:
        try:
            la_date  = datetime.fromisoformat(last_active).date()
            days_ago = (date.today() - la_date).days
            recency  = max(0.0, 1.0 - days_ago / 180.0)
            components.append(recency)
        except (ValueError, TypeError):
            components.append(0.5)
    else:
        components.append(0.5)

    # 2. Recruiter response rate (direct 0–1)
    rrr = signals.get("recruiter_response_rate", -1)
    if isinstance(rrr, (int, float)) and rrr >= 0:
        components.append(float(rrr))
    else:
        components.append(0.5)

    # 3. Interview completion rate (direct 0–1)
    icr = signals.get("interview_completion_rate", -1)
    if isinstance(icr, (int, float)) and icr >= 0:
        components.append(float(icr))
    else:
        components.append(0.5)

    # 4. Offer acceptance rate (−1 sentinel = no prior offers → neutral)
    oar = signals.get("offer_acceptance_rate", -1)
    if isinstance(oar, (int, float)) and oar >= 0:
        components.append(float(oar))
    # else: don't penalise — -1 means no prior offers

    # 5. Open to work flag (binary boost)
    components.append(1.0 if signals.get("open_to_work_flag") else 0.3)

    return sum(components) / len(components) if components else 0.5


# --------------------------------------------------------------------------
# Quick smoke test — run: python build_text.py
# --------------------------------------------------------------------------

SAMPLE_CANDIDATE = {
    "candidate_id": "CAND_0000001",
    "profile": {
        "anonymized_name": "Ira Vora",
        "headline": "Backend Engineer | SQL, Spark, Cloud",
        "summary": (
            "Software / data professional with 6.9 years of experience building "
            "data pipelines, backend systems, and analytics infrastructure. "
            "I'm a backend/data hybrid — Spark, Airflow, SQL warehouses are home "
            "territory; I'm building competence on the ML side."
        ),
        "location": "Toronto",
        "country": "Canada",
        "years_of_experience": 6.9,
        "current_title": "Backend Engineer",
        "current_company": "Mindtree",
        "current_company_size": "10001+",
        "current_industry": "IT Services",
    },
    "career_history": [
        {
            "company": "Mindtree",
            "title": "Backend Engineer",
            "start_date": "2024-03-08",
            "end_date": None,
            "duration_months": 27,
            "is_current": True,
            "industry": "IT Services",
            "company_size": "10001+",
            "description": (
                "Implemented streaming data pipelines on Kafka and Spark Streaming "
                "for a real-time user-activity processing platform."
            ),
        },
        {
            "company": "Dunder Mifflin",
            "title": "Analytics Engineer",
            "start_date": "2019-07-03",
            "end_date": "2024-01-08",
            "duration_months": 55,
            "is_current": False,
            "industry": "Paper Products",
            "company_size": "201-500",
            "description": (
                "Built and maintained data pipelines on Apache Airflow processing "
                "~500GB of daily transactional data. Worked extensively with Spark "
                "(PySpark) for batch processing and dbt for the transformation layer."
            ),
        },
    ],
    "education": [
        {
            "institution": "Lovely Professional University",
            "degree": "B.E.",
            "field_of_study": "Computer Science",
            "start_year": 2017,
            "end_year": 2020,
            "grade": "8.24 CGPA",
            "tier": "tier_3",
        }
    ],
    "skills": [
        {"name": "NLP",            "proficiency": "advanced",      "endorsements": 37, "duration_months": 26},
        {"name": "Fine-tuning LLMs","proficiency": "advanced",     "endorsements": 21, "duration_months": 36},
        {"name": "Milvus",         "proficiency": "advanced",      "endorsements": 40, "duration_months": 35},
        {"name": "LoRA",           "proficiency": "intermediate",  "endorsements": 0,  "duration_months": 28},
        {"name": "Speech Recognition","proficiency": "advanced",   "endorsements": 52, "duration_months": 33},
        {"name": "TTS",            "proficiency": "advanced",      "endorsements": 56, "duration_months": 60},
        {"name": "AWS",            "proficiency": "beginner",      "endorsements": 5,  "duration_months": 8},
        {"name": "Flask",          "proficiency": "beginner",      "endorsements": 15, "duration_months": 15},
        {"name": "Tailwind",       "proficiency": "intermediate",  "endorsements": 3,  "duration_months": 13},
        {"name": "Photoshop",      "proficiency": "intermediate",  "endorsements": 8,  "duration_months": 24},
        # Honeypot-style skill: expert with 0 months
        {"name": "GCP",            "proficiency": "beginner",      "endorsements": 7,  "duration_months": 2},
    ],
    "certifications": [],
    "languages": [
        {"language": "English", "proficiency": "professional"},
    ],
    "redrob_signals": {
        "profile_completeness_score": 86.9,
        "last_active_date": "2026-05-20",
        "open_to_work_flag": True,
        "recruiter_response_rate": 0.34,
        "interview_completion_rate": 0.71,
        "offer_acceptance_rate": 0.58,
        "notice_period_days": 60,
        "github_activity_score": 9.2,
        "willing_to_relocate": False,
        "verified_email": True,
        "verified_phone": True,
        "linkedin_connected": False,
    },
}


if __name__ == "__main__":
    text = build_text(SAMPLE_CANDIDATE)

    print("=" * 70)
    print("CANDIDATE:", SAMPLE_CANDIDATE["candidate_id"])
    print("=" * 70)
    print(text)
    print()
    print(f"Total characters : {len(text)}")
    print(f"Total words      : {len(text.split())}")
    print(f"Consulting-only? : {is_consulting_only(SAMPLE_CANDIDATE)}")
    print(f"Honeypot?        : {detect_honeypot(SAMPLE_CANDIDATE)}")
    print(f"Engagement       : {compute_engagement_score(SAMPLE_CANDIDATE):.3f}")
    print(f"Disqualifiers    : {compute_disqualifiers(SAMPLE_CANDIDATE)}")
    print()

    # Show what the HIGH_VALUE_SKILLS filter caught
    caught = [
        sk["name"] for sk in SAMPLE_CANDIDATE["skills"]
        if any(hv in sk["name"].lower() for hv in HIGH_VALUE_SKILLS)
        and sk.get("duration_months", 0) > 0
    ]
    print(f"High-value skills detected: {caught}")

    # Show what was filtered out
    filtered = [
        sk["name"] for sk in SAMPLE_CANDIDATE["skills"]
        if sk.get("duration_months", 0) == 0
    ]
    if filtered:
        print(f"Filtered (duration=0)     : {filtered}")


In [ ]:
%%writefile features.py
"""
features.py — All feature extraction and scoring functions.
Approach A: Structured Feature Ranker for Redrob Hackathon.

Each function takes a candidate dict and returns a float score.
All scores are normalized to [0, 1] (or [0.5, 1.15] for avail_mod).
"""

import math
import re
from datetime import datetime, date

from config import (
    TIER1_SKILLS, TIER2_SKILLS, TIER3_SKILLS, ALL_JD_SKILLS,
    SKILL_TIER_WEIGHT, PROFICIENCY_WEIGHT,
    TECH_TITLE_RE, ML_AI_TITLE_RE, NON_TECH_TITLES,
    CAREER_EVIDENCE_RE, PRODUCTION_RE,
    CONSULTING_FIRMS,
    PRODUCT_INDUSTRIES,
    TIER1_LOCATIONS, TIER2_LOCATIONS, INDIA_KEYWORDS,
    CV_SPEECH_ROBOTICS_SKILLS, NLP_IR_SKILLS,
    experience_score,
)

# Reference date for "days since last active" — use a fixed date for reproducibility
REFERENCE_DATE = date(2026, 6, 25)


def _normalize(text: str) -> str:
    """Lowercase and strip a string for matching."""
    return (text or "").strip().lower()


def _is_consulting(company_name: str) -> bool:
    """Check if a company name matches the consulting firms list."""
    name = _normalize(company_name)
    for firm in CONSULTING_FIRMS:
        if firm in name:
            return True
    return False


def _is_tech_title(title: str) -> bool:
    """Check if title matches any tech/engineering pattern."""
    for pattern in TECH_TITLE_RE:
        if pattern.search(title):
            return True
    return False


def _is_ml_ai_title(title: str) -> bool:
    """Check if title matches high-value ML/AI patterns."""
    for pattern in ML_AI_TITLE_RE:
        if pattern.search(title):
            return True
    return False


def _is_non_tech_title(title: str) -> bool:
    """Check if title matches known non-tech titles."""
    title_lower = _normalize(title)
    for nt in NON_TECH_TITLES:
        if nt in title_lower:
            return True
    return False


def _has_career_evidence(descriptions: list[str]) -> tuple[float, list[str]]:
    """
    Check career descriptions for evidence of building ranking/retrieval/recsys.

    Returns:
        (score: float [0-1], evidence_snippets: list[str])
    """
    evidence_count = 0
    snippets = []
    for desc in descriptions:
        if not desc:
            continue
        for pattern in CAREER_EVIDENCE_RE:
            match = pattern.search(desc)
            if match:
                evidence_count += 1
                # Extract a short snippet around the match
                start = max(0, match.start() - 20)
                end = min(len(desc), match.end() + 30)
                snippets.append(desc[start:end].strip())
                break  # one match per description is enough

    if evidence_count >= 3:
        return 1.0, snippets
    elif evidence_count == 2:
        return 0.8, snippets
    elif evidence_count == 1:
        return 0.5, snippets
    return 0.0, snippets


def _has_production_evidence(descriptions: list[str]) -> float:
    """Check if career descriptions mention production/deployment. Returns [0-1]."""
    prod_count = 0
    for desc in descriptions:
        if not desc:
            continue
        for pattern in PRODUCTION_RE:
            if pattern.search(desc):
                prod_count += 1
                break
    if prod_count >= 2:
        return 1.0
    elif prod_count == 1:
        return 0.6
    return 0.0


# ═══════════════════════════════════════════════════════════════════════════════
# FEATURE 1: Title + Career Fit (the decisive anti-stuffer feature)
# ═══════════════════════════════════════════════════════════════════════════════

def title_career_fit(candidate: dict) -> tuple[float, dict]:
    """
    Score how well the candidate's title + career history fits the JD.

    This is the primary anti-stuffer feature. Skills only matter when
    corroborated by title and career evidence.

    Returns: (score [0-1], detail_dict)
    """
    profile = candidate.get("profile", {})
    career = candidate.get("career_history", [])

    current_title = _normalize(profile.get("current_title", ""))
    all_titles = [_normalize(job.get("title", "")) for job in career]
    all_descriptions = [job.get("description", "") or "" for job in career]

    score = 0.0
    detail = {"current_title": current_title, "is_tech": False, "is_ml_ai": False,
              "career_evidence": 0.0, "production_evidence": 0.0}

    # Current title scoring
    if _is_ml_ai_title(current_title):
        score += 0.40
        detail["is_ml_ai"] = True
        detail["is_tech"] = True
    elif _is_tech_title(current_title):
        score += 0.25
        detail["is_tech"] = True
    elif _is_non_tech_title(current_title):
        score += 0.0  # No credit for non-tech titles
    else:
        score += 0.05  # Unknown title, small benefit of the doubt

    # Career history title bonus — any past ML/AI title is a strong signal
    has_past_ml = any(_is_ml_ai_title(t) for t in all_titles)
    has_past_tech = any(_is_tech_title(t) for t in all_titles)

    if has_past_ml:
        score += 0.15
    elif has_past_tech:
        score += 0.08

    # Career evidence — the most important sub-signal
    career_ev, snippets = _has_career_evidence(all_descriptions)
    score += career_ev * 0.30
    detail["career_evidence"] = career_ev
    detail["evidence_snippets"] = snippets

    # Production evidence
    prod_ev = _has_production_evidence(all_descriptions)
    score += prod_ev * 0.15
    detail["production_evidence"] = prod_ev

    return min(score, 1.0), detail


# ═══════════════════════════════════════════════════════════════════════════════
# FEATURE 2: Trust-Weighted Skill Score
# ═══════════════════════════════════════════════════════════════════════════════

def skill_trust_score(candidate: dict, title_is_tech: bool) -> tuple[float, dict]:
    """
    Compute trust-weighted skill overlap with JD-relevant skills.
    Skills only "count" when the title/career corroborates them (anti-stuffer gate).

    Returns: (score [0-1], detail_dict)
    """
    skills = candidate.get("skills", [])
    if not skills:
        return 0.0, {"matched_skills": [], "raw_score": 0.0}

    # If title is non-tech, heavily discount skill scores (stuffer protection)
    gate_multiplier = 1.0 if title_is_tech else 0.15

    total_trust = 0.0
    max_possible = 0.0
    matched = []

    for skill in skills:
        name = _normalize(skill.get("name", ""))
        prof = _normalize(skill.get("proficiency", "intermediate"))
        endorsements = skill.get("endorsements", 0) or 0
        duration_months = skill.get("duration_months", 0) or 0

        # Determine tier
        if name in TIER1_SKILLS:
            tier_w = SKILL_TIER_WEIGHT["tier1"]
        elif name in TIER2_SKILLS:
            tier_w = SKILL_TIER_WEIGHT["tier2"]
        elif name in TIER3_SKILLS:
            tier_w = SKILL_TIER_WEIGHT["tier3"]
        else:
            continue  # Not a JD-relevant skill

        prof_w = PROFICIENCY_WEIGHT.get(prof, 0.3)
        trust = prof_w * math.log1p(duration_months) * math.log1p(endorsements) * tier_w

        total_trust += trust
        max_possible += 1.0 * math.log1p(60) * math.log1p(50) * SKILL_TIER_WEIGHT["tier1"]

        matched.append({
            "name": skill.get("name", ""),
            "proficiency": prof,
            "duration_months": duration_months,
            "endorsements": endorsements,
            "trust": round(trust, 3),
        })

    if max_possible == 0:
        raw_score = 0.0
    else:
        raw_score = min(total_trust / (max_possible * 0.3), 1.0)  # soft cap

    final_score = raw_score * gate_multiplier

    # Sort matched skills by trust score (highest first)
    matched.sort(key=lambda x: x["trust"], reverse=True)

    return final_score, {"matched_skills": matched[:8], "raw_score": round(raw_score, 4)}


# ═══════════════════════════════════════════════════════════════════════════════
# FEATURE 3: Experience Fit
# ═══════════════════════════════════════════════════════════════════════════════

def experience_fit(candidate: dict) -> tuple[float, dict]:
    """Score experience band fit (ideal 6-8 years). Returns (score, detail)."""
    yoe = candidate.get("profile", {}).get("years_of_experience", 0) or 0
    score = experience_score(yoe)
    return score, {"years_of_experience": yoe}


# ═══════════════════════════════════════════════════════════════════════════════
# FEATURE 4: Product Company Signal
# ═══════════════════════════════════════════════════════════════════════════════

def product_company_signal(candidate: dict) -> tuple[float, dict]:
    """
    Score based on product-company vs consulting experience.
    Returns (score [0-1], detail).
    """
    career = candidate.get("career_history", [])
    if not career:
        return 0.3, {"product_jobs": 0, "consulting_jobs": 0, "total_jobs": 0}

    product_jobs = 0
    consulting_jobs = 0
    product_months = 0
    total_months = 0

    for job in career:
        company = job.get("company", "")
        industry = _normalize(job.get("industry", ""))
        company_size = job.get("company_size", "")
        duration = job.get("duration_months", 0) or 0
        total_months += duration

        if _is_consulting(company):
            consulting_jobs += 1
        else:
            # Check if it's likely a product company
            is_product = False
            if any(ind in industry for ind in PRODUCT_INDUSTRIES):
                is_product = True
            elif company_size in ("1-10", "11-50", "51-200", "201-500"):
                # Smaller companies are more likely product companies
                is_product = True

            if is_product:
                product_jobs += 1
                product_months += duration
            else:
                # Unknown — give partial credit
                product_jobs += 0.3

    total_jobs = len(career)

    if total_jobs == 0:
        return 0.3, {"product_jobs": 0, "consulting_jobs": 0, "total_jobs": 0}

    # Score based on ratio of product experience
    product_ratio = product_jobs / total_jobs if total_jobs > 0 else 0
    consulting_ratio = consulting_jobs / total_jobs if total_jobs > 0 else 0

    score = product_ratio * 0.7 + (1.0 - consulting_ratio) * 0.3

    return min(score, 1.0), {
        "product_jobs": product_jobs,
        "consulting_jobs": consulting_jobs,
        "total_jobs": total_jobs,
    }


# ═══════════════════════════════════════════════════════════════════════════════
# FEATURE 5: Location Fit
# ═══════════════════════════════════════════════════════════════════════════════

def location_fit(candidate: dict) -> tuple[float, dict]:
    """Score location fit for the JD. Returns (score [0-1], detail)."""
    profile = candidate.get("profile", {})
    signals = candidate.get("redrob_signals", {})

    location = _normalize(profile.get("location", ""))
    country = _normalize(profile.get("country", ""))
    willing_to_relocate = signals.get("willing_to_relocate", False)

    # Check tier 1 locations (Pune/Noida)
    for loc in TIER1_LOCATIONS:
        if loc in location:
            return 1.0, {"location": location, "tier": "tier1"}

    # Check tier 2 locations (other major Indian cities)
    for loc in TIER2_LOCATIONS:
        if loc in location:
            return 0.85, {"location": location, "tier": "tier2"}

    # Check if in India
    is_india = "india" in country or "india" in location
    if is_india:
        if willing_to_relocate:
            return 0.65, {"location": location, "tier": "india_relocatable"}
        else:
            return 0.45, {"location": location, "tier": "india_other"}

    # Outside India
    if willing_to_relocate:
        return 0.25, {"location": location, "tier": "international_relocatable"}

    return 0.10, {"location": location, "tier": "international"}


# ═══════════════════════════════════════════════════════════════════════════════
# FEATURE 6: Education Signal (light tiebreaker)
# ═══════════════════════════════════════════════════════════════════════════════

def education_signal(candidate: dict) -> tuple[float, dict]:
    """Light education tiebreaker. Returns (score [0-1], detail)."""
    education = candidate.get("education", [])
    if not education:
        return 0.3, {"degree": "none", "field": "none", "tier": "none"}

    best_score = 0.3
    best_detail = {}

    for edu in education:
        score = 0.3
        field = _normalize(edu.get("field_of_study", ""))
        degree = _normalize(edu.get("degree", ""))
        tier = _normalize(edu.get("tier", ""))

        # Relevant field boost
        relevant_fields = {"computer science", "cs", "information technology", "it",
                          "artificial intelligence", "machine learning", "data science",
                          "electronics", "ece", "electrical", "mathematics", "statistics",
                          "computational"}
        if any(f in field for f in relevant_fields):
            score += 0.25

        # Degree level boost
        if any(d in degree for d in ("phd", "ph.d", "doctorate")):
            score += 0.20
        elif any(d in degree for d in ("m.tech", "m.e.", "m.s.", "msc", "m.sc", "master")):
            score += 0.15
        elif any(d in degree for d in ("b.tech", "b.e.", "bsc", "b.sc", "bachelor")):
            score += 0.05

        # Tier boost
        if tier in ("tier_1", "tier1"):
            score += 0.20
        elif tier in ("tier_2", "tier2"):
            score += 0.10
        elif tier in ("tier_3", "tier3"):
            score += 0.05

        if score > best_score:
            best_score = score
            best_detail = {"degree": degree, "field": field, "tier": tier}

    return min(best_score, 1.0), best_detail


# ═══════════════════════════════════════════════════════════════════════════════
# PENALTY 1: Keyword Stuffer
# ═══════════════════════════════════════════════════════════════════════════════

def keyword_stuffer_penalty(candidate: dict, title_detail: dict) -> float:
    """
    Penalty for non-tech title + many AI skills (keyword stuffer).
    Returns penalty value [0-1] (higher = more penalized).
    """
    profile = candidate.get("profile", {})
    skills = candidate.get("skills", [])
    current_title = _normalize(profile.get("current_title", ""))

    is_non_tech = _is_non_tech_title(current_title)
    is_tech = title_detail.get("is_tech", False)

    if not is_non_tech:
        return 0.0

    # Count AI-relevant skills
    ai_skill_count = sum(
        1 for s in skills
        if _normalize(s.get("name", "")) in ALL_JD_SKILLS
    )

    # Non-tech title with many AI skills = stuffer
    career_evidence = title_detail.get("career_evidence", 0)

    if ai_skill_count >= 6 and career_evidence < 0.3:
        return 1.0  # Strong stuffer signal
    elif ai_skill_count >= 4 and career_evidence < 0.3:
        return 0.8
    elif ai_skill_count >= 3 and is_non_tech and not is_tech:
        return 0.5
    elif is_non_tech and not is_tech:
        return 0.3

    return 0.0


# ═══════════════════════════════════════════════════════════════════════════════
# PENALTY 2: Consulting-Only
# ═══════════════════════════════════════════════════════════════════════════════

def consulting_only_penalty(candidate: dict) -> float:
    """
    Penalty for ALL career history at consulting firms.
    Reduced if there's any product-company experience.
    Returns penalty [0-1].
    """
    career = candidate.get("career_history", [])
    if not career:
        return 0.0

    consulting_count = sum(1 for job in career if _is_consulting(job.get("company", "")))

    if consulting_count == len(career):
        return 1.0  # All consulting
    elif consulting_count > len(career) * 0.7:
        return 0.5  # Mostly consulting
    elif consulting_count > 0:
        return 0.1  # Some consulting
    return 0.0


# ═══════════════════════════════════════════════════════════════════════════════
# PENALTY 3: Research-Only
# ═══════════════════════════════════════════════════════════════════════════════

def research_only_penalty(candidate: dict) -> float:
    """
    Penalty for pure research / academic with no production deployment.
    Returns penalty [0-1].
    """
    career = candidate.get("career_history", [])
    if not career:
        return 0.0

    all_titles = [_normalize(job.get("title", "")) for job in career]
    all_descs = [job.get("description", "") or "" for job in career]

    research_titles = sum(
        1 for t in all_titles
        if "research" in t and not any(w in t for w in ["engineer", "developer"])
    )

    # Check for production language
    has_production = _has_production_evidence(all_descs) > 0.3

    if research_titles == len(career) and not has_production:
        return 1.0  # Pure research, no production
    elif research_titles > len(career) * 0.7 and not has_production:
        return 0.6
    return 0.0


# ═══════════════════════════════════════════════════════════════════════════════
# PENALTY 4: CV / Speech / Robotics Specialist
# ═══════════════════════════════════════════════════════════════════════════════

def cv_speech_penalty(candidate: dict) -> float:
    """
    Penalty for candidates whose primary skills are CV/Speech/Robotics
    with little NLP/IR exposure.
    Returns penalty [0-1].
    """
    skills = candidate.get("skills", [])
    if not skills:
        return 0.0

    cv_speech_count = 0
    nlp_ir_count = 0

    for skill in skills:
        name = _normalize(skill.get("name", ""))
        if name in CV_SPEECH_ROBOTICS_SKILLS:
            cv_speech_count += 1
        if name in NLP_IR_SKILLS:
            nlp_ir_count += 1

    if cv_speech_count >= 3 and nlp_ir_count <= 1:
        return 0.8  # Primarily CV/Speech, little NLP/IR
    elif cv_speech_count >= 2 and nlp_ir_count == 0:
        return 0.5
    return 0.0


# ═══════════════════════════════════════════════════════════════════════════════
# PENALTY 5: Job Hopper
# ═══════════════════════════════════════════════════════════════════════════════

def job_hopper_penalty(candidate: dict) -> float:
    """
    Penalty for frequent job changes (avg tenure < 1.5 years).
    JD explicitly says: "we need someone who plans to be here for 3+ years."
    Returns penalty [0-1].
    """
    career = candidate.get("career_history", [])
    if len(career) <= 1:
        return 0.0

    durations = [job.get("duration_months", 0) or 0 for job in career]
    avg_tenure = sum(durations) / len(durations) if durations else 0

    if avg_tenure < 12:  # Less than 1 year average
        return 0.8
    elif avg_tenure < 18:  # Less than 1.5 years average
        return 0.5
    elif avg_tenure < 24:  # Less than 2 years average
        return 0.2
    return 0.0


# ═══════════════════════════════════════════════════════════════════════════════
# BEHAVIORAL AVAILABILITY MODIFIER
# ═══════════════════════════════════════════════════════════════════════════════

def availability_modifier(candidate: dict) -> tuple[float, dict]:
    """
    Behavioral availability modifier — bounded multiplier [0.5, 1.15].
    A cold candidate is discounted but not zeroed.

    Factors: response rate, recency, open-to-work, interview completion, notice period.
    Returns: (modifier, detail_dict)
    """
    signals = candidate.get("redrob_signals", {})

    response_rate = signals.get("recruiter_response_rate", 0.5) or 0
    open_to_work = signals.get("open_to_work_flag", False)
    interview_completion = signals.get("interview_completion_rate", 0.5) or 0
    notice_days = signals.get("notice_period_days", 60) or 60

    # Days since last active
    last_active_str = signals.get("last_active_date", "")
    if last_active_str:
        try:
            last_active = date.fromisoformat(last_active_str)
            days_inactive = (REFERENCE_DATE - last_active).days
        except (ValueError, TypeError):
            days_inactive = 180  # assume cold
    else:
        days_inactive = 180

    # ── Factor 1: Recruiter response rate (biggest behavioral signal) ────
    # Weight: 35%
    if response_rate >= 0.7:
        resp_score = 1.0
    elif response_rate >= 0.4:
        resp_score = 0.7 + (response_rate - 0.4) / 0.3 * 0.3
    elif response_rate >= 0.1:
        resp_score = 0.3 + (response_rate - 0.1) / 0.3 * 0.4
    else:
        resp_score = 0.2

    # ── Factor 2: Recency / Activity ─────────────────────────────────────
    # Weight: 25%
    if days_inactive <= 14:
        recency_score = 1.0
    elif days_inactive <= 30:
        recency_score = 0.9
    elif days_inactive <= 60:
        recency_score = 0.75
    elif days_inactive <= 120:
        recency_score = 0.5
    else:
        recency_score = 0.3

    # ── Factor 3: Open to work ───────────────────────────────────────────
    # Weight: 15%
    otw_score = 1.0 if open_to_work else 0.4

    # ── Factor 4: Interview completion rate ──────────────────────────────
    # Weight: 15%
    if interview_completion >= 0.8:
        interview_score = 1.0
    elif interview_completion >= 0.5:
        interview_score = 0.7
    else:
        interview_score = 0.4

    # ── Factor 5: Notice period ──────────────────────────────────────────
    # Weight: 10%
    if notice_days <= 30:
        notice_score = 1.0
    elif notice_days <= 60:
        notice_score = 0.7
    elif notice_days <= 90:
        notice_score = 0.4
    else:
        notice_score = 0.2

    # Weighted combination
    raw = (
        0.35 * resp_score +
        0.25 * recency_score +
        0.15 * otw_score +
        0.15 * interview_score +
        0.10 * notice_score
    )

    # Map to [0.5, 1.15] — never zero, never more than 15% boost
    modifier = 0.5 + raw * 0.65

    detail = {
        "response_rate": response_rate,
        "days_inactive": days_inactive,
        "open_to_work": open_to_work,
        "interview_completion": interview_completion,
        "notice_days": notice_days,
        "modifier": round(modifier, 4),
    }

    return round(modifier, 4), detail


# ═══════════════════════════════════════════════════════════════════════════════
# REASONING GENERATOR
# ═══════════════════════════════════════════════════════════════════════════════

def generate_reasoning(candidate: dict, features: dict) -> str:
    """
    Generate a reasoning string from actual matched features.
    No hallucination — every number is pulled from the profile.
    """
    profile = candidate.get("profile", {})
    signals = candidate.get("redrob_signals", {})

    yoe = profile.get("years_of_experience", 0) or 0
    title = profile.get("current_title", "Unknown")
    company = profile.get("current_company", "Unknown")

    parts = []

    # Core identity
    parts.append(f"{yoe:.1f} yrs exp; {title} at {company}")

    # Career evidence
    title_detail = features.get("title_career_detail", {})
    if title_detail.get("career_evidence", 0) >= 0.5:
        snippets = title_detail.get("evidence_snippets", [])
        if snippets:
            # Take shortest snippet
            snippet = min(snippets, key=len)[:80]
            parts.append(f"career evidence: {snippet}")
        else:
            parts.append("strong career evidence of building ranking/retrieval systems")

    # Top matched skills
    skill_detail = features.get("skill_detail", {})
    matched = skill_detail.get("matched_skills", [])
    if matched:
        top_skills = [s["name"] for s in matched[:4]]
        parts.append(f"key skills: {', '.join(top_skills)}")

    # Location
    loc_detail = features.get("location_detail", {})
    location = loc_detail.get("location", "")
    if location:
        parts.append(f"location: {location}")

    # Availability concern (if any)
    avail_detail = features.get("avail_detail", {})
    concerns = []
    resp_rate = avail_detail.get("response_rate", 0)
    days_inactive = avail_detail.get("days_inactive", 0)

    if resp_rate < 0.3:
        concerns.append(f"low response rate ({resp_rate:.2f})")
    if days_inactive > 90:
        concerns.append(f"inactive {days_inactive}d")
    if not avail_detail.get("open_to_work", True):
        concerns.append("not open to work")

    if concerns:
        parts.append(f"concerns: {'; '.join(concerns)}")

    # Penalties
    penalties = []
    if features.get("stuffer_penalty", 0) > 0.3:
        penalties.append("non-tech title with AI skills (possible stuffer)")
    if features.get("consulting_penalty", 0) > 0.5:
        penalties.append("consulting-heavy career")
    if features.get("honeypot", False):
        penalties.append("HONEYPOT — impossible profile data")

    if penalties:
        parts.append(f"flags: {'; '.join(penalties)}")

    reasoning = ". ".join(parts) + "."

    # Ensure it's not too long (keep under 300 chars for CSV readability)
    if len(reasoning) > 350:
        reasoning = reasoning[:347] + "..."

    return reasoning


In [ ]:
%%writefile honeypot.py
"""
honeypot.py — High-precision honeypot / impossible-profile detection.

DESIGN NOTE (why this is conservative):
    The dataset contains ~80 honeypots with *subtly impossible* profiles.
    They are forced to relevance tier 0 in the ground truth, and a submission
    with >10% honeypots in its top 100 is DISQUALIFIED.

    Naive impossibility heuristics badly over-trigger. Empirically, on the real
    100K pool:
        - "skill duration > years_of_experience"   flags ~9,800  (FALSE POSITIVES)
        - "sum of tenures > yoe"                    flags ~2,800  (FALSE POSITIVES)
    Both fire on normal candidates because skill/role durations legitimately
    overlap and the synthetic data sets skill durations high.

    So we ONLY use *internal contradictions* — cases where the profile's own
    numbers disagree with each other. These are near-certain honeypots and do
    not zero real fits. The union of the three rules below catches ~65 of the
    ~80 on the real pool, with no observed false positives.

Rules (a candidate is a honeypot if ANY fire):
    R1  Expert proficiency in a skill with 0 months of use.
    R2  A job's stated duration_months contradicts its own start/end dates
        (claims > 1.5x the actual calendar span, and by > 12 months).
    R3  years_of_experience far exceeds the calendar span since the earliest
        job start (claims > 1.6x the timeline, and by > 3 years).

All dates compared against a FIXED reference date for reproducibility.
"""

from datetime import date

# Fixed reference date — never use date.today() (non-reproducible across runs).
REFERENCE_DATE = date(2026, 6, 25)


def _parse(d: str):
    try:
        return date.fromisoformat(d)
    except (ValueError, TypeError):
        return None


def is_honeypot(candidate: dict) -> tuple[bool, str]:
    """
    Return (is_honeypot, reason). reason is '' when not a honeypot.
    Uses only high-precision internal-contradiction rules.
    """
    profile = candidate.get("profile", {}) or {}
    skills = candidate.get("skills", []) or []
    career = candidate.get("career_history", []) or []
    yoe = profile.get("years_of_experience", 0) or 0

    # ── R1: Expert proficiency with 0 months used ────────────────────────────
    for s in skills:
        prof = (s.get("proficiency") or "").lower()
        dur = s.get("duration_months", 0) or 0
        if prof == "expert" and dur == 0:
            return True, f"Expert proficiency in '{s.get('name')}' with 0 months of use"

    # ── R2: Job duration contradicts its own start/end calendar span ─────────
    for j in career:
        sd = _parse(j.get("start_date") or "")
        ed = _parse(j.get("end_date") or "") or REFERENCE_DATE
        claimed = j.get("duration_months", 0) or 0
        if sd and claimed > 0:
            actual = (ed - sd).days / 30.44
            if actual > 0 and claimed > actual * 1.5 and (claimed - actual) > 12:
                return True, (
                    f"Role at '{j.get('company')}' claims {claimed} months but its "
                    f"own dates span only ~{actual:.0f} months"
                )

    # ── R3: Total experience far exceeds the career timeline ─────────────────
    starts = [_parse(j.get("start_date") or "") for j in career]
    starts = [s for s in starts if s]
    if starts:
        earliest = min(starts)
        span_years = (REFERENCE_DATE - earliest).days / 365.25
        if span_years > 0 and yoe > span_years * 1.6 and (yoe - span_years) > 3:
            return True, (
                f"Claims {yoe:.1f} years of experience but first job began only "
                f"~{span_years:.1f} years ago"
            )

    return False, ""


In [ ]:
%%writefile semantic_lite.py
"""
semantic_lite.py — Self-contained semantic layer (Option B), ZERO heavy deps.

This is the no-setup version of the Option B semantic signal. Instead of dense
neural embeddings (which need torch + sentence-transformers + a model download),
it computes a TF-IDF cosine similarity between the JD and each candidate's text,
in pure Python (stdlib only — no numpy, no network, no precompute, no artifacts).

It is a legitimate *sparse-semantic* relevance signal: candidates whose career
text overlaps the JD's vocabulary (retrieval, ranking, embeddings, recommendation,
production, eval, ...) score higher, weighted by term rarity (IDF). It captures
much of the Option B value while letting the whole pipeline run with a single
`python rank.py ...` command in VSCode.

If you want the higher-quality dense embeddings (your original Option B / BGE),
run precompute.py to build ./artifacts and rank.py will use those instead.

Two streaming passes over the candidates file:
    Pass 1 — document frequencies -> IDF
    Pass 2 — per-candidate TF-IDF cosine vs. the JD
Runtime ~ a few seconds beyond the base scan; memory is a single vocab dict.
"""

import json
import math
import re
from collections import Counter, defaultdict

from build_text import build_text

_TOKEN_RE = re.compile(r"[a-z0-9][a-z0-9+#.\-]*")


def _tokenize(text: str):
    return _TOKEN_RE.findall(text.lower())


def _stream(path: str, is_sample: bool):
    if is_sample:
        with open(path, "r", encoding="utf-8") as f:
            for c in json.load(f):
                yield c
    else:
        with open(path, "r", encoding="utf-8") as f:
            for line in f:
                line = line.strip()
                if not line:
                    continue
                try:
                    yield json.loads(line)
                except json.JSONDecodeError:
                    continue


def compute_semantic_scores(candidates_path: str, jd_text: str, is_sample: bool = False):
    """
    Return (scores, meta):
        scores : {candidate_id: semantic_fit in [0,1]}
        meta   : {'available': True, 'mode': 'tfidf', 'n': N}
    """
    # ── Pass 1: build_text ONCE per candidate; cache term counts + doc freq ──
    # We cache each candidate's non-zero term frequencies so we never recompute
    # build_text (the expensive step). IDF needs a full pass first, so caching is
    # what lets the whole thing stay a single build_text pass.
    df = defaultdict(int)
    cached = []          # list of (cid, {term: tf})
    n_docs = 0
    for c in _stream(candidates_path, is_sample):
        n_docs += 1
        tf = Counter(_tokenize(build_text(c)))
        cached.append((c.get("candidate_id"), tf))
        for t in tf:
            df[t] += 1

    if n_docs == 0:
        return {}, {"available": False, "mode": "tfidf", "n": 0}

    idf = {t: math.log((n_docs + 1) / (dfi + 1)) + 1.0 for t, dfi in df.items()}

    # ── JD vector (TF-IDF over known vocab) ──────────────────────────────────
    jd_tf = Counter(_tokenize(jd_text))
    jd_vec = {t: (1.0 + math.log(tf)) * idf.get(t, 0.0)
              for t, tf in jd_tf.items() if t in idf}
    jd_norm = math.sqrt(sum(w * w for w in jd_vec.values())) or 1.0

    # ── Pass 2 (in-memory over the cache): cosine(JD, candidate) ─────────────
    raw = {}
    for cid, tf in cached:
        norm_sq = 0.0
        dot = 0.0
        for t, f in tf.items():
            w = (1.0 + math.log(f)) * idf[t]
            norm_sq += w * w
            jw = jd_vec.get(t)
            if jw is not None:
                dot += w * jw
        cand_norm = math.sqrt(norm_sq) or 1.0
        raw[cid] = dot / (jd_norm * cand_norm)

    # ── Min-max normalize to [0, 1] across the pool ──────────────────────────
    vals = list(raw.values())
    mn, mx = min(vals), max(vals)
    rng = (mx - mn) or 1.0
    scores = {cid: (v - mn) / rng for cid, v in raw.items()}
    return scores, {"available": True, "mode": "tfidf", "n": n_docs}


In [ ]:
%%writefile semantic.py
"""
semantic.py — Option B semantic layer, consumed as a feature by the combined ranker.

At RANK TIME this module does no ML and needs no network or torch. It only:
  1. Loads the precomputed artifacts (embeddings, ids, JD embedding, BM25 index)
     produced offline by precompute.py.
  2. Computes a dense cosine score (candidate embeddings . JD embedding) for the
     whole pool in a single matrix-vector product.
  3. Computes a sparse BM25 lexical score for the JD query over the whole pool.
  4. Min-max normalizes each across the pool and blends them into one
     `semantic_fit` value per candidate_id, in [0, 1].

The result is a dict {candidate_id: semantic_fit}. rank.py looks each candidate up
and adds WEIGHTS["semantic_fit"] * semantic_fit to the structured Option A score.

Design guarantees:
  - Reproducible: pure numpy, deterministic.
  - Fast: one matvec + one BM25 query over 100K docs — seconds on CPU.
  - Safe: if artifacts are missing/incomplete, load_semantic_scores() returns
    ({}, meta) and the ranker falls back to pure Option A (semantic_fit = 0).
"""

import os
import pickle

import numpy as np


def _minmax(arr: np.ndarray) -> np.ndarray:
    """Min-max normalize a 1-D array to [0, 1]. Flat array -> zeros."""
    mn = float(arr.min())
    mx = float(arr.max())
    if mx - mn < 1e-9:
        return np.zeros_like(arr, dtype=np.float32)
    return ((arr - mn) / (mx - mn)).astype(np.float32)


def load_semantic_scores(
    artifacts_dir: str,
    jd_text: str,
    dense_weight: float = 0.65,
    sparse_weight: float = 0.35,
) -> tuple[dict, dict]:
    """
    Load artifacts and compute a blended semantic score per candidate_id.

    Returns:
        (scores, meta)
        scores : {candidate_id (str): semantic_fit (float in [0,1])}
                 Empty dict if artifacts are unavailable -> pure Option A fallback.
        meta   : diagnostics (which components were used, counts, availability).
    """
    meta = {
        "available": False,
        "dense_used": False,
        "sparse_used": False,
        "n": 0,
        "reason": "",
    }

    emb_path = os.path.join(artifacts_dir, "candidate_embeddings.npy")
    ids_path = os.path.join(artifacts_dir, "candidate_ids.npy")
    jd_path = os.path.join(artifacts_dir, "jd_embedding.npy")
    bm25_path = os.path.join(artifacts_dir, "bm25_index.pkl")

    # Dense component requires embeddings + ids + JD embedding.
    if not (os.path.exists(emb_path) and os.path.exists(ids_path) and os.path.exists(jd_path)):
        meta["reason"] = f"embeddings/ids/jd not found in {artifacts_dir}"
        return {}, meta

    cand_emb = np.load(emb_path)                                   # (N, D) float32, L2-normalized
    cand_ids = np.load(ids_path, allow_pickle=True)               # (N,)
    jd_emb = np.load(jd_path).astype(np.float32).reshape(-1)      # (D,)

    n = len(cand_ids)
    meta["n"] = int(n)

    # ── Dense: cosine == dot product (both sides L2-normalized in precompute) ──
    dense = cand_emb @ jd_emb                                      # (N,)
    dense_norm = _minmax(dense)
    meta["dense_used"] = True

    # ── Sparse: BM25 over the JD query, if the index is present ───────────────
    sparse_norm = None
    if os.path.exists(bm25_path):
        try:
            with open(bm25_path, "rb") as f:
                bm25 = pickle.load(f)
            jd_tokens = jd_text.lower().split()
            bm25_scores = np.asarray(bm25.get_scores(jd_tokens), dtype=np.float32)
            if bm25_scores.shape[0] == n:
                sparse_norm = _minmax(bm25_scores)
                meta["sparse_used"] = True
        except Exception as e:  # noqa: BLE001 - never let sparse break dense
            meta["reason"] = f"bm25 skipped: {e}"

    # ── Blend ─────────────────────────────────────────────────────────────────
    if sparse_norm is not None:
        total = dense_weight + sparse_weight
        dw, sw = dense_weight / total, sparse_weight / total
        blended = dw * dense_norm + sw * sparse_norm
    else:
        blended = dense_norm  # dense-only if no BM25 index

    blended = _minmax(blended)  # re-normalize the blend to [0, 1]

    scores = {str(cid): float(s) for cid, s in zip(cand_ids, blended)}
    meta["available"] = True
    return scores, meta


In [ ]:
%%writefile rank.py
#!/usr/bin/env python3
"""
rank.py — Combined A+B ranker for the Redrob Hackathon.

ARCHITECTURE  (Option A is the floor, Option B is a feature on top)
    final_score = fit_score * availability_modifier * honeypot_ok

    fit_score  = W.title_career_fit    * title_career_fit(A)
               + W.semantic_fit        * semantic_fit(B)          <-- embeddings + BM25
               + W.skill_trust         * skill_trust(A)
               + W.experience_fit      * experience_fit(A)
               + W.product_company     * product_company(A)
               + W.career_evidence     * career_evidence(A)
               + W.location_fit        * location_fit(A)
               + W.education           * education(A)
               - W.keyword_stuffer     * keyword_stuffer_penalty(A)
               - W.consulting_only     * consulting_only_penalty(A)
               - W.research_only       * research_only_penalty(A)
               - W.cv_speech_only      * cv_speech_penalty(A)
               - W.job_hopper          * job_hopper_penalty(A)

    availability_modifier ∈ [0.5, 1.15]   (behavioral signals, bounded multiplier)
    honeypot_ok ∈ {0, 1}                   (high-precision impossible-profile gate)

WHY THIS SHAPE
    - The structured layer (A) decides ORDER at the top of the list, where 80% of
      the score lives (NDCG@10 + NDCG@50). Its penalties + honeypot gate keep
      keyword-stuffers and honeypots out of the top ranks.
    - The semantic layer (B) is an additive feature that lifts genuinely-strong
      but quietly-worded candidates ("built a recommendation system", never says
      "RAG") that a keyword-only system would miss.
    - We score ALL 100K (a full scan is only ~30s on CPU), so recall is perfect
      and A truly is the floor — no candidate is dropped by a retrieval shortlist.

COMPUTE (rank step): <=5 min, <=16 GB, CPU-only, NO network, NO torch.
    Embeddings are precomputed offline by precompute.py into ./artifacts.
    If ./artifacts is absent, the ranker runs as PURE OPTION A (semantic_fit = 0).

USAGE
    # Combined (needs precomputed artifacts):
    python rank.py --candidates ./candidates.jsonl --artifacts ./artifacts --out ./submission.csv

    # Pure Option A fallback (no artifacts needed):
    python rank.py --candidates ./candidates.jsonl --out ./submission.csv

    # Small sample (JSON array like sample_candidates.json):
    python rank.py --candidates ./sample_candidates.json --out ./out.csv --sample
"""

import argparse
import csv
import heapq
import json
import os
import sys
import time

from config import WEIGHTS, TOP_K, CSV_COLUMNS, SEMANTIC_DENSE_WEIGHT, SEMANTIC_SPARSE_WEIGHT
from features import (
    title_career_fit,
    skill_trust_score,
    experience_fit,
    product_company_signal,
    location_fit,
    education_signal,
    keyword_stuffer_penalty,
    consulting_only_penalty,
    research_only_penalty,
    cv_speech_penalty,
    job_hopper_penalty,
    availability_modifier,
    generate_reasoning,
)
from honeypot import is_honeypot
# NOTE: the semantic backends (semantic.py -> numpy; semantic_lite.py -> stdlib)
# are imported lazily inside run_ranking so the default TF-IDF / pure-A paths need
# no third-party packages installed.


def compute_score(candidate: dict, semantic_fit: float) -> tuple[float, dict]:
    """Combined score for one candidate. semantic_fit in [0,1] (0 if no artifacts)."""
    W = WEIGHTS

    # ── Structured positive features (Option A) ──────────────────────────────
    tcf, title_detail = title_career_fit(candidate)
    is_tech = title_detail.get("is_tech", False)
    sts, skill_detail = skill_trust_score(candidate, is_tech)
    exf, _ = experience_fit(candidate)
    pcs, _ = product_company_signal(candidate)
    lof, loc_detail = location_fit(candidate)
    edu, _ = education_signal(candidate)
    career_ev = title_detail.get("career_evidence", 0)

    fit_score = (
        W["title_career_fit"]      * tcf +
        W["semantic_fit"]          * semantic_fit +          # <-- Option B, fused in
        W["skill_trust"]           * sts +
        W["experience_fit"]        * exf +
        W["product_company"]       * pcs +
        W["career_evidence_bonus"] * career_ev +
        W["location_fit"]          * lof +
        W["education"]             * edu
    )

    # ── Penalties (Option A) ─────────────────────────────────────────────────
    stf_pen = keyword_stuffer_penalty(candidate, title_detail)
    con_pen = consulting_only_penalty(candidate)
    res_pen = research_only_penalty(candidate)
    cvs_pen = cv_speech_penalty(candidate)
    jh_pen = job_hopper_penalty(candidate)

    penalty = (
        W["keyword_stuffer"] * stf_pen +
        W["consulting_only"] * con_pen +
        W["research_only"]   * res_pen +
        W["cv_speech_only"]  * cvs_pen +
        W["job_hopper"]      * jh_pen
    )
    fit_score = max(fit_score - penalty, 0.0)

    # ── Behavioral availability modifier (bounded multiplier) ────────────────
    avail_mod, avail_detail = availability_modifier(candidate)

    # ── Honeypot hard gate (high-precision) ──────────────────────────────────
    hp_flag, hp_reason = is_honeypot(candidate)
    honeypot_ok = 0.0 if hp_flag else 1.0

    final_score = fit_score * avail_mod * honeypot_ok

    features = {
        "title_career_detail": title_detail,
        "skill_detail": skill_detail,
        "location_detail": loc_detail,
        "avail_detail": avail_detail,
        "career_evidence": round(career_ev, 4),
        "semantic_fit": round(semantic_fit, 4),
        "stuffer_penalty": round(stf_pen, 4),
        "consulting_penalty": round(con_pen, 4),
        "honeypot": hp_flag,
        "honeypot_reason": hp_reason,
        "fit_score": round(fit_score, 4),
        "avail_mod": round(avail_mod, 4),
        "final_score": round(final_score, 6),
    }
    return final_score, features


def _augment_reasoning(base: str, features: dict) -> str:
    """Append a short, honest semantic note (no hallucination)."""
    sem = features.get("semantic_fit", 0.0)
    if sem >= 0.80:
        note = "strong semantic match to JD"
    elif sem >= 0.60:
        note = "good semantic match to JD"
    else:
        return base
    # Insert the note before the trailing period.
    base = base.rstrip()
    if base.endswith("."):
        base = base[:-1]
    return f"{base}; {note}."


def load_jsonl_stream(filepath: str):
    with open(filepath, "r", encoding="utf-8") as f:
        for line_num, line in enumerate(f, 1):
            line = line.strip()
            if not line:
                continue
            try:
                yield json.loads(line)
            except json.JSONDecodeError as e:
                print(f"  [WARN] Skipping malformed line {line_num}: {e}", file=sys.stderr)


def load_json_sample(filepath: str):
    with open(filepath, "r", encoding="utf-8") as f:
        return json.load(f)


def run_ranking(candidates_path, output_path, artifacts_dir, jd_path,
                is_sample=False, semantic_mode="auto"):
    print("=" * 72)
    print("  REDROB HACKATHON — Combined Ranker (A structured + B semantic)")
    print("=" * 72)

    start = time.time()

    # JD text (used by both semantic backends as the query).
    jd_text = ""
    if jd_path and os.path.exists(jd_path):
        with open(jd_path, "r", encoding="utf-8") as f:
            jd_text = f.read()

    # ── Resolve which semantic backend to use ────────────────────────────────
    #   auto  -> dense embeddings if ./artifacts exists, else built-in TF-IDF
    #   tfidf -> built-in TF-IDF (pure stdlib; no setup)
    #   embeddings -> precomputed BGE/MiniLM artifacts (needs numpy + precompute.py)
    #   off   -> pure Option A (semantic_fit = 0)
    has_artifacts = bool(artifacts_dir) and os.path.exists(
        os.path.join(artifacts_dir, "candidate_embeddings.npy")
    )
    resolved = semantic_mode
    if resolved == "auto":
        resolved = "embeddings" if has_artifacts else "tfidf"

    semantic_scores = {}
    if resolved == "embeddings":
        from semantic import load_semantic_scores  # lazy: needs numpy
        semantic_scores, meta = load_semantic_scores(
            artifacts_dir, jd_text,
            dense_weight=SEMANTIC_DENSE_WEIGHT, sparse_weight=SEMANTIC_SPARSE_WEIGHT,
        )
        if meta["available"]:
            comps = [c for c, u in (("dense/embeddings", meta["dense_used"]),
                                    ("sparse/BM25", meta["sparse_used"])) if u]
            print(f"  Semantic layer: ON — dense embeddings ({' + '.join(comps)}; {meta['n']:,} candidates)")
        else:
            print(f"  Semantic layer: embeddings unavailable ({meta['reason']}); using built-in TF-IDF.")
            resolved = "tfidf"

    if resolved == "tfidf":
        from semantic_lite import compute_semantic_scores  # lazy: stdlib only
        semantic_scores, meta = compute_semantic_scores(candidates_path, jd_text, is_sample=is_sample)
        print(f"  Semantic layer: ON — built-in TF-IDF ({meta['n']:,} candidates; no setup required)")
    elif resolved == "off":
        print("  Semantic layer: OFF (--semantic off) -> PURE OPTION A.")

    print(f"  Input : {candidates_path}")
    print(f"  Output: {output_path}")
    print()

    # ── Score every candidate; keep a top-K min-heap ─────────────────────────
    iterator = load_json_sample(candidates_path) if is_sample else load_jsonl_stream(candidates_path)
    heap = []  # (score, cid, candidate, features)
    total = 0
    honeypots = 0

    for candidate in iterator:
        total += 1
        if total % 20000 == 0:
            print(f"  Scored {total:,} in {time.time()-start:.1f}s...")
        cid = candidate.get("candidate_id", f"UNKNOWN_{total}")
        sem = semantic_scores.get(cid, 0.0)
        score, features = compute_score(candidate, sem)
        if features["honeypot"]:
            honeypots += 1
        if len(heap) < TOP_K:
            heapq.heappush(heap, (score, cid, candidate, features))
        elif score > heap[0][0]:
            heapq.heapreplace(heap, (score, cid, candidate, features))

    # ── Sort: score desc, then candidate_id asc (tie-break per spec) ─────────
    top = sorted(heap, key=lambda x: (-round(x[0], 4), x[1]))

    rows = []
    for rank, (score, cid, candidate, features) in enumerate(top, 1):
        reasoning = _augment_reasoning(generate_reasoning(candidate, features), features)
        rows.append({
            "candidate_id": cid,
            "rank": rank,
            "score": f"{score:.4f}",
            "reasoning": reasoning,
        })

    with open(output_path, "w", newline="", encoding="utf-8") as f:
        writer = csv.DictWriter(f, fieldnames=CSV_COLUMNS)
        writer.writeheader()
        writer.writerows(rows)

    elapsed = time.time() - start

    # ── Diagnostics ──────────────────────────────────────────────────────────
    print(f"\n  Scored {total:,} candidates in {elapsed:.1f}s  (limit 300s)")
    print(f"  Honeypots detected in pool: {honeypots}")
    if rows:
        print(f"  Score range: {rows[0]['score']} (rank 1) -> {rows[-1]['score']} (rank {len(rows)})")

    hp_top = sum(1 for _, _, _, ft in top if ft["honeypot"])
    hp_rate = 100.0 * hp_top / max(len(top), 1)
    print(f"  Honeypot rate in top {len(top)}: {hp_top} ({hp_rate:.1f}%)  "
          f"{'[OK]' if hp_rate <= 10 else '[DQ RISK!]'}")

    # Keyword-stuffer sanity: non-tech titles in the top 25 should be ~none.
    stuffer_top = sum(
        1 for _, _, _, ft in top[:25]
        if ft["stuffer_penalty"] >= 0.5
    )
    print(f"  Keyword-stuffer flags in top 25: {stuffer_top}  {'[OK]' if stuffer_top == 0 else '[CHECK]'}")

    print("\n  TOP 10:")
    for row, (_, _, cand, _) in zip(rows[:10], top[:10]):
        p = cand.get("profile", {})
        print(f"   {row['rank']:>3}. {row['candidate_id']} s={row['score']} "
              f"{p.get('current_title','?')} @ {p.get('current_company','?')} "
              f"({p.get('years_of_experience',0)}y)")

    # Format self-checks (mirror the official validator)
    scores = [float(r["score"]) for r in rows]
    checks = {
        "exactly 100 rows": len(rows) == TOP_K or is_sample,
        "scores non-increasing": all(scores[i] >= scores[i+1] for i in range(len(scores)-1)),
        "unique ranks": len({r["rank"] for r in rows}) == len(rows),
        "unique candidate_ids": len({r["candidate_id"] for r in rows}) == len(rows),
        "scores differentiated": len(set(scores)) > 1,
        "no empty reasoning": all(r["reasoning"].strip() for r in rows),
        "within time budget": elapsed <= 300,
    }
    print("\n  SELF-CHECKS:")
    for k, v in checks.items():
        print(f"   [{'OK' if v else 'FAIL'}] {k}")
    print("=" * 72)

    if not all(checks.values()):
        print("  [WARNING] One or more self-checks failed — inspect before submitting.")


def main():
    ap = argparse.ArgumentParser(description="Redrob combined A+B ranker")
    ap.add_argument("--candidates", required=True, help="candidates.jsonl (or JSON array with --sample)")
    ap.add_argument("--out", default="submission.csv", help="output CSV path")
    ap.add_argument("--artifacts", default="./artifacts",
                    help="precomputed dense-embedding dir (used automatically if present)")
    ap.add_argument("--jd", default="./job_description.txt", help="JD text (semantic + BM25 query)")
    ap.add_argument("--semantic", choices=["auto", "tfidf", "embeddings", "off"], default="auto",
                    help="semantic backend: auto (embeddings if ./artifacts else built-in TF-IDF), "
                         "tfidf (pure stdlib, no setup), embeddings (needs precompute.py), or off (pure A)")
    ap.add_argument("--sample", action="store_true", help="input is a JSON array, not JSONL")
    args = ap.parse_args()

    if not os.path.exists(args.candidates):
        print(f"ERROR: candidates file not found: {args.candidates}", file=sys.stderr)
        sys.exit(1)

    run_ranking(args.candidates, args.out, args.artifacts, args.jd,
                is_sample=args.sample, semantic_mode=args.semantic)


if __name__ == "__main__":
    main()


In [ ]:
%%writefile validate_submission.py
#!/usr/bin/env python3
"""
Validate submission CSV per challenge rules (sections 2–3).
Row 1 = header. Rows 2–101 = exactly 100 data rows. CSV only.
"""

import csv
import re
import sys
from pathlib import Path

REQUIRED_HEADER = ["candidate_id", "rank", "score", "reasoning"]
CANDIDATE_ID_PATTERN = re.compile(r"^CAND_[0-9]{7}$")
DATA_ROW_START = 2
EXPECTED_DATA_ROWS = 100


def validate_submission(csv_path):
    errors = []
    path = Path(csv_path)

    if path.suffix.lower() != ".csv":
        errors.append("Filename must use a .csv extension.")
    elif not path.stem:
        errors.append("Filename must be your registered participant ID (e.g. team_xxx.csv).")

    try:
        with open(path, "r", encoding="utf-8", newline="") as f:
            reader = csv.reader(f)

            try:
                header = next(reader)
            except StopIteration:
                errors.append("Row 1 must be the header row; file is empty.")
                return errors

            # Row 1: column names and their order come from this line only
            if header != REQUIRED_HEADER:
                errors.append(
                    "Row 1 (header) must be exactly:\n"
                    f"  {','.join(REQUIRED_HEADER)}\n"
                    f"Found:\n"
                    f"  {','.join(header)}"
                )

            data_rows = []
            for row in reader:
                if any(cell.strip() for cell in row):
                    data_rows.append(row)

    except UnicodeDecodeError:
        errors.append("File must be UTF-8 encoded.")
        return errors
    except OSError as e:
        errors.append(f"Cannot read file: {e}")
        return errors

    n = len(data_rows)
    if n != EXPECTED_DATA_ROWS:
        errors.append(
            f"After the header (row 1), there must be exactly {EXPECTED_DATA_ROWS} "
            f"data rows (rows {DATA_ROW_START}–{DATA_ROW_START + EXPECTED_DATA_ROWS - 1}); "
            f"found {n}."
        )

    seen_ids = set()
    seen_ranks = set()
    by_rank = []

    for i, cells in enumerate(data_rows):
        row_num = DATA_ROW_START + i

        if len(cells) != len(REQUIRED_HEADER):
            errors.append(
                f"Row {row_num}: expected {len(REQUIRED_HEADER)} columns "
                f"({','.join(REQUIRED_HEADER)}), got {len(cells)}."
            )
            continue

        row = dict(zip(REQUIRED_HEADER, cells))
        cid = row["candidate_id"].strip()
        rank_s = row["rank"].strip()
        score_s = row["score"].strip()

        if not cid:
            errors.append(f"Row {row_num}: candidate_id is required.")
        elif not CANDIDATE_ID_PATTERN.match(cid):
            errors.append(
                f"Row {row_num}: candidate_id must be CAND_XXXXXXX (7 digits)."
            )
        elif cid in seen_ids:
            errors.append(f"Row {row_num}: duplicate candidate_id '{cid}'.")
        else:
            seen_ids.add(cid)

        try:
            rank = int(rank_s)
            if str(rank) != rank_s:
                raise ValueError
            if not 1 <= rank <= 100:
                errors.append(f"Row {row_num}: rank must be between 1 and 100.")
            elif rank in seen_ranks:
                errors.append(f"Row {row_num}: duplicate rank {rank}.")
            else:
                seen_ranks.add(rank)
        except ValueError:
            errors.append(f"Row {row_num}: rank must be an integer (1–100).")
            rank = None

        try:
            score = float(score_s)
        except ValueError:
            errors.append(f"Row {row_num}: score must be a float.")
            score = None

        if rank is not None and score is not None and cid:
            by_rank.append((rank, score, cid))

    missing = set(range(1, 101)) - seen_ranks
    if missing:
        errors.append(
            f"Each rank 1–100 must appear exactly once; missing: {sorted(missing)}"
        )

    by_rank.sort(key=lambda x: x[0])

    for i in range(len(by_rank) - 1):
        r1, s1, _ = by_rank[i]
        r2, s2, _ = by_rank[i + 1]
        if s1 < s2:
            errors.append(
                f"score must be non-increasing by rank: "
                f"rank {r1} ({s1}) < rank {r2} ({s2})."
            )

    for i in range(len(by_rank) - 1):
        r1, s1, c1 = by_rank[i]
        r2, s2, c2 = by_rank[i + 1]
        if s1 == s2 and c1 > c2:
            errors.append(
                f"Equal scores at ranks {r1} and {r2}: "
                f"tie-break requires candidate_id ascending "
                f"({c1!r} > {c2!r})."
            )

    return errors


def main():
    if len(sys.argv) != 2:
        print("Usage: python validate_submission.py <participant_id>.csv")
        sys.exit(1)

    errors = validate_submission(sys.argv[1])
    if errors:
        print(f"Validation failed ({len(errors)} issue(s)):\n")
        for e in errors:
            print(f"- {e}")
        sys.exit(1)

    print("Submission is valid.")


if __name__ == "__main__":
    main()


In [ ]:
%%writefile job_description.txt
Job Description: Senior AI Engineer — Founding Team
Company: Redrob AI (Series A AI-native talent intelligence platform)
Location: Pune/Noida, India (Hybrid — flexible cadence) | Open to relocation candidates from Tier-1 Indian cities
Employment Type: Full-time
Experience Required: 5–9 years (see "what we mean by this" below)

Let's be honest about this role
We're going to write this JD differently from most. We're a Series A company that just raised our round and we're building a new AI Engineering org from scratch. This is the kind of role where the JD changes every six months because the company changes every six months. So instead of pretending we have a fixed checklist, we're going to tell you what we actually need and what we've gotten wrong before.
If you've spent your career at Google or Meta and you want a well-scoped role with a defined ladder, this isn't it.
If you've spent your career bouncing between early-stage startups and you want to "just code" without having to think about product or recruiter workflows or eval frameworks, this also isn't it.
We need someone who is simultaneously comfortable with two things that sound contradictory:
Deep technical depth in modern ML systems — embeddings, retrieval, ranking, LLMs, fine-tuning.
Scrappy product-engineering attitude — willing to ship a working ranker in a week even if the underlying ML is "obviously suboptimal," because we need to learn from real users before we know what to actually optimize for.
These are not contradictory in real life. They feel contradictory because of how engineering culture sorted itself into "researcher" vs "shipper" archetypes. We need both modes available in the same person, and we'd rather you tilt slightly toward shipper than toward researcher.

What you'd actually be doing
The high-level mandate: own the intelligence layer of Redrob's product. That means the ranking, retrieval, and matching systems that decide what recruiters see when they search for candidates and what candidates see when they search for roles.
In practical terms, your first 90 days will probably look like:
Weeks 1-3: Audit what we currently have (it's mostly BM25 + rule-based scoring, working but not great). Identify the 3-4 highest-leverage things to fix.
Weeks 4-8: Ship a v2 ranking system that demonstrably improves recruiter-engagement metrics. This will involve embeddings, hybrid retrieval, and probably some LLM-based re-ranking, but the architecture is your call.
Weeks 9-12: Set up the evaluation infrastructure — offline benchmarks, online A/B testing, recruiter-feedback loops — so we can keep improving without flying blind.
Beyond that, you'll be driving the long-term architecture of how we do candidate-JD matching at scale, mentoring the next round of hires (we're growing the team from 4 to 12 engineers in the next year), and working closely with our recruiter-experience PM on what to build.

What we mean by "5-9 years"
This is a range, not a requirement. Some people hit "senior engineer" judgment at 4 years; some never hit it after 15. We've used 5-9 because it's roughly where people we've hired into this kind of role have landed, but we'll seriously consider candidates outside the band if other signals are strong.
That said, here are the disqualifiers we actually apply:
If you've spent your career in pure research environments (academic labs, research-only roles) without any production deployment — we will not move forward. We are explicit about this. We've tried it twice and it didn't work for either side.
If your "AI experience" consists primarily of recent (under 12 months) projects using LangChain to call OpenAI — we will probably not move forward, unless you can demonstrate substantial pre-LLM-era ML production experience. We're looking for people who understood retrieval and ranking before it became fashionable.
If you are a senior engineer who hasn't written production code in the last 18 months because you've moved into "architecture" or "tech lead" roles — we will probably not move forward. This role writes code.

The skills inventory (please read carefully)
Most JDs list 20 skills and you're supposed to have all of them. We're going to do this differently.
Things you absolutely need
Production experience with embeddings-based retrieval systems (sentence-transformers, OpenAI embeddings, BGE, E5, or similar) deployed to real users. We don't care which model — we care that you've handled embedding drift, index refresh, retrieval-quality regression in production.
Production experience with vector databases or hybrid search infrastructure — Pinecone, Weaviate, Qdrant, Milvus, OpenSearch, Elasticsearch, FAISS, or something similar. Again, the specific tech doesn't matter; the operational experience does.
Strong Python. Yes really, we care about code quality.
Hands-on experience designing evaluation frameworks for ranking systems — NDCG, MRR, MAP, offline-to-online correlation, A/B test interpretation. If you've never thought about how to evaluate a ranking system rigorously, this role will be very painful.
Things we'd like you to have but won't reject you for
LLM fine-tuning experience (LoRA, QLoRA, PEFT)
Experience with learning-to-rank models (XGBoost-based or neural)
Prior exposure to HR-tech, recruiting tech, or marketplace products
Background in distributed systems or large-scale inference optimization
Open-source contributions in the AI/ML space
Things we explicitly do NOT want
This is the section most JDs skip but we think it's the most important:
Title-chasers. If your career trajectory shows you optimizing for "Senior" → "Staff" → "Principal" titles by switching companies every 1.5 years, we're not a fit. We need someone who plans to be here for 3+ years.
Framework enthusiasts. If your GitHub is full of LangChain tutorials and your blog posts are "How I used [hot framework] to build [demo]" — that's fine but it's not what we need. We need people who think about systems, not frameworks.
People who have only worked at consulting firms (TCS, Infosys, Wipro, Accenture, Cognizant, Capgemini, etc.) in their entire career. We've had bad fit experiences in both directions. If you're currently at one of these companies but have prior product-company experience, that's fine.
People whose primary expertise is computer vision, speech, or robotics without significant NLP/IR exposure. We respect your work but you'd be re-learning fundamentals here.
People whose work has been entirely on closed-source proprietary systems for 5+ years without external validation (papers, talks, open-source). We need to see how you think, not just trust that you can think.

On location, comp, and logistics
Location: Pune/Noida-preferred but flexible. We have offices in Noida and Pune(mostly used Tue/Thu). We don't require any specific number of in-office days but we expect quarterly travel for offsites. Candidates in Hyderabad, Pune, Mumbai, Delhi NCR welcome to apply. Outside India: case-by-case, but we don't sponsor work visas.
Notice period: We'd love sub-30-day notice. We can buy out up to 30 days. 30+ day notice candidates are still in scope but the bar gets higher.

The vibe check
We genuinely believe culture-fit matters more at this stage than skills-fit. Skills are teachable; the rest mostly isn't.
We work async-first and write a lot. If you find writing painful, you'll find this role painful.
We disagree openly and decide quickly. If you find that style abrasive, you'll find this role abrasive.
We move fast and break things, with the caveat that "things" are usually our internal assumptions, not user-facing systems. If you need a stable, mature codebase to be productive, you'll find this role unstable.

How to read between the lines
The "ideal candidate" we're imagining is roughly:
6-8 years total experience, of which 4-5 are in applied ML/AI roles at product companies (not pure services).
Has shipped at least one end-to-end ranking, search, or recommendation system to real users at meaningful scale.
Has strong opinions about retrieval (hybrid vs dense), evaluation (offline vs online), and LLM integration (when to fine-tune vs prompt) — and can defend them with reference to systems they actually built.
Located in or willing to relocate to Noida or Pune.
Active on Redrob platform (or has clear signal of being in the job market) so we can actually talk to them.
We are aware this is a narrow profile. We're not expecting to find many matches in a 100K candidate pool. We're explicitly OK with that — we'd rather see 10 great matches than 1000 maybes.

Final note for the participants of the Redrob hackathon
If you're reading this in the context of the Intelligent Candidate Discovery & Ranking Challenge:
The "right answer" to this JD is not "find candidates whose skills section contains the most AI keywords." That's a trap we've explicitly built into the dataset.
The right answer involves reasoning about the gap between what the JD says and what the JD means. A Tier 5 candidate may not use the words "RAG" or "Pinecone" in their profile, but if their career history shows they built a recommendation system at a product company, they're a fit. A candidate who has all the AI keywords listed as skills but whose title is "Marketing Manager" is not a fit, no matter how perfect their skill list looks.
Your ranking system should also weigh behavioral signals — a perfect-on-paper candidate who hasn't logged in for 6 months and has a 5% recruiter response rate is, for hiring purposes, not actually available. Down-weight them appropriately.
Good luck.


## Step 2 - Load a small candidate sample (pre-loaded, 50 candidates)
This is the official `sample_candidates.json` from the hackathon bundle.


In [ ]:
%%writefile sample_candidates.json
[
  {
    "candidate_id": "CAND_0000001",
    "profile": {
      "anonymized_name": "Ira Vora",
      "headline": "Backend Engineer | SQL, Spark, Cloud",
      "summary": "Software / data professional with 6.9 years of experience building data pipelines, backend systems, and analytics infrastructure. I'm a backend/data hybrid \u2014 Spark, Airflow, SQL warehouses are home territory; I'm building competence on the ML side. My toolkit is solid on the data engineering side \u2014 Python, SQL, Spark, Airflow, warehouse design \u2014 and I've completed a couple of self-directed ML projects (Kaggle competitions, side projects fine-tuning small models). Interested in transitioning toward more AI/ML-focused work, ideally at a company where I can leverage my existing data-infra skills while learning modern ML practice.",
      "location": "Toronto",
      "country": "Canada",
      "years_of_experience": 6.9,
      "current_title": "Backend Engineer",
      "current_company": "Mindtree",
      "current_company_size": "10001+",
      "current_industry": "IT Services"
    },
    "career_history": [
      {
        "company": "Mindtree",
        "title": "Backend Engineer",
        "start_date": "2024-03-08",
        "end_date": null,
        "duration_months": 27,
        "is_current": true,
        "industry": "IT Services",
        "company_size": "10001+",
        "description": "Implemented streaming data pipelines on Kafka and Spark Streaming for a real-time user-activity processing platform. Designed the schema-registry integration, the watermark/state management approach, and the deduplication logic for late-arriving events. Worked closely with the data science team to make sure feature pipelines aligned with what their models needed. Most of my career has been data engineering, with some adjacent ML exposure."
      },
      {
        "company": "Dunder Mifflin",
        "title": "Analytics Engineer",
        "start_date": "2019-07-03",
        "end_date": "2024-01-08",
        "duration_months": 55,
        "is_current": false,
        "industry": "Paper Products",
        "company_size": "201-500",
        "description": "Built and maintained data pipelines on Apache Airflow processing ~500GB of daily transactional data across 12 source systems. Worked extensively with Spark (PySpark) for batch processing and dbt for the transformation/modeling layer in our Snowflake warehouse. Owned the on-call rotation for data quality issues \u2014 wrote most of the data quality checks that detect schema drift and unusual volume changes. The pipeline supports the analytics team and a few internal ML models."
      }
    ],
    "education": [
      {
        "institution": "Lovely Professional University",
        "degree": "B.E.",
        "field_of_study": "Computer Science",
        "start_year": 2017,
        "end_year": 2020,
        "grade": "8.24 CGPA",
        "tier": "tier_3"
      }
    ],
    "skills": [
      {
        "name": "Tailwind",
        "proficiency": "intermediate",
        "endorsements": 3,
        "duration_months": 13
      },
      {
        "name": "NLP",
        "proficiency": "advanced",
        "endorsements": 37,
        "duration_months": 26
      },
      {
        "name": "Image Classification",
        "proficiency": "advanced",
        "endorsements": 7,
        "duration_months": 40
      },
      {
        "name": "Fine-tuning LLMs",
        "proficiency": "advanced",
        "endorsements": 21,
        "duration_months": 36
      },
      {
        "name": "Weights & Biases",
        "proficiency": "intermediate",
        "endorsements": 13,
        "duration_months": 30
      },
      {
        "name": "Speech Recognition",
        "proficiency": "advanced",
        "endorsements": 52,
        "duration_months": 33
      },
      {
        "name": "Photoshop",
        "proficiency": "intermediate",
        "endorsements": 8,
        "duration_months": 24
      },
      {
        "name": "TTS",
        "proficiency": "advanced",
        "endorsements": 56,
        "duration_months": 60
      },
      {
        "name": "LoRA",
        "proficiency": "intermediate",
        "endorsements": 0,
        "duration_months": 28
      },
      {
        "name": "Apache Beam",
        "proficiency": "intermediate",
        "endorsements": 4,
        "duration_months": 9
      },
      {
        "name": "AWS",
        "proficiency": "beginner",
        "endorsements": 5,
        "duration_months": 8
      },
      {
        "name": "Flask",
        "proficiency": "beginner",
        "endorsements": 15,
        "duration_months": 15
      },
      {
        "name": "BentoML",
        "proficiency": "intermediate",
        "endorsements": 3,
        "duration_months": 36
      },
      {
        "name": "Milvus",
        "proficiency": "advanced",
        "endorsements": 40,
        "duration_months": 35
      },
      {
        "name": "GANs",
        "proficiency": "advanced",
        "endorsements": 12,
        "duration_months": 19
      },
      {
        "name": "Statistical Modeling",
        "proficiency": "intermediate",
        "endorsements": 9,
        "duration_months": 8
      },
      {
        "name": "GCP",
        "proficiency": "beginner",
        "endorsements": 7,
        "duration_months": 2
      }
    ],
    "certifications": [],
    "languages": [
      {
        "language": "English",
        "proficiency": "professional"
      },
      {
        "language": "Hindi",
        "proficiency": "conversational"
      }
    ],
    "redrob_signals": {
      "profile_completeness_score": 86.9,
      "signup_date": "2025-10-16",
      "last_active_date": "2026-05-20",
      "open_to_work_flag": true,
      "profile_views_received_30d": 23,
      "applications_submitted_30d": 2,
      "recruiter_response_rate": 0.34,
      "avg_response_time_hours": 177.8,
      "skill_assessment_scores": {
        "NLP": 38.8,
        "Image Classification": 64.8,
        "Fine-tuning LLMs": 41.6,
        "Speech Recognition": 53.7
      },
      "connection_count": 356,
      "endorsements_received": 35,
      "notice_period_days": 60,
      "expected_salary_range_inr_lpa": {
        "min": 18.7,
        "max": 36.1
      },
      "preferred_work_mode": "onsite",
      "willing_to_relocate": false,
      "github_activity_score": 9.2,
      "search_appearance_30d": 249,
      "saved_by_recruiters_30d": 4,
      "interview_completion_rate": 0.71,
      "offer_acceptance_rate": 0.58,
      "verified_email": true,
      "verified_phone": true,
      "linkedin_connected": false
    }
  },
  {
    "candidate_id": "CAND_0000002",
    "profile": {
      "anonymized_name": "Saanvi Sethi",
      "headline": "Operations Manager | 12.5+ yrs experience",
      "summary": "Professional with 12.5+ years of experience. My professional background is in marketing manager \u2014 I've built and led teams, owned KPIs, and driven business outcomes in this domain. Lately I've been curious about how AI tools could augment my work \u2014 I've experimented with ChatGPT and a few other tools for productivity and content creation, and I think the space is exciting. Open to roles where I can apply my domain expertise alongside emerging AI capabilities.",
      "location": "Chennai, Tamil Nadu",
      "country": "India",
      "years_of_experience": 12.5,
      "current_title": "Operations Manager",
      "current_company": "Wipro",
      "current_company_size": "10001+",
      "current_industry": "IT Services"
    },
    "career_history": [
      {
        "company": "Wipro",
        "title": "Operations Manager",
        "start_date": "2022-11-14",
        "end_date": null,
        "duration_months": 43,
        "is_current": true,
        "industry": "IT Services",
        "company_size": "10001+",
        "description": "Customer support team lead at a SaaS product. Managed a team of 8 support agents handling tier-1 and tier-2 tickets; owned the escalation process to engineering and the customer-feedback loop to product. Built out the support knowledge base and the agent training program. Strong on the people-management side and the process side; lighter on technical depth beyond product expertise."
      },
      {
        "company": "Wipro",
        "title": "Operations Manager",
        "start_date": "2021-09-13",
        "end_date": "2022-11-07",
        "duration_months": 14,
        "is_current": false,
        "industry": "IT Services",
        "company_size": "10001+",
        "description": "Mechanical engineering design role at a hardware-product company. Led the design of two product subsystems through full lifecycle: concept, DFM/DFMA review, prototype, production tooling. Comfortable with CAD (SolidWorks, Creo), FEA (ANSYS), and the typical hardware-development cadence. Worked closely with manufacturing partners on production scale-up."
      },
      {
        "company": "Acme Corp",
        "title": "Marketing Manager",
        "start_date": "2017-03-08",
        "end_date": "2021-08-14",
        "duration_months": 54,
        "is_current": false,
        "industry": "Manufacturing",
        "company_size": "201-500",
        "description": "Content writing and SEO strategy for a tech-focused publication. Wrote longform articles on developer tools, cloud platforms, and AI/ML topics \u2014 including some that ranked on the first page of search for high-competition keywords. Managed a freelance writer pool and the editorial calendar. Recent work has been on AI-assisted content production, using LLM tools for research, drafting, and editing while maintaining editorial quality."
      },
      {
        "company": "Dunder Mifflin",
        "title": "Marketing Manager",
        "start_date": "2014-01-23",
        "end_date": "2017-03-08",
        "duration_months": 38,
        "is_current": false,
        "industry": "Paper Products",
        "company_size": "201-500",
        "description": "Brand design and creative direction at a consumer-products company. Owned brand identity (logo, visual system, typography), packaging design, and digital creative across web and social. Led the recent rebrand and managed a small external agency for production work. Comfortable across the Adobe suite, Figma, and the production side of brand and packaging design."
      }
    ],
    "education": [
      {
        "institution": "Local Engineering College",
        "degree": "B.Sc",
        "field_of_study": "Mathematics",
        "start_year": 2007,
        "end_year": 2011,
        "grade": "77%",
        "tier": "tier_4"
      }
    ],
    "skills": [
      {
        "name": "Project Management",
        "proficiency": "intermediate",
        "endorsements": 14,
        "duration_months": 23
      },
      {
        "name": "React",
        "proficiency": "intermediate",
        "endorsements": 6,
        "duration_months": 35
      },
      {
        "name": "Photoshop",
        "proficiency": "intermediate",
        "endorsements": 9,
        "duration_months": 35
      },
      {
        "name": "TypeScript",
        "proficiency": "beginner",
        "endorsements": 2,
        "duration_months": 3
      },
      {
        "name": "Marketing",
        "proficiency": "beginner",
        "endorsements": 9,
        "duration_months": 11
      },
      {
        "name": "Kafka",
        "proficiency": "intermediate",
        "endorsements": 3,
        "duration_months": 16
      },
      {
        "name": "JavaScript",
        "proficiency": "beginner",
        "endorsements": 9,
        "duration_months": 3
      },
      {
        "name": "Feature Engineering",
        "proficiency": "intermediate",
        "endorsements": 11,
        "duration_months": 27
      },
      {
        "name": "GCP",
        "proficiency": "intermediate",
        "endorsements": 7,
        "duration_months": 30
      }
    ],
    "certifications": [],
    "languages": [
      {
        "language": "English",
        "proficiency": "professional"
      },
      {
        "language": "Hindi",
        "proficiency": "conversational"
      }
    ],
    "redrob_signals": {
      "profile_completeness_score": 78.7,
      "signup_date": "2025-07-28",
      "last_active_date": "2025-11-12",
      "open_to_work_flag": true,
      "profile_views_received_30d": 7,
      "applications_submitted_30d": 1,
      "recruiter_response_rate": 0.29,
      "avg_response_time_hours": 171.6,
      "skill_assessment_scores": {},
      "connection_count": 179,
      "endorsements_received": 3,
      "notice_period_days": 60,
      "expected_salary_range_inr_lpa": {
        "min": 8.8,
        "max": 9.0
      },
      "preferred_work_mode": "flexible",
      "willing_to_relocate": false,
      "github_activity_score": -1,
      "search_appearance_30d": 107,
      "saved_by_recruiters_30d": 10,
      "interview_completion_rate": 0.62,
      "offer_acceptance_rate": -1,
      "verified_email": false,
      "verified_phone": false,
      "linkedin_connected": false
    }
  },
  {
    "candidate_id": "CAND_0000003",
    "profile": {
      "anonymized_name": "Yash Agarwal",
      "headline": "Customer Support | 1.1+ yrs experience",
      "summary": "Professional with 1.1+ years of experience. I've spent my career in marketing manager roles, mostly focused on driving outcomes through process, people, and customer relationships. Lately I've been curious about how AI tools could augment my work \u2014 I've experimented with ChatGPT and a few other tools for productivity and content creation, and I think the space is exciting. Open to roles where I can apply my domain expertise alongside emerging AI capabilities.",
      "location": "Austin",
      "country": "USA",
      "years_of_experience": 1.1,
      "current_title": "Customer Support",
      "current_company": "TCS",
      "current_company_size": "10001+",
      "current_industry": "IT Services"
    },
    "career_history": [
      {
        "company": "TCS",
        "title": "Customer Support",
        "start_date": "2025-05-02",
        "end_date": null,
        "duration_months": 13,
        "is_current": true,
        "industry": "IT Services",
        "company_size": "10001+",
        "description": "Business analyst at a consulting firm, working primarily with retail and CPG clients. Conducted business diagnostics, process re-engineering work, and digital transformation strategy projects. Strong on stakeholder management, structured problem-solving, and the typical consulting toolkit (slide-craft, Excel modeling, executive communication). Recent project work involved AI-strategy advisory but my own technical depth in AI is limited."
      }
    ],
    "education": [
      {
        "institution": "Local Engineering College",
        "degree": "M.E.",
        "field_of_study": "Chemical Engineering",
        "start_year": 2005,
        "end_year": 2010,
        "grade": "6.82 CGPA",
        "tier": "tier_4"
      },
      {
        "institution": "Chandigarh University",
        "degree": "M.Sc",
        "field_of_study": "Information Technology",
        "start_year": 2017,
        "end_year": 2021,
        "grade": "87%",
        "tier": "tier_3"
      }
    ],
    "skills": [
      {
        "name": "Angular",
        "proficiency": "intermediate",
        "endorsements": 13,
        "duration_months": 10
      },
      {
        "name": "SEO",
        "proficiency": "beginner",
        "endorsements": 11,
        "duration_months": 11
      },
      {
        "name": "Excel",
        "proficiency": "intermediate",
        "endorsements": 2,
        "duration_months": 15
      },
      {
        "name": "Accounting",
        "proficiency": "beginner",
        "endorsements": 7,
        "duration_months": 18
      },
      {
        "name": "Kubernetes",
        "proficiency": "intermediate",
        "endorsements": 0,
        "duration_months": 34
      },
      {
        "name": "Databricks",
        "proficiency": "beginner",
        "endorsements": 14,
        "duration_months": 18
      }
    ],
    "certifications": [],
    "languages": [
      {
        "language": "English",
        "proficiency": "professional"
      },
      {
        "language": "Hindi",
        "proficiency": "professional"
      }
    ],
    "redrob_signals": {
      "profile_completeness_score": 31.9,
      "signup_date": "2024-08-02",
      "last_active_date": "2026-03-21",
      "open_to_work_flag": false,
      "profile_views_received_30d": 1,
      "applications_submitted_30d": 9,
      "recruiter_response_rate": 0.46,
      "avg_response_time_hours": 119.4,
      "skill_assessment_scores": {},
      "connection_count": 19,
      "endorsements_received": 46,
      "notice_period_days": 150,
      "expected_salary_range_inr_lpa": {
        "min": 11.2,
        "max": 18.1
      },
      "preferred_work_mode": "hybrid",
      "willing_to_relocate": false,
      "github_activity_score": -1,
      "search_appearance_30d": 28,
      "saved_by_recruiters_30d": 4,
      "interview_completion_rate": 0.86,
      "offer_acceptance_rate": -1,
      "verified_email": true,
      "verified_phone": false,
      "linkedin_connected": false
    }
  },
  {
    "candidate_id": "CAND_0000004",
    "profile": {
      "anonymized_name": "Anil Bose",
      "headline": "Marketing Manager | Driving business outcomes",
      "summary": "Professional with 3.8+ years of experience. My professional background is in marketing manager \u2014 I've built and led teams, owned KPIs, and driven business outcomes in this domain. Lately I've been curious about how AI tools could augment my work \u2014 I've experimented with ChatGPT and a few other tools for productivity and content creation, and I think the space is exciting. Open to roles where I can apply my domain expertise alongside emerging AI capabilities.",
      "location": "Sydney",
      "country": "Australia",
      "years_of_experience": 3.8,
      "current_title": "Marketing Manager",
      "current_company": "Dunder Mifflin",
      "current_company_size": "201-500",
      "current_industry": "Paper Products"
    },
    "career_history": [
      {
        "company": "Dunder Mifflin",
        "title": "Marketing Manager",
        "start_date": "2025-04-02",
        "end_date": null,
        "duration_months": 14,
        "is_current": true,
        "industry": "Paper Products",
        "company_size": "201-500",
        "description": "Mechanical engineering design role at a hardware-product company. Led the design of two product subsystems through full lifecycle: concept, DFM/DFMA review, prototype, production tooling. Comfortable with CAD (SolidWorks, Creo), FEA (ANSYS), and the typical hardware-development cadence. Worked closely with manufacturing partners on production scale-up."
      },
      {
        "company": "Infosys",
        "title": "Operations Manager",
        "start_date": "2023-07-28",
        "end_date": "2025-03-19",
        "duration_months": 20,
        "is_current": false,
        "industry": "IT Services",
        "company_size": "10001+",
        "description": "Content writing and SEO strategy for a tech-focused publication. Wrote longform articles on developer tools, cloud platforms, and AI/ML topics \u2014 including some that ranked on the first page of search for high-competition keywords. Managed a freelance writer pool and the editorial calendar. Recent work has been on AI-assisted content production, using LLM tools for research, drafting, and editing while maintaining editorial quality."
      },
      {
        "company": "Globex Inc",
        "title": "Business Analyst",
        "start_date": "2022-08-02",
        "end_date": "2023-05-29",
        "duration_months": 10,
        "is_current": false,
        "industry": "Manufacturing",
        "company_size": "501-1000",
        "description": "Operations management role at a logistics company. Owned daily fulfillment operations across 3 warehouses, managing a team of 80 across receiving, picking, packing, and outbound. Built and tracked the operational KPIs (on-time fulfillment, accuracy, cost per order) and led the continuous improvement initiatives that drove a 22% productivity gain over 18 months."
      }
    ],
    "education": [
      {
        "institution": "Local Engineering College",
        "degree": "B.Tech",
        "field_of_study": "Machine Learning",
        "start_year": 2015,
        "end_year": 2019,
        "grade": "7.72 CGPA",
        "tier": "tier_4"
      },
      {
        "institution": "Lovely Professional University",
        "degree": "Ph.D",
        "field_of_study": "Electronics",
        "start_year": 2013,
        "end_year": 2016,
        "grade": "7.61 CGPA",
        "tier": "tier_3"
      }
    ],
    "skills": [
      {
        "name": "Node.js",
        "proficiency": "intermediate",
        "endorsements": 1,
        "duration_months": 20
      },
      {
        "name": "Content Writing",
        "proficiency": "beginner",
        "endorsements": 7,
        "duration_months": 3
      },
      {
        "name": "Redux",
        "proficiency": "beginner",
        "endorsements": 14,
        "duration_months": 8
      },
      {
        "name": "Airflow",
        "proficiency": "intermediate",
        "endorsements": 11,
        "duration_months": 27
      },
      {
        "name": "GraphQL",
        "proficiency": "beginner",
        "endorsements": 13,
        "duration_months": 2
      },
      {
        "name": "Object Detection",
        "proficiency": "intermediate",
        "endorsements": 3,
        "duration_months": 17
      },
      {
        "name": "Webpack",
        "proficiency": "beginner",
        "endorsements": 10,
        "duration_months": 7
      },
      {
        "name": "Six Sigma",
        "proficiency": "beginner",
        "endorsements": 1,
        "duration_months": 12
      },
      {
        "name": "SAP",
        "proficiency": "intermediate",
        "endorsements": 6,
        "duration_months": 9
      },
      {
        "name": "JavaScript",
        "proficiency": "intermediate",
        "endorsements": 14,
        "duration_months": 29
      }
    ],
    "certifications": [
      {
        "name": "AWS Certified Cloud Practitioner",
        "issuer": "AWS",
        "year": 2025
      },
      {
        "name": "Scrum Master Certified",
        "issuer": "Scrum Alliance",
        "year": 2025
      }
    ],
    "languages": [
      {
        "language": "English",
        "proficiency": "professional"
      },
      {
        "language": "Hindi",
        "proficiency": "professional"
      }
    ],
    "redrob_signals": {
      "profile_completeness_score": 28.5,
      "signup_date": "2025-07-21",
      "last_active_date": "2026-03-25",
      "open_to_work_flag": false,
      "profile_views_received_30d": 3,
      "applications_submitted_30d": 9,
      "recruiter_response_rate": 0.26,
      "avg_response_time_hours": 104.1,
      "skill_assessment_scores": {},
      "connection_count": 485,
      "endorsements_received": 22,
      "notice_period_days": 120,
      "expected_salary_range_inr_lpa": {
        "min": 4.6,
        "max": 6.7
      },
      "preferred_work_mode": "onsite",
      "willing_to_relocate": true,
      "github_activity_score": -1,
      "search_appearance_30d": 5,
      "saved_by_recruiters_30d": 8,
      "interview_completion_rate": 0.35,
      "offer_acceptance_rate": -1,
      "verified_email": true,
      "verified_phone": true,
      "linkedin_connected": true
    }
  },
  {
    "candidate_id": "CAND_0000005",
    "profile": {
      "anonymized_name": "Aisha Sethi",
      "headline": "Accountant | Helping teams scale",
      "summary": "Professional with 11.0+ years of experience. I've spent my career in marketing manager roles, mostly focused on driving outcomes through process, people, and customer relationships. Lately I've been curious about how AI tools could augment my work \u2014 I've experimented with ChatGPT and a few other tools for productivity and content creation, and I think the space is exciting. Open to roles where I can apply my domain expertise alongside emerging AI capabilities.",
      "location": "Gurgaon, Haryana",
      "country": "India",
      "years_of_experience": 11.0,
      "current_title": "Accountant",
      "current_company": "Stark Industries",
      "current_company_size": "1001-5000",
      "current_industry": "Manufacturing"
    },
    "career_history": [
      {
        "company": "Stark Industries",
        "title": "Accountant",
        "start_date": "2022-02-17",
        "end_date": null,
        "duration_months": 52,
        "is_current": true,
        "industry": "Manufacturing",
        "company_size": "1001-5000",
        "description": "Business analyst at a consulting firm, working primarily with retail and CPG clients. Conducted business diagnostics, process re-engineering work, and digital transformation strategy projects. Strong on stakeholder management, structured problem-solving, and the typical consulting toolkit (slide-craft, Excel modeling, executive communication). Recent project work involved AI-strategy advisory but my own technical depth in AI is limited."
      },
      {
        "company": "Wipro",
        "title": "HR Manager",
        "start_date": "2018-05-25",
        "end_date": "2022-02-03",
        "duration_months": 45,
        "is_current": false,
        "industry": "IT Services",
        "company_size": "10001+",
        "description": "Senior accounting role at a mid-sized company \u2014 month-end close, financial reporting, statutory compliance (GAAP / Ind-AS), and tax filings. Owned the GL, fixed-asset register, and the audit-readiness function. Managed a team of 3 staff accountants. Built strong process discipline around the close cycle, reducing close time from 12 days to 7 over the last two years."
      },
      {
        "company": "Initech",
        "title": "HR Manager",
        "start_date": "2016-06-04",
        "end_date": "2018-05-25",
        "duration_months": 24,
        "is_current": false,
        "industry": "Software",
        "company_size": "51-200",
        "description": "Business analyst at a consulting firm, working primarily with retail and CPG clients. Conducted business diagnostics, process re-engineering work, and digital transformation strategy projects. Strong on stakeholder management, structured problem-solving, and the typical consulting toolkit (slide-craft, Excel modeling, executive communication). Recent project work involved AI-strategy advisory but my own technical depth in AI is limited."
      },
      {
        "company": "TCS",
        "title": "Accountant",
        "start_date": "2015-09-08",
        "end_date": "2016-06-04",
        "duration_months": 9,
        "is_current": false,
        "industry": "IT Services",
        "company_size": "10001+",
        "description": "Customer support team lead at a SaaS product. Managed a team of 8 support agents handling tier-1 and tier-2 tickets; owned the escalation process to engineering and the customer-feedback loop to product. Built out the support knowledge base and the agent training program. Strong on the people-management side and the process side; lighter on technical depth beyond product expertise."
      }
    ],
    "education": [
      {
        "institution": "Chandigarh University",
        "degree": "M.Sc",
        "field_of_study": "Information Technology",
        "start_year": 2007,
        "end_year": 2012,
        "grade": "87%",
        "tier": "tier_3"
      }
    ],
    "skills": [
      {
        "name": "SQL",
        "proficiency": "beginner",
        "endorsements": 12,
        "duration_months": 12
      },
      {
        "name": "PowerPoint",
        "proficiency": "beginner",
        "endorsements": 6,
        "duration_months": 14
      },
      {
        "name": "Photoshop",
        "proficiency": "beginner",
        "endorsements": 4,
        "duration_months": 18
      },
      {
        "name": "Tailwind",
        "proficiency": "intermediate",
        "endorsements": 15,
        "duration_months": 35
      },
      {
        "name": "Apache Flink",
        "proficiency": "intermediate",
        "endorsements": 1,
        "duration_months": 30
      },
      {
        "name": "Image Classification",
        "proficiency": "advanced",
        "endorsements": 50,
        "duration_months": 38
      }
    ],
    "certifications": [],
    "languages": [
      {
        "language": "English",
        "proficiency": "professional"
      },
      {
        "language": "Hindi",
        "proficiency": "conversational"
      }
    ],
    "redrob_signals": {
      "profile_completeness_score": 84.6,
      "signup_date": "2023-10-07",
      "last_active_date": "2025-10-01",
      "open_to_work_flag": true,
      "profile_views_received_30d": 12,
      "applications_submitted_30d": 2,
      "recruiter_response_rate": 0.37,
      "avg_response_time_hours": 116.7,
      "skill_assessment_scores": {},
      "connection_count": 300,
      "endorsements_received": 14,
      "notice_period_days": 30,
      "expected_salary_range_inr_lpa": {
        "min": 12.4,
        "max": 19.7
      },
      "preferred_work_mode": "hybrid",
      "willing_to_relocate": true,
      "github_activity_score": -1,
      "search_appearance_30d": 67,
      "saved_by_recruiters_30d": 1,
      "interview_completion_rate": 0.74,
      "offer_acceptance_rate": -1,
      "verified_email": false,
      "verified_phone": true,
      "linkedin_connected": true
    }
  },
  {
    "candidate_id": "CAND_0000006",
    "profile": {
      "anonymized_name": "Rajesh Desai",
      "headline": "Business Analyst | 6.0+ yrs experience",
      "summary": "Professional with 6.0+ years of experience. I've spent my career in marketing manager roles, mostly focused on driving outcomes through process, people, and customer relationships. Lately I've been curious about how AI tools could augment my work \u2014 I've experimented with ChatGPT and a few other tools for productivity and content creation, and I think the space is exciting. Open to roles where I can apply my domain expertise alongside emerging AI capabilities.",
      "location": "Austin",
      "country": "USA",
      "years_of_experience": 6.0,
      "current_title": "Business Analyst",
      "current_company": "Wayne Enterprises",
      "current_company_size": "10001+",
      "current_industry": "Conglomerate"
    },
    "career_history": [
      {
        "company": "Wayne Enterprises",
        "title": "Business Analyst",
        "start_date": "2023-09-10",
        "end_date": null,
        "duration_months": 33,
        "is_current": true,
        "industry": "Conglomerate",
        "company_size": "10001+",
        "description": "Senior accounting role at a mid-sized company \u2014 month-end close, financial reporting, statutory compliance (GAAP / Ind-AS), and tax filings. Owned the GL, fixed-asset register, and the audit-readiness function. Managed a team of 3 staff accountants. Built strong process discipline around the close cycle, reducing close time from 12 days to 7 over the last two years."
      },
      {
        "company": "Pied Piper",
        "title": "Mechanical Engineer",
        "start_date": "2020-07-27",
        "end_date": "2023-09-10",
        "duration_months": 38,
        "is_current": false,
        "industry": "Software",
        "company_size": "11-50",
        "description": "Business analyst at a consulting firm, working primarily with retail and CPG clients. Conducted business diagnostics, process re-engineering work, and digital transformation strategy projects. Strong on stakeholder management, structured problem-solving, and the typical consulting toolkit (slide-craft, Excel modeling, executive communication). Recent project work involved AI-strategy advisory but my own technical depth in AI is limited."
      }
    ],
    "education": [
      {
        "institution": "Lovely Professional University",
        "degree": "B.Sc",
        "field_of_study": "Artificial Intelligence",
        "start_year": 2005,
        "end_year": 2008,
        "grade": "9.26 CGPA",
        "tier": "tier_3"
      }
    ],
    "skills": [
      {
        "name": "Content Writing",
        "proficiency": "intermediate",
        "endorsements": 0,
        "duration_months": 33
      },
      {
        "name": "SEO",
        "proficiency": "intermediate",
        "endorsements": 13,
        "duration_months": 31
      },
      {
        "name": "Redux",
        "proficiency": "beginner",
        "endorsements": 15,
        "duration_months": 12
      },
      {
        "name": "SQL",
        "proficiency": "beginner",
        "endorsements": 9,
        "duration_months": 11
      },
      {
        "name": "Sales",
        "proficiency": "intermediate",
        "endorsements": 5,
        "duration_months": 27
      },
      {
        "name": "gRPC",
        "proficiency": "beginner",
        "endorsements": 8,
        "duration_months": 3
      },
      {
        "name": "Django",
        "proficiency": "intermediate",
        "endorsements": 3,
        "duration_months": 11
      },
      {
        "name": "Terraform",
        "proficiency": "beginner",
        "endorsements": 4,
        "duration_months": 13
      }
    ],
    "certifications": [],
    "languages": [
      {
        "language": "English",
        "proficiency": "professional"
      },
      {
        "language": "Hindi",
        "proficiency": "conversational"
      }
    ],
    "redrob_signals": {
      "profile_completeness_score": 29.7,
      "signup_date": "2026-04-26",
      "last_active_date": "2026-02-28",
      "open_to_work_flag": false,
      "profile_views_received_30d": 53,
      "applications_submitted_30d": 8,
      "recruiter_response_rate": 0.12,
      "avg_response_time_hours": 172.1,
      "skill_assessment_scores": {},
      "connection_count": 389,
      "endorsements_received": 29,
      "notice_period_days": 150,
      "expected_salary_range_inr_lpa": {
        "min": 7.7,
        "max": 11.7
      },
      "preferred_work_mode": "remote",
      "willing_to_relocate": true,
      "github_activity_score": -1,
      "search_appearance_30d": 131,
      "saved_by_recruiters_30d": 9,
      "interview_completion_rate": 0.57,
      "offer_acceptance_rate": -1,
      "verified_email": true,
      "verified_phone": true,
      "linkedin_connected": false
    }
  },
  {
    "candidate_id": "CAND_0000007",
    "profile": {
      "anonymized_name": "Vihaan Bose",
      "headline": "Civil Engineer | 5.5+ yrs experience",
      "summary": "Professional with 5.5+ years of experience. My professional background is in marketing manager \u2014 I've built and led teams, owned KPIs, and driven business outcomes in this domain. Lately I've been curious about how AI tools could augment my work \u2014 I've experimented with ChatGPT and a few other tools for productivity and content creation, and I think the space is exciting. Open to roles where I can apply my domain expertise alongside emerging AI capabilities.",
      "location": "Gurgaon, Haryana",
      "country": "India",
      "years_of_experience": 5.5,
      "current_title": "Civil Engineer",
      "current_company": "Wipro",
      "current_company_size": "10001+",
      "current_industry": "IT Services"
    },
    "career_history": [
      {
        "company": "Wipro",
        "title": "Civil Engineer",
        "start_date": "2023-04-13",
        "end_date": null,
        "duration_months": 38,
        "is_current": true,
        "industry": "IT Services",
        "company_size": "10001+",
        "description": "Brand design and creative direction at a consumer-products company. Owned brand identity (logo, visual system, typography), packaging design, and digital creative across web and social. Led the recent rebrand and managed a small external agency for production work. Comfortable across the Adobe suite, Figma, and the production side of brand and packaging design."
      },
      {
        "company": "Initech",
        "title": "Mechanical Engineer",
        "start_date": "2021-01-16",
        "end_date": "2023-04-06",
        "duration_months": 27,
        "is_current": false,
        "industry": "Software",
        "company_size": "51-200",
        "description": "Customer support team lead at a SaaS product. Managed a team of 8 support agents handling tier-1 and tier-2 tickets; owned the escalation process to engineering and the customer-feedback loop to product. Built out the support knowledge base and the agent training program. Strong on the people-management side and the process side; lighter on technical depth beyond product expertise."
      }
    ],
    "education": [
      {
        "institution": "SRM University",
        "degree": "M.E.",
        "field_of_study": "Data Science",
        "start_year": 2009,
        "end_year": 2013,
        "grade": "8.28 CGPA",
        "tier": "tier_2"
      }
    ],
    "skills": [
      {
        "name": "Content Writing",
        "proficiency": "beginner",
        "endorsements": 12,
        "duration_months": 14
      },
      {
        "name": "MongoDB",
        "proficiency": "intermediate",
        "endorsements": 13,
        "duration_months": 9
      },
      {
        "name": "Sales",
        "proficiency": "intermediate",
        "endorsements": 0,
        "duration_months": 27
      },
      {
        "name": "Spark",
        "proficiency": "beginner",
        "endorsements": 14,
        "duration_months": 14
      },
      {
        "name": "Scrum",
        "proficiency": "beginner",
        "endorsements": 4,
        "duration_months": 18
      },
      {
        "name": "Apache Beam",
        "proficiency": "beginner",
        "endorsements": 4,
        "duration_months": 3
      },
      {
        "name": "Illustrator",
        "proficiency": "beginner",
        "endorsements": 14,
        "duration_months": 2
      }
    ],
    "certifications": [],
    "languages": [
      {
        "language": "English",
        "proficiency": "professional"
      },
      {
        "language": "Hindi",
        "proficiency": "professional"
      }
    ],
    "redrob_signals": {
      "profile_completeness_score": 74.6,
      "signup_date": "2025-09-29",
      "last_active_date": "2026-05-25",
      "open_to_work_flag": false,
      "profile_views_received_30d": 2,
      "applications_submitted_30d": 1,
      "recruiter_response_rate": 0.62,
      "avg_response_time_hours": 61.3,
      "skill_assessment_scores": {},
      "connection_count": 122,
      "endorsements_received": 50,
      "notice_period_days": 30,
      "expected_salary_range_inr_lpa": {
        "min": 6.7,
        "max": 14.6
      },
      "preferred_work_mode": "onsite",
      "willing_to_relocate": true,
      "github_activity_score": -1,
      "search_appearance_30d": 104,
      "saved_by_recruiters_30d": 8,
      "interview_completion_rate": 0.47,
      "offer_acceptance_rate": -1,
      "verified_email": true,
      "verified_phone": true,
      "linkedin_connected": true
    }
  },
  {
    "candidate_id": "CAND_0000008",
    "profile": {
      "anonymized_name": "Shaurya Chatterjee",
      "headline": "Operations Manager | 3.6+ yrs experience",
      "summary": "Professional with 3.6+ years of experience. I've spent my career in marketing manager roles, mostly focused on driving outcomes through process, people, and customer relationships. Lately I've been curious about how AI tools could augment my work \u2014 I've experimented with ChatGPT and a few other tools for productivity and content creation, and I think the space is exciting. Open to roles where I can apply my domain expertise alongside emerging AI capabilities.",
      "location": "Noida, Uttar Pradesh",
      "country": "India",
      "years_of_experience": 3.6,
      "current_title": "Operations Manager",
      "current_company": "Wipro",
      "current_company_size": "10001+",
      "current_industry": "IT Services"
    },
    "career_history": [
      {
        "company": "Wipro",
        "title": "Operations Manager",
        "start_date": "2022-11-14",
        "end_date": null,
        "duration_months": 43,
        "is_current": true,
        "industry": "IT Services",
        "company_size": "10001+",
        "description": "Marketing leadership role at a B2B SaaS company. Owned the demand-generation function \u2014 content marketing, paid acquisition, SEO, email nurture. Built and managed a team of 5 across content, performance marketing, and marketing operations. Worked closely with sales on lead-quality definitions and the SDR-handoff process. Recent focus has been on account-based marketing for our enterprise segment."
      }
    ],
    "education": [
      {
        "institution": "Anna University",
        "degree": "B.Tech",
        "field_of_study": "Data Science",
        "start_year": 2008,
        "end_year": 2012,
        "grade": "8.60 CGPA",
        "tier": "tier_2"
      },
      {
        "institution": "SRM University",
        "degree": "M.Sc",
        "field_of_study": "Computer Engineering",
        "start_year": 2009,
        "end_year": 2013,
        "grade": "67%",
        "tier": "tier_2"
      }
    ],
    "skills": [
      {
        "name": "Java",
        "proficiency": "intermediate",
        "endorsements": 2,
        "duration_months": 32
      },
      {
        "name": "BigQuery",
        "proficiency": "beginner",
        "endorsements": 5,
        "duration_months": 9
      },
      {
        "name": "Spark",
        "proficiency": "beginner",
        "endorsements": 4,
        "duration_months": 6
      },
      {
        "name": "Accounting",
        "proficiency": "beginner",
        "endorsements": 8,
        "duration_months": 3
      },
      {
        "name": "Kubernetes",
        "proficiency": "intermediate",
        "endorsements": 2,
        "duration_months": 9
      },
      {
        "name": "TypeScript",
        "proficiency": "intermediate",
        "endorsements": 14,
        "duration_months": 11
      },
      {
        "name": "Rust",
        "proficiency": "intermediate",
        "endorsements": 12,
        "duration_months": 16
      },
      {
        "name": "HTML",
        "proficiency": "beginner",
        "endorsements": 8,
        "duration_months": 11
      }
    ],
    "certifications": [
      {
        "name": "Six Sigma Green Belt",
        "issuer": "ASQ",
        "year": 2018
      }
    ],
    "languages": [
      {
        "language": "English",
        "proficiency": "professional"
      },
      {
        "language": "Hindi",
        "proficiency": "professional"
      }
    ],
    "redrob_signals": {
      "profile_completeness_score": 63.0,
      "signup_date": "2022-06-26",
      "last_active_date": "2025-12-13",
      "open_to_work_flag": false,
      "profile_views_received_30d": 28,
      "applications_submitted_30d": 5,
      "recruiter_response_rate": 0.42,
      "avg_response_time_hours": 98.4,
      "skill_assessment_scores": {},
      "connection_count": 285,
      "endorsements_received": 7,
      "notice_period_days": 90,
      "expected_salary_range_inr_lpa": {
        "min": 6.6,
        "max": 17.2
      },
      "preferred_work_mode": "onsite",
      "willing_to_relocate": false,
      "github_activity_score": -1,
      "search_appearance_30d": 91,
      "saved_by_recruiters_30d": 0,
      "interview_completion_rate": 0.74,
      "offer_acceptance_rate": -1,
      "verified_email": true,
      "verified_phone": false,
      "linkedin_connected": false
    }
  },
  {
    "candidate_id": "CAND_0000009",
    "profile": {
      "anonymized_name": "Amit Shah",
      "headline": "Mechanical Engineer | Driving business outcomes",
      "summary": "Professional with 11.0+ years of experience. My professional background is in marketing manager \u2014 I've built and led teams, owned KPIs, and driven business outcomes in this domain. Lately I've been curious about how AI tools could augment my work \u2014 I've experimented with ChatGPT and a few other tools for productivity and content creation, and I think the space is exciting. Open to roles where I can apply my domain expertise alongside emerging AI capabilities.",
      "location": "New York",
      "country": "USA",
      "years_of_experience": 11.0,
      "current_title": "Mechanical Engineer",
      "current_company": "Dunder Mifflin",
      "current_company_size": "201-500",
      "current_industry": "Paper Products"
    },
    "career_history": [
      {
        "company": "Dunder Mifflin",
        "title": "Mechanical Engineer",
        "start_date": "2022-10-15",
        "end_date": null,
        "duration_months": 44,
        "is_current": true,
        "industry": "Paper Products",
        "company_size": "201-500",
        "description": "Business analyst at a consulting firm, working primarily with retail and CPG clients. Conducted business diagnostics, process re-engineering work, and digital transformation strategy projects. Strong on stakeholder management, structured problem-solving, and the typical consulting toolkit (slide-craft, Excel modeling, executive communication). Recent project work involved AI-strategy advisory but my own technical depth in AI is limited."
      },
      {
        "company": "Wipro",
        "title": "Content Writer",
        "start_date": "2021-02-22",
        "end_date": "2022-10-15",
        "duration_months": 20,
        "is_current": false,
        "industry": "IT Services",
        "company_size": "10001+",
        "description": "Marketing leadership role at a B2B SaaS company. Owned the demand-generation function \u2014 content marketing, paid acquisition, SEO, email nurture. Built and managed a team of 5 across content, performance marketing, and marketing operations. Worked closely with sales on lead-quality definitions and the SDR-handoff process. Recent focus has been on account-based marketing for our enterprise segment."
      },
      {
        "company": "Stark Industries",
        "title": "Customer Support",
        "start_date": "2017-02-13",
        "end_date": "2021-02-22",
        "duration_months": 49,
        "is_current": false,
        "industry": "Manufacturing",
        "company_size": "1001-5000",
        "description": "Mechanical engineering design role at a hardware-product company. Led the design of two product subsystems through full lifecycle: concept, DFM/DFMA review, prototype, production tooling. Comfortable with CAD (SolidWorks, Creo), FEA (ANSYS), and the typical hardware-development cadence. Worked closely with manufacturing partners on production scale-up."
      },
      {
        "company": "Acme Corp",
        "title": "Project Manager",
        "start_date": "2015-08-23",
        "end_date": "2017-02-13",
        "duration_months": 18,
        "is_current": false,
        "industry": "Manufacturing",
        "company_size": "201-500",
        "description": "Enterprise sales of cloud software solutions into the mid-market segment. Carried a $1.8M ARR quota and consistently delivered against it across the last three years. Owned the full sales cycle: prospecting, discovery, technical evaluation (with SE support), commercial negotiation, and close. Strong on consultative selling for technical buyers; comfortable engaging with both engineering and finance stakeholders."
      }
    ],
    "education": [
      {
        "institution": "KIIT University",
        "degree": "B.Tech",
        "field_of_study": "Electronics",
        "start_year": 2009,
        "end_year": 2014,
        "grade": "7.89 CGPA",
        "tier": "tier_3"
      }
    ],
    "skills": [
      {
        "name": "Snowflake",
        "proficiency": "intermediate",
        "endorsements": 5,
        "duration_months": 8
      },
      {
        "name": "gRPC",
        "proficiency": "beginner",
        "endorsements": 6,
        "duration_months": 12
      },
      {
        "name": "JavaScript",
        "proficiency": "intermediate",
        "endorsements": 0,
        "duration_months": 23
      },
      {
        "name": "OpenCV",
        "proficiency": "intermediate",
        "endorsements": 12,
        "duration_months": 36
      },
      {
        "name": "Go",
        "proficiency": "intermediate",
        "endorsements": 12,
        "duration_months": 20
      },
      {
        "name": "PowerPoint",
        "proficiency": "intermediate",
        "endorsements": 1,
        "duration_months": 10
      }
    ],
    "certifications": [],
    "languages": [
      {
        "language": "English",
        "proficiency": "native"
      },
      {
        "language": "Hindi",
        "proficiency": "professional"
      }
    ],
    "redrob_signals": {
      "profile_completeness_score": 39.6,
      "signup_date": "2025-10-19",
      "last_active_date": "2026-01-27",
      "open_to_work_flag": false,
      "profile_views_received_30d": 50,
      "applications_submitted_30d": 8,
      "recruiter_response_rate": 0.53,
      "avg_response_time_hours": 202.0,
      "skill_assessment_scores": {},
      "connection_count": 516,
      "endorsements_received": 34,
      "notice_period_days": 150,
      "expected_salary_range_inr_lpa": {
        "min": 16.0,
        "max": 7.3
      },
      "preferred_work_mode": "remote",
      "willing_to_relocate": false,
      "github_activity_score": -1,
      "search_appearance_30d": 74,
      "saved_by_recruiters_30d": 1,
      "interview_completion_rate": 0.54,
      "offer_acceptance_rate": 0.48,
      "verified_email": true,
      "verified_phone": true,
      "linkedin_connected": false
    }
  },
  {
    "candidate_id": "CAND_0000010",
    "profile": {
      "anonymized_name": "Aarav Kapoor",
      "headline": "Data Engineer | Data pipelines & analytics",
      "summary": "Software / data professional with 4.6 years of experience building data pipelines, backend systems, and analytics infrastructure. Most of my work has been data pipelines and analytics infrastructure; I've shipped a couple of small ML features but it's not the core of my day. My toolkit is solid on the data engineering side \u2014 Python, SQL, Spark, Airflow, warehouse design \u2014 and I've completed a couple of self-directed ML projects (Kaggle competitions, side projects fine-tuning small models). Interested in transitioning toward more AI/ML-focused work, ideally at a company where I can leverage my existing data-infra skills while learning modern ML practice.",
      "location": "London",
      "country": "UK",
      "years_of_experience": 4.6,
      "current_title": "Data Engineer",
      "current_company": "Ola",
      "current_company_size": "5001-10000",
      "current_industry": "Transportation"
    },
    "career_history": [
      {
        "company": "Ola",
        "title": "Data Engineer",
        "start_date": "2021-11-19",
        "end_date": null,
        "duration_months": 55,
        "is_current": true,
        "industry": "Transportation",
        "company_size": "5001-10000",
        "description": "Mixed data science and analytics-engineering role at a marketing-analytics startup. Spent maybe 30% of my time on lightweight ML (clustering, classification, churn prediction in sklearn/XGBoost) and 70% on data infrastructure and dashboards. Comfortable with the modeling work but I wouldn't call myself an ML specialist. Built our experimentation framework that supports the product team's A/B tests."
      }
    ],
    "education": [
      {
        "institution": "Generic State University",
        "degree": "B.E.",
        "field_of_study": "Mathematics",
        "start_year": 2007,
        "end_year": 2011,
        "grade": "85%",
        "tier": "tier_4"
      },
      {
        "institution": "Local Engineering College",
        "degree": "M.S.",
        "field_of_study": "Computer Engineering",
        "start_year": 2013,
        "end_year": 2018,
        "grade": "7.73 CGPA",
        "tier": "tier_4"
      }
    ],
    "skills": [
      {
        "name": "GCP",
        "proficiency": "beginner",
        "endorsements": 7,
        "duration_months": 8
      },
      {
        "name": "Spring Boot",
        "proficiency": "beginner",
        "endorsements": 3,
        "duration_months": 2
      },
      {
        "name": "Kubeflow",
        "proficiency": "intermediate",
        "endorsements": 11,
        "duration_months": 19
      },
      {
        "name": "Java",
        "proficiency": "intermediate",
        "endorsements": 12,
        "duration_months": 19
      },
      {
        "name": "GANs",
        "proficiency": "advanced",
        "endorsements": 58,
        "duration_months": 57
      },
      {
        "name": "Figma",
        "proficiency": "beginner",
        "endorsements": 4,
        "duration_months": 3
      },
      {
        "name": "Elasticsearch",
        "proficiency": "intermediate",
        "endorsements": 15,
        "duration_months": 17
      },
      {
        "name": "OpenCV",
        "proficiency": "advanced",
        "endorsements": 0,
        "duration_months": 24
      },
      {
        "name": "CNN",
        "proficiency": "intermediate",
        "endorsements": 15,
        "duration_months": 8
      },
      {
        "name": "Azure",
        "proficiency": "beginner",
        "endorsements": 7,
        "duration_months": 11
      },
      {
        "name": "Prompt Engineering",
        "proficiency": "advanced",
        "endorsements": 42,
        "duration_months": 35
      },
      {
        "name": "Object Detection",
        "proficiency": "advanced",
        "endorsements": 55,
        "duration_months": 58
      },
      {
        "name": "MLOps",
        "proficiency": "intermediate",
        "endorsements": 3,
        "duration_months": 10
      },
      {
        "name": "Python",
        "proficiency": "intermediate",
        "endorsements": 7,
        "duration_months": 14
      },
      {
        "name": "BM25",
        "proficiency": "advanced",
        "endorsements": 55,
        "duration_months": 55
      },
      {
        "name": "Weights & Biases",
        "proficiency": "advanced",
        "endorsements": 4,
        "duration_months": 21
      },
      {
        "name": "Sales",
        "proficiency": "beginner",
        "endorsements": 5,
        "duration_months": 18
      }
    ],
    "certifications": [],
    "languages": [
      {
        "language": "English",
        "proficiency": "native"
      },
      {
        "language": "Hindi",
        "proficiency": "professional"
      }
    ],
    "redrob_signals": {
      "profile_completeness_score": 81.6,
      "signup_date": "2026-01-09",
      "last_active_date": "2026-04-29",
      "open_to_work_flag": false,
      "profile_views_received_30d": 60,
      "applications_submitted_30d": 13,
      "recruiter_response_rate": 0.4,
      "avg_response_time_hours": 19.0,
      "skill_assessment_scores": {
        "GANs": 53.3,
        "OpenCV": 65.5,
        "Prompt Engineering": 73.8,
        "Object Detection": 81.3
      },
      "connection_count": 712,
      "endorsements_received": 38,
      "notice_period_days": 120,
      "expected_salary_range_inr_lpa": {
        "min": 13.0,
        "max": 32.0
      },
      "preferred_work_mode": "hybrid",
      "willing_to_relocate": false,
      "github_activity_score": 33.7,
      "search_appearance_30d": 256,
      "saved_by_recruiters_30d": 2,
      "interview_completion_rate": 0.53,
      "offer_acceptance_rate": -1,
      "verified_email": true,
      "verified_phone": true,
      "linkedin_connected": false
    }
  },
  {
    "candidate_id": "CAND_0000011",
    "profile": {
      "anonymized_name": "Deepak Desai",
      "headline": "QA Engineer | Cloud & DevOps",
      "summary": "Software engineer with 2.0 years of experience across web, backend, and cloud systems. Strong fundamentals in software development and system design. Most of my work is conventional backend engineering \u2014 APIs, databases, queues, deployments. I've been keeping up with AI/ML at a self-learner level \u2014 taken some online courses, played with the OpenAI and Anthropic APIs, built a small RAG side project \u2014 but I haven't done it in a professional capacity yet. Open to roles where I can either deepen my software engineering work or, if the team is open to it, start contributing to ML-adjacent systems.",
      "location": "Hyderabad, Telangana",
      "country": "India",
      "years_of_experience": 2.0,
      "current_title": "QA Engineer",
      "current_company": "Pied Piper",
      "current_company_size": "11-50",
      "current_industry": "Software"
    },
    "career_history": [
      {
        "company": "Pied Piper",
        "title": "QA Engineer",
        "start_date": "2025-05-02",
        "end_date": null,
        "duration_months": 13,
        "is_current": true,
        "industry": "Software",
        "company_size": "11-50",
        "description": "Android mobile development using Java and (more recently) Kotlin at a consumer-app company. Built and maintained multiple production features including the main shopping flow, push notification system, and the offline-first sync layer. Comfortable with the Android framework, Jetpack components, and the typical patterns (MVVM, Hilt, Coroutines). My career has been entirely on mobile so far; interested in expanding into broader backend or platform engineering."
      },
      {
        "company": "Hooli",
        "title": "QA Engineer",
        "start_date": "2024-06-06",
        "end_date": "2025-04-02",
        "duration_months": 10,
        "is_current": false,
        "industry": "Software",
        "company_size": "1001-5000",
        "description": "Test automation and QA engineering for a fintech product. Built and maintained the end-to-end test suite using Selenium and pytest, plus the load-testing setup using Locust. Worked closely with developers on testability patterns and with product on acceptance criteria. Recent work has been on shifting test responsibility into the dev team \u2014 moving from QA-as-gate to QA-as-coach. Career has been entirely in QA/test engineering."
      }
    ],
    "education": [
      {
        "institution": "Chandigarh University",
        "degree": "B.Tech",
        "field_of_study": "Data Science",
        "start_year": 2014,
        "end_year": 2019,
        "grade": "6.96 CGPA",
        "tier": "tier_3"
      },
      {
        "institution": "Anna University",
        "degree": "B.Sc",
        "field_of_study": "Information Technology",
        "start_year": 2015,
        "end_year": 2020,
        "grade": "9.16 CGPA",
        "tier": "tier_2"
      }
    ],
    "skills": [
      {
        "name": "Recommendation Systems",
        "proficiency": "advanced",
        "endorsements": 5,
        "duration_months": 40
      },
      {
        "name": "Scrum",
        "proficiency": "beginner",
        "endorsements": 13,
        "duration_months": 7
      },
      {
        "name": "FastAPI",
        "proficiency": "beginner",
        "endorsements": 4,
        "duration_months": 12
      },
      {
        "name": "Hugging Face Transformers",
        "proficiency": "intermediate",
        "endorsements": 1,
        "duration_months": 30
      },
      {
        "name": "AWS",
        "proficiency": "beginner",
        "endorsements": 4,
        "duration_months": 18
      },
      {
        "name": "Snowflake",
        "proficiency": "beginner",
        "endorsements": 4,
        "duration_months": 11
      },
      {
        "name": "Spring Boot",
        "proficiency": "intermediate",
        "endorsements": 12,
        "duration_months": 31
      },
      {
        "name": "PostgreSQL",
        "proficiency": "intermediate",
        "endorsements": 7,
        "duration_months": 24
      },
      {
        "name": "Kubeflow",
        "proficiency": "advanced",
        "endorsements": 6,
        "duration_months": 59
      },
      {
        "name": "Azure",
        "proficiency": "beginner",
        "endorsements": 12,
        "duration_months": 8
      }
    ],
    "certifications": [
      {
        "name": "AWS Certified Cloud Practitioner",
        "issuer": "AWS",
        "year": 2019
      },
      {
        "name": "Six Sigma Green Belt",
        "issuer": "ASQ",
        "year": 2021
      }
    ],
    "languages": [
      {
        "language": "English",
        "proficiency": "native"
      },
      {
        "language": "Hindi",
        "proficiency": "professional"
      }
    ],
    "redrob_signals": {
      "profile_completeness_score": 59.2,
      "signup_date": "2023-07-22",
      "last_active_date": "2026-01-19",
      "open_to_work_flag": false,
      "profile_views_received_30d": 112,
      "applications_submitted_30d": 0,
      "recruiter_response_rate": 0.56,
      "avg_response_time_hours": 184.4,
      "skill_assessment_scores": {
        "Recommendation Systems": 29.8
      },
      "connection_count": 496,
      "endorsements_received": 9,
      "notice_period_days": 90,
      "expected_salary_range_inr_lpa": {
        "min": 15.5,
        "max": 13.9
      },
      "preferred_work_mode": "flexible",
      "willing_to_relocate": false,
      "github_activity_score": 32.3,
      "search_appearance_30d": 200,
      "saved_by_recruiters_30d": 13,
      "interview_completion_rate": 0.45,
      "offer_acceptance_rate": -1,
      "verified_email": true,
      "verified_phone": true,
      "linkedin_connected": true
    }
  },
  {
    "candidate_id": "CAND_0000012",
    "profile": {
      "anonymized_name": "Anjali Krishnan",
      "headline": "Operations Manager | Driving business outcomes",
      "summary": "Professional with 1.1+ years of experience. My professional background is in marketing manager \u2014 I've built and led teams, owned KPIs, and driven business outcomes in this domain. Lately I've been curious about how AI tools could augment my work \u2014 I've experimented with ChatGPT and a few other tools for productivity and content creation, and I think the space is exciting. Open to roles where I can apply my domain expertise alongside emerging AI capabilities.",
      "location": "Chandigarh, Chandigarh",
      "country": "India",
      "years_of_experience": 1.1,
      "current_title": "Operations Manager",
      "current_company": "Stark Industries",
      "current_company_size": "1001-5000",
      "current_industry": "Manufacturing"
    },
    "career_history": [
      {
        "company": "Stark Industries",
        "title": "Operations Manager",
        "start_date": "2025-05-02",
        "end_date": null,
        "duration_months": 13,
        "is_current": true,
        "industry": "Manufacturing",
        "company_size": "1001-5000",
        "description": "Senior accounting role at a mid-sized company \u2014 month-end close, financial reporting, statutory compliance (GAAP / Ind-AS), and tax filings. Owned the GL, fixed-asset register, and the audit-readiness function. Managed a team of 3 staff accountants. Built strong process discipline around the close cycle, reducing close time from 12 days to 7 over the last two years."
      }
    ],
    "education": [
      {
        "institution": "Symbiosis International",
        "degree": "B.Sc",
        "field_of_study": "Physics",
        "start_year": 2018,
        "end_year": 2022,
        "grade": "68%",
        "tier": "tier_3"
      },
      {
        "institution": "Christ University",
        "degree": "B.Sc",
        "field_of_study": "Mechanical Engineering",
        "start_year": 2011,
        "end_year": 2015,
        "grade": "7.28 CGPA",
        "tier": "tier_3"
      }
    ],
    "skills": [
      {
        "name": "Azure",
        "proficiency": "beginner",
        "endorsements": 7,
        "duration_months": 10
      },
      {
        "name": "Airflow",
        "proficiency": "intermediate",
        "endorsements": 2,
        "duration_months": 15
      },
      {
        "name": "AWS",
        "proficiency": "intermediate",
        "endorsements": 15,
        "duration_months": 30
      },
      {
        "name": "gRPC",
        "proficiency": "beginner",
        "endorsements": 9,
        "duration_months": 2
      },
      {
        "name": "Vue.js",
        "proficiency": "intermediate",
        "endorsements": 2,
        "duration_months": 15
      },
      {
        "name": "dbt",
        "proficiency": "intermediate",
        "endorsements": 11,
        "duration_months": 22
      },
      {
        "name": "Agile",
        "proficiency": "intermediate",
        "endorsements": 4,
        "duration_months": 14
      },
      {
        "name": "PowerPoint",
        "proficiency": "intermediate",
        "endorsements": 1,
        "duration_months": 27
      },
      {
        "name": "Content Writing",
        "proficiency": "intermediate",
        "endorsements": 3,
        "duration_months": 36
      },
      {
        "name": "Project Management",
        "proficiency": "beginner",
        "endorsements": 5,
        "duration_months": 3
      }
    ],
    "certifications": [],
    "languages": [
      {
        "language": "English",
        "proficiency": "professional"
      },
      {
        "language": "Hindi",
        "proficiency": "conversational"
      }
    ],
    "redrob_signals": {
      "profile_completeness_score": 53.4,
      "signup_date": "2024-01-28",
      "last_active_date": "2025-10-28",
      "open_to_work_flag": false,
      "profile_views_received_30d": 60,
      "applications_submitted_30d": 1,
      "recruiter_response_rate": 0.16,
      "avg_response_time_hours": 38.4,
      "skill_assessment_scores": {},
      "connection_count": 165,
      "endorsements_received": 31,
      "notice_period_days": 60,
      "expected_salary_range_inr_lpa": {
        "min": 14.8,
        "max": 7.9
      },
      "preferred_work_mode": "flexible",
      "willing_to_relocate": false,
      "github_activity_score": -1,
      "search_appearance_30d": 61,
      "saved_by_recruiters_30d": 3,
      "interview_completion_rate": 0.42,
      "offer_acceptance_rate": -1,
      "verified_email": true,
      "verified_phone": false,
      "linkedin_connected": false
    }
  },
  {
    "candidate_id": "CAND_0000013",
    "profile": {
      "anonymized_name": "Pari Nair",
      "headline": "Civil Engineer | Driving business outcomes",
      "summary": "Professional with 1.1+ years of experience. I'm a marketing manager with substantial experience in cross-functional collaboration, stakeholder management, and execution. Lately I've been curious about how AI tools could augment my work \u2014 I've experimented with ChatGPT and a few other tools for productivity and content creation, and I think the space is exciting. Open to roles where I can apply my domain expertise alongside emerging AI capabilities.",
      "location": "Dubai",
      "country": "UAE",
      "years_of_experience": 1.1,
      "current_title": "Civil Engineer",
      "current_company": "Globex Inc",
      "current_company_size": "501-1000",
      "current_industry": "Manufacturing"
    },
    "career_history": [
      {
        "company": "Globex Inc",
        "title": "Civil Engineer",
        "start_date": "2025-05-02",
        "end_date": null,
        "duration_months": 13,
        "is_current": true,
        "industry": "Manufacturing",
        "company_size": "501-1000",
        "description": "Customer support team lead at a SaaS product. Managed a team of 8 support agents handling tier-1 and tier-2 tickets; owned the escalation process to engineering and the customer-feedback loop to product. Built out the support knowledge base and the agent training program. Strong on the people-management side and the process side; lighter on technical depth beyond product expertise."
      }
    ],
    "education": [
      {
        "institution": "Delhi College of Engineering",
        "degree": "B.E.",
        "field_of_study": "Information Technology",
        "start_year": 2019,
        "end_year": 2022,
        "grade": "8.84 CGPA",
        "tier": "tier_2"
      },
      {
        "institution": "Amity University",
        "degree": "Ph.D",
        "field_of_study": "Information Technology",
        "start_year": 2008,
        "end_year": 2013,
        "grade": "8.29 CGPA",
        "tier": "tier_3"
      }
    ],
    "skills": [
      {
        "name": "React",
        "proficiency": "intermediate",
        "endorsements": 5,
        "duration_months": 23
      },
      {
        "name": "Redux",
        "proficiency": "intermediate",
        "endorsements": 11,
        "duration_months": 9
      },
      {
        "name": "Vue.js",
        "proficiency": "beginner",
        "endorsements": 12,
        "duration_months": 12
      },
      {
        "name": "Six Sigma",
        "proficiency": "beginner",
        "endorsements": 1,
        "duration_months": 2
      },
      {
        "name": "Spring Boot",
        "proficiency": "beginner",
        "endorsements": 12,
        "duration_months": 12
      },
      {
        "name": "Spark",
        "proficiency": "intermediate",
        "endorsements": 7,
        "duration_months": 30
      },
      {
        "name": "Data Pipelines",
        "proficiency": "intermediate",
        "endorsements": 6,
        "duration_months": 36
      },
      {
        "name": "GCP",
        "proficiency": "intermediate",
        "endorsements": 0,
        "duration_months": 18
      },
      {
        "name": "Flask",
        "proficiency": "intermediate",
        "endorsements": 12,
        "duration_months": 8
      },
      {
        "name": "Snowflake",
        "proficiency": "intermediate",
        "endorsements": 2,
        "duration_months": 8
      }
    ],
    "certifications": [],
    "languages": [
      {
        "language": "English",
        "proficiency": "native"
      },
      {
        "language": "Hindi",
        "proficiency": "conversational"
      }
    ],
    "redrob_signals": {
      "profile_completeness_score": 44.2,
      "signup_date": "2024-06-14",
      "last_active_date": "2026-03-26",
      "open_to_work_flag": true,
      "profile_views_received_30d": 16,
      "applications_submitted_30d": 3,
      "recruiter_response_rate": 0.28,
      "avg_response_time_hours": 80.3,
      "skill_assessment_scores": {},
      "connection_count": 281,
      "endorsements_received": 9,
      "notice_period_days": 30,
      "expected_salary_range_inr_lpa": {
        "min": 11.6,
        "max": 8.1
      },
      "preferred_work_mode": "remote",
      "willing_to_relocate": false,
      "github_activity_score": 35.6,
      "search_appearance_30d": 40,
      "saved_by_recruiters_30d": 12,
      "interview_completion_rate": 0.33,
      "offer_acceptance_rate": 0.26,
      "verified_email": true,
      "verified_phone": false,
      "linkedin_connected": false
    }
  },
  {
    "candidate_id": "CAND_0000014",
    "profile": {
      "anonymized_name": "Atharv Joshi",
      "headline": "Frontend Engineer | Full-stack development",
      "summary": "Software engineer with 8.4 years of experience across web, backend, and cloud systems. Strong fundamentals in software development and system design. My background is full-stack, but my comfort zone is the backend and the database. I've been keeping up with AI/ML at a self-learner level \u2014 taken some online courses, played with the OpenAI and Anthropic APIs, built a small RAG side project \u2014 but I haven't done it in a professional capacity yet. Open to roles where I can either deepen my software engineering work or, if the team is open to it, start contributing to ML-adjacent systems.",
      "location": "Hyderabad, Telangana",
      "country": "India",
      "years_of_experience": 8.4,
      "current_title": "Frontend Engineer",
      "current_company": "Zomato",
      "current_company_size": "5001-10000",
      "current_industry": "Food Delivery"
    },
    "career_history": [
      {
        "company": "Zomato",
        "title": "Frontend Engineer",
        "start_date": "2023-09-10",
        "end_date": null,
        "duration_months": 33,
        "is_current": true,
        "industry": "Food Delivery",
        "company_size": "5001-10000",
        "description": "Frontend engineering at a media company. React, TypeScript, and the typical surrounding tooling (Webpack, Jest, Cypress). Built the company's design system from scratch and led the migration from a legacy AngularJS app. Strong on the frontend craft \u2014 accessibility, performance, animations \u2014 but limited backend exposure."
      },
      {
        "company": "Dunder Mifflin",
        "title": "Software Engineer",
        "start_date": "2019-10-01",
        "end_date": "2023-09-10",
        "duration_months": 48,
        "is_current": false,
        "industry": "Paper Products",
        "company_size": "201-500",
        "description": "Test automation and QA engineering for a fintech product. Built and maintained the end-to-end test suite using Selenium and pytest, plus the load-testing setup using Locust. Worked closely with developers on testability patterns and with product on acceptance criteria. Recent work has been on shifting test responsibility into the dev team \u2014 moving from QA-as-gate to QA-as-coach. Career has been entirely in QA/test engineering."
      },
      {
        "company": "Acme Corp",
        "title": "Java Developer",
        "start_date": "2018-03-03",
        "end_date": "2019-09-24",
        "duration_months": 19,
        "is_current": false,
        "industry": "Manufacturing",
        "company_size": "201-500",
        "description": "Android mobile development using Java and (more recently) Kotlin at a consumer-app company. Built and maintained multiple production features including the main shopping flow, push notification system, and the offline-first sync layer. Comfortable with the Android framework, Jetpack components, and the typical patterns (MVVM, Hilt, Coroutines). My career has been entirely on mobile so far; interested in expanding into broader backend or platform engineering."
      }
    ],
    "education": [
      {
        "institution": "Lovely Professional University",
        "degree": "B.E.",
        "field_of_study": "Statistics",
        "start_year": 2012,
        "end_year": 2015,
        "grade": "7.45 CGPA",
        "tier": "tier_3"
      }
    ],
    "skills": [
      {
        "name": "FAISS",
        "proficiency": "advanced",
        "endorsements": 40,
        "duration_months": 44
      },
      {
        "name": "BigQuery",
        "proficiency": "intermediate",
        "endorsements": 6,
        "duration_months": 24
      },
      {
        "name": "React",
        "proficiency": "beginner",
        "endorsements": 11,
        "duration_months": 10
      },
      {
        "name": "OpenSearch",
        "proficiency": "advanced",
        "endorsements": 47,
        "duration_months": 59
      },
      {
        "name": "OpenCV",
        "proficiency": "advanced",
        "endorsements": 30,
        "duration_months": 60
      },
      {
        "name": "YOLO",
        "proficiency": "advanced",
        "endorsements": 1,
        "duration_months": 44
      },
      {
        "name": "SAP",
        "proficiency": "intermediate",
        "endorsements": 5,
        "duration_months": 30
      },
      {
        "name": "SEO",
        "proficiency": "beginner",
        "endorsements": 0,
        "duration_months": 12
      },
      {
        "name": "REST APIs",
        "proficiency": "beginner",
        "endorsements": 3,
        "duration_months": 4
      },
      {
        "name": "GANs",
        "proficiency": "advanced",
        "endorsements": 9,
        "duration_months": 33
      },
      {
        "name": "dbt",
        "proficiency": "beginner",
        "endorsements": 0,
        "duration_months": 13
      },
      {
        "name": "Photoshop",
        "proficiency": "intermediate",
        "endorsements": 1,
        "duration_months": 32
      },
      {
        "name": "Tailwind",
        "proficiency": "intermediate",
        "endorsements": 2,
        "duration_months": 32
      }
    ],
    "certifications": [],
    "languages": [
      {
        "language": "English",
        "proficiency": "professional"
      },
      {
        "language": "Hindi",
        "proficiency": "native"
      }
    ],
    "redrob_signals": {
      "profile_completeness_score": 61.9,
      "signup_date": "2025-04-29",
      "last_active_date": "2026-04-12",
      "open_to_work_flag": false,
      "profile_views_received_30d": 21,
      "applications_submitted_30d": 1,
      "recruiter_response_rate": 0.8,
      "avg_response_time_hours": 7.7,
      "skill_assessment_scores": {
        "FAISS": 77.6
      },
      "connection_count": 708,
      "endorsements_received": 63,
      "notice_period_days": 90,
      "expected_salary_range_inr_lpa": {
        "min": 9.0,
        "max": 30.0
      },
      "preferred_work_mode": "remote",
      "willing_to_relocate": false,
      "github_activity_score": -1,
      "search_appearance_30d": 12,
      "saved_by_recruiters_30d": 0,
      "interview_completion_rate": 0.55,
      "offer_acceptance_rate": -1,
      "verified_email": true,
      "verified_phone": true,
      "linkedin_connected": false
    }
  },
  {
    "candidate_id": "CAND_0000015",
    "profile": {
      "anonymized_name": "Rahul Agarwal",
      "headline": "Software Engineer | Cloud & DevOps",
      "summary": "Software engineer with 5.4 years of experience across web, backend, and cloud systems. Strong fundamentals in software development and system design. I've worked across web frontends, REST APIs, and cloud deployments; comfortable in most parts of a typical SaaS stack. I've been keeping up with AI/ML at a self-learner level \u2014 taken some online courses, played with the OpenAI and Anthropic APIs, built a small RAG side project \u2014 but I haven't done it in a professional capacity yet. Open to roles where I can either deepen my software engineering work or, if the team is open to it, start contributing to ML-adjacent systems.",
      "location": "Trivandrum, Kerala",
      "country": "India",
      "years_of_experience": 5.4,
      "current_title": "Software Engineer",
      "current_company": "Razorpay",
      "current_company_size": "1001-5000",
      "current_industry": "Fintech"
    },
    "career_history": [
      {
        "company": "Razorpay",
        "title": "Software Engineer",
        "start_date": "2023-11-09",
        "end_date": null,
        "duration_months": 31,
        "is_current": true,
        "industry": "Fintech",
        "company_size": "1001-5000",
        "description": "Cloud infrastructure and DevOps work at an enterprise SaaS company. Owned the AWS account architecture (VPC, IAM, networking), the Terraform modules for our service deployments, and the Kubernetes cluster operations. Designed the CI/CD pipelines (GitLab CI + ArgoCD) and the monitoring stack (Prometheus, Grafana, Loki). Strong on the infra and ops side; haven't done much application development."
      },
      {
        "company": "Hooli",
        "title": "Mobile Developer",
        "start_date": "2021-11-12",
        "end_date": "2023-11-02",
        "duration_months": 24,
        "is_current": false,
        "industry": "Software",
        "company_size": "1001-5000",
        "description": "Android mobile development using Java and (more recently) Kotlin at a consumer-app company. Built and maintained multiple production features including the main shopping flow, push notification system, and the offline-first sync layer. Comfortable with the Android framework, Jetpack components, and the typical patterns (MVVM, Hilt, Coroutines). My career has been entirely on mobile so far; interested in expanding into broader backend or platform engineering."
      },
      {
        "company": "Globex Inc",
        "title": "DevOps Engineer",
        "start_date": "2021-02-15",
        "end_date": "2021-11-12",
        "duration_months": 9,
        "is_current": false,
        "industry": "Manufacturing",
        "company_size": "501-1000",
        "description": "Java backend development at a large enterprise \u2014 Spring Boot microservices, Kafka for inter-service messaging, Postgres + Redis for storage. Worked on the customer onboarding flow which involved orchestrating multiple downstream services. Solid on the Spring ecosystem, transaction handling, and the operational side of Java services. Looking to either go deeper on distributed systems or expand into modern application stacks."
      }
    ],
    "education": [
      {
        "institution": "Local Engineering College",
        "degree": "Ph.D",
        "field_of_study": "Mathematics",
        "start_year": 2013,
        "end_year": 2017,
        "grade": "8.15 CGPA",
        "tier": "tier_4"
      }
    ],
    "skills": [
      {
        "name": "PyTorch",
        "proficiency": "intermediate",
        "endorsements": 10,
        "duration_months": 15
      },
      {
        "name": "Content Writing",
        "proficiency": "intermediate",
        "endorsements": 8,
        "duration_months": 31
      },
      {
        "name": "Weights & Biases",
        "proficiency": "intermediate",
        "endorsements": 5,
        "duration_months": 24
      },
      {
        "name": "Qdrant",
        "proficiency": "intermediate",
        "endorsements": 13,
        "duration_months": 9
      },
      {
        "name": "Sales",
        "proficiency": "intermediate",
        "endorsements": 13,
        "duration_months": 30
      },
      {
        "name": "Figma",
        "proficiency": "intermediate",
        "endorsements": 9,
        "duration_months": 29
      },
      {
        "name": "BigQuery",
        "proficiency": "beginner",
        "endorsements": 0,
        "duration_months": 7
      },
      {
        "name": "Computer Vision",
        "proficiency": "intermediate",
        "endorsements": 6,
        "duration_months": 20
      },
      {
        "name": "Next.js",
        "proficiency": "intermediate",
        "endorsements": 12,
        "duration_months": 35
      },
      {
        "name": "SEO",
        "proficiency": "intermediate",
        "endorsements": 10,
        "duration_months": 29
      }
    ],
    "certifications": [],
    "languages": [
      {
        "language": "English",
        "proficiency": "native"
      },
      {
        "language": "Hindi",
        "proficiency": "conversational"
      }
    ],
    "redrob_signals": {
      "profile_completeness_score": 79.4,
      "signup_date": "2023-02-16",
      "last_active_date": "2026-02-12",
      "open_to_work_flag": true,
      "profile_views_received_30d": 93,
      "applications_submitted_30d": 3,
      "recruiter_response_rate": 0.32,
      "avg_response_time_hours": 117.7,
      "skill_assessment_scores": {},
      "connection_count": 361,
      "endorsements_received": 61,
      "notice_period_days": 90,
      "expected_salary_range_inr_lpa": {
        "min": 21.8,
        "max": 18.9
      },
      "preferred_work_mode": "remote",
      "willing_to_relocate": false,
      "github_activity_score": -1,
      "search_appearance_30d": 164,
      "saved_by_recruiters_30d": 8,
      "interview_completion_rate": 0.72,
      "offer_acceptance_rate": -1,
      "verified_email": true,
      "verified_phone": true,
      "linkedin_connected": false
    }
  },
  {
    "candidate_id": "CAND_0000016",
    "profile": {
      "anonymized_name": "Aanya Malhotra",
      "headline": "Accountant | Helping teams scale",
      "summary": "Professional with 5.3+ years of experience. I'm a marketing manager with substantial experience in cross-functional collaboration, stakeholder management, and execution. Lately I've been curious about how AI tools could augment my work \u2014 I've experimented with ChatGPT and a few other tools for productivity and content creation, and I think the space is exciting. Open to roles where I can apply my domain expertise alongside emerging AI capabilities.",
      "location": "Gurgaon, Haryana",
      "country": "India",
      "years_of_experience": 5.3,
      "current_title": "Accountant",
      "current_company": "Infosys",
      "current_company_size": "10001+",
      "current_industry": "IT Services"
    },
    "career_history": [
      {
        "company": "Infosys",
        "title": "Accountant",
        "start_date": "2024-12-03",
        "end_date": null,
        "duration_months": 18,
        "is_current": true,
        "industry": "IT Services",
        "company_size": "10001+",
        "description": "Enterprise sales of cloud software solutions into the mid-market segment. Carried a $1.8M ARR quota and consistently delivered against it across the last three years. Owned the full sales cycle: prospecting, discovery, technical evaluation (with SE support), commercial negotiation, and close. Strong on consultative selling for technical buyers; comfortable engaging with both engineering and finance stakeholders."
      },
      {
        "company": "TCS",
        "title": "Mechanical Engineer",
        "start_date": "2021-09-06",
        "end_date": "2024-11-19",
        "duration_months": 39,
        "is_current": false,
        "industry": "IT Services",
        "company_size": "10001+",
        "description": "Customer support team lead at a SaaS product. Managed a team of 8 support agents handling tier-1 and tier-2 tickets; owned the escalation process to engineering and the customer-feedback loop to product. Built out the support knowledge base and the agent training program. Strong on the people-management side and the process side; lighter on technical depth beyond product expertise."
      },
      {
        "company": "Globex Inc",
        "title": "Operations Manager",
        "start_date": "2021-02-08",
        "end_date": "2021-08-07",
        "duration_months": 6,
        "is_current": false,
        "industry": "Manufacturing",
        "company_size": "501-1000",
        "description": "Mechanical engineering design role at a hardware-product company. Led the design of two product subsystems through full lifecycle: concept, DFM/DFMA review, prototype, production tooling. Comfortable with CAD (SolidWorks, Creo), FEA (ANSYS), and the typical hardware-development cadence. Worked closely with manufacturing partners on production scale-up."
      }
    ],
    "education": [
      {
        "institution": "Christ University",
        "degree": "B.E.",
        "field_of_study": "Electronics",
        "start_year": 2001,
        "end_year": 2006,
        "grade": "8.32 CGPA",
        "tier": "tier_3"
      }
    ],
    "skills": [
      {
        "name": "Node.js",
        "proficiency": "beginner",
        "endorsements": 4,
        "duration_months": 11
      },
      {
        "name": "Figma",
        "proficiency": "beginner",
        "endorsements": 12,
        "duration_months": 6
      },
      {
        "name": "Data Pipelines",
        "proficiency": "beginner",
        "endorsements": 2,
        "duration_months": 7
      },
      {
        "name": "Go",
        "proficiency": "beginner",
        "endorsements": 0,
        "duration_months": 10
      },
      {
        "name": "Photoshop",
        "proficiency": "intermediate",
        "endorsements": 10,
        "duration_months": 22
      },
      {
        "name": "Kubeflow",
        "proficiency": "advanced",
        "endorsements": 2,
        "duration_months": 54
      },
      {
        "name": "Accounting",
        "proficiency": "intermediate",
        "endorsements": 4,
        "duration_months": 31
      },
      {
        "name": "SQL",
        "proficiency": "intermediate",
        "endorsements": 14,
        "duration_months": 16
      }
    ],
    "certifications": [],
    "languages": [
      {
        "language": "English",
        "proficiency": "native"
      },
      {
        "language": "Hindi",
        "proficiency": "professional"
      }
    ],
    "redrob_signals": {
      "profile_completeness_score": 69.4,
      "signup_date": "2022-12-12",
      "last_active_date": "2025-12-21",
      "open_to_work_flag": true,
      "profile_views_received_30d": 76,
      "applications_submitted_30d": 5,
      "recruiter_response_rate": 0.64,
      "avg_response_time_hours": 205.6,
      "skill_assessment_scores": {},
      "connection_count": 148,
      "endorsements_received": 2,
      "notice_period_days": 60,
      "expected_salary_range_inr_lpa": {
        "min": 6.1,
        "max": 8.1
      },
      "preferred_work_mode": "hybrid",
      "willing_to_relocate": false,
      "github_activity_score": 42.9,
      "search_appearance_30d": 126,
      "saved_by_recruiters_30d": 5,
      "interview_completion_rate": 0.66,
      "offer_acceptance_rate": -1,
      "verified_email": false,
      "verified_phone": false,
      "linkedin_connected": true
    }
  },
  {
    "candidate_id": "CAND_0000017",
    "profile": {
      "anonymized_name": "Anil Pandey",
      "headline": "Accountant | 12.3+ yrs experience",
      "summary": "Professional with 12.3+ years of experience. My professional background is in marketing manager \u2014 I've built and led teams, owned KPIs, and driven business outcomes in this domain. Lately I've been curious about how AI tools could augment my work \u2014 I've experimented with ChatGPT and a few other tools for productivity and content creation, and I think the space is exciting. Open to roles where I can apply my domain expertise alongside emerging AI capabilities.",
      "location": "Bangalore, Karnataka",
      "country": "India",
      "years_of_experience": 12.3,
      "current_title": "Accountant",
      "current_company": "Wipro",
      "current_company_size": "10001+",
      "current_industry": "IT Services"
    },
    "career_history": [
      {
        "company": "Wipro",
        "title": "Accountant",
        "start_date": "2024-03-08",
        "end_date": null,
        "duration_months": 27,
        "is_current": true,
        "industry": "IT Services",
        "company_size": "10001+",
        "description": "Customer support team lead at a SaaS product. Managed a team of 8 support agents handling tier-1 and tier-2 tickets; owned the escalation process to engineering and the customer-feedback loop to product. Built out the support knowledge base and the agent training program. Strong on the people-management side and the process side; lighter on technical depth beyond product expertise."
      },
      {
        "company": "Infosys",
        "title": "Customer Support",
        "start_date": "2021-02-08",
        "end_date": "2024-02-23",
        "duration_months": 37,
        "is_current": false,
        "industry": "IT Services",
        "company_size": "10001+",
        "description": "Mechanical engineering design role at a hardware-product company. Led the design of two product subsystems through full lifecycle: concept, DFM/DFMA review, prototype, production tooling. Comfortable with CAD (SolidWorks, Creo), FEA (ANSYS), and the typical hardware-development cadence. Worked closely with manufacturing partners on production scale-up."
      },
      {
        "company": "Initech",
        "title": "Mechanical Engineer",
        "start_date": "2017-07-29",
        "end_date": "2021-02-08",
        "duration_months": 43,
        "is_current": false,
        "industry": "Software",
        "company_size": "51-200",
        "description": "Enterprise sales of cloud software solutions into the mid-market segment. Carried a $1.8M ARR quota and consistently delivered against it across the last three years. Owned the full sales cycle: prospecting, discovery, technical evaluation (with SE support), commercial negotiation, and close. Strong on consultative selling for technical buyers; comfortable engaging with both engineering and finance stakeholders."
      },
      {
        "company": "Acme Corp",
        "title": "Accountant",
        "start_date": "2014-05-16",
        "end_date": "2017-07-29",
        "duration_months": 39,
        "is_current": false,
        "industry": "Manufacturing",
        "company_size": "201-500",
        "description": "Senior accounting role at a mid-sized company \u2014 month-end close, financial reporting, statutory compliance (GAAP / Ind-AS), and tax filings. Owned the GL, fixed-asset register, and the audit-readiness function. Managed a team of 3 staff accountants. Built strong process discipline around the close cycle, reducing close time from 12 days to 7 over the last two years."
      }
    ],
    "education": [
      {
        "institution": "Tier-3 Engineering College",
        "degree": "M.Tech",
        "field_of_study": "Data Science",
        "start_year": 2017,
        "end_year": 2022,
        "grade": "7.58 CGPA",
        "tier": "tier_4"
      }
    ],
    "skills": [
      {
        "name": "Next.js",
        "proficiency": "beginner",
        "endorsements": 9,
        "duration_months": 18
      },
      {
        "name": "Java",
        "proficiency": "intermediate",
        "endorsements": 13,
        "duration_months": 15
      },
      {
        "name": "Apache Flink",
        "proficiency": "intermediate",
        "endorsements": 4,
        "duration_months": 16
      },
      {
        "name": "Sales",
        "proficiency": "intermediate",
        "endorsements": 4,
        "duration_months": 8
      },
      {
        "name": "Tally",
        "proficiency": "intermediate",
        "endorsements": 14,
        "duration_months": 12
      },
      {
        "name": "PostgreSQL",
        "proficiency": "intermediate",
        "endorsements": 11,
        "duration_months": 25
      },
      {
        "name": "REST APIs",
        "proficiency": "beginner",
        "endorsements": 15,
        "duration_months": 4
      },
      {
        "name": "Hadoop",
        "proficiency": "intermediate",
        "endorsements": 4,
        "duration_months": 35
      }
    ],
    "certifications": [
      {
        "name": "Six Sigma Green Belt",
        "issuer": "ASQ",
        "year": 2018
      },
      {
        "name": "Scrum Master Certified",
        "issuer": "Scrum Alliance",
        "year": 2022
      }
    ],
    "languages": [
      {
        "language": "English",
        "proficiency": "native"
      },
      {
        "language": "Hindi",
        "proficiency": "conversational"
      }
    ],
    "redrob_signals": {
      "profile_completeness_score": 38.7,
      "signup_date": "2025-08-11",
      "last_active_date": "2025-11-04",
      "open_to_work_flag": false,
      "profile_views_received_30d": 3,
      "applications_submitted_30d": 4,
      "recruiter_response_rate": 0.27,
      "avg_response_time_hours": 197.4,
      "skill_assessment_scores": {},
      "connection_count": 35,
      "endorsements_received": 23,
      "notice_period_days": 90,
      "expected_salary_range_inr_lpa": {
        "min": 13.8,
        "max": 8.4
      },
      "preferred_work_mode": "hybrid",
      "willing_to_relocate": false,
      "github_activity_score": -1,
      "search_appearance_30d": 110,
      "saved_by_recruiters_30d": 2,
      "interview_completion_rate": 0.32,
      "offer_acceptance_rate": 0.17,
      "verified_email": true,
      "verified_phone": false,
      "linkedin_connected": false
    }
  },
  {
    "candidate_id": "CAND_0000018",
    "profile": {
      "anonymized_name": "Vivaan Reddy",
      "headline": "Frontend Engineer | Full-stack development",
      "summary": "Software engineer with 6.6 years of experience across web, backend, and cloud systems. Strong fundamentals in software development and system design. My background is full-stack, but my comfort zone is the backend and the database. I've been keeping up with AI/ML at a self-learner level \u2014 taken some online courses, played with the OpenAI and Anthropic APIs, built a small RAG side project \u2014 but I haven't done it in a professional capacity yet. Open to roles where I can either deepen my software engineering work or, if the team is open to it, start contributing to ML-adjacent systems.",
      "location": "Bhubaneswar, Odisha",
      "country": "India",
      "years_of_experience": 6.6,
      "current_title": "Frontend Engineer",
      "current_company": "Acme Corp",
      "current_company_size": "201-500",
      "current_industry": "Manufacturing"
    },
    "career_history": [
      {
        "company": "Acme Corp",
        "title": "Frontend Engineer",
        "start_date": "2023-09-10",
        "end_date": null,
        "duration_months": 33,
        "is_current": true,
        "industry": "Manufacturing",
        "company_size": "201-500",
        "description": "Test automation and QA engineering for a fintech product. Built and maintained the end-to-end test suite using Selenium and pytest, plus the load-testing setup using Locust. Worked closely with developers on testability patterns and with product on acceptance criteria. Recent work has been on shifting test responsibility into the dev team \u2014 moving from QA-as-gate to QA-as-coach. Career has been entirely in QA/test engineering."
      },
      {
        "company": "Pied Piper",
        "title": "Frontend Engineer",
        "start_date": "2021-01-23",
        "end_date": "2023-09-10",
        "duration_months": 32,
        "is_current": false,
        "industry": "Software",
        "company_size": "11-50",
        "description": "Test automation and QA engineering for a fintech product. Built and maintained the end-to-end test suite using Selenium and pytest, plus the load-testing setup using Locust. Worked closely with developers on testability patterns and with product on acceptance criteria. Recent work has been on shifting test responsibility into the dev team \u2014 moving from QA-as-gate to QA-as-coach. Career has been entirely in QA/test engineering."
      },
      {
        "company": "Initech",
        "title": "Full Stack Developer",
        "start_date": "2019-12-30",
        "end_date": "2021-01-23",
        "duration_months": 13,
        "is_current": false,
        "industry": "Software",
        "company_size": "51-200",
        "description": "Android mobile development using Java and (more recently) Kotlin at a consumer-app company. Built and maintained multiple production features including the main shopping flow, push notification system, and the offline-first sync layer. Comfortable with the Android framework, Jetpack components, and the typical patterns (MVVM, Hilt, Coroutines). My career has been entirely on mobile so far; interested in expanding into broader backend or platform engineering."
      }
    ],
    "education": [
      {
        "institution": "Lovely Professional University",
        "degree": "Ph.D",
        "field_of_study": "Computer Engineering",
        "start_year": 2016,
        "end_year": 2020,
        "grade": "7.25 CGPA",
        "tier": "tier_3"
      }
    ],
    "skills": [
      {
        "name": "CNN",
        "proficiency": "advanced",
        "endorsements": 53,
        "duration_months": 55
      },
      {
        "name": "Java",
        "proficiency": "intermediate",
        "endorsements": 10,
        "duration_months": 12
      },
      {
        "name": "Accounting",
        "proficiency": "intermediate",
        "endorsements": 9,
        "duration_months": 20
      },
      {
        "name": "Data Pipelines",
        "proficiency": "beginner",
        "endorsements": 3,
        "duration_months": 13
      },
      {
        "name": "Node.js",
        "proficiency": "intermediate",
        "endorsements": 0,
        "duration_months": 9
      },
      {
        "name": "Tailwind",
        "proficiency": "beginner",
        "endorsements": 10,
        "duration_months": 10
      }
    ],
    "certifications": [],
    "languages": [
      {
        "language": "English",
        "proficiency": "native"
      },
      {
        "language": "Hindi",
        "proficiency": "native"
      }
    ],
    "redrob_signals": {
      "profile_completeness_score": 34.8,
      "signup_date": "2025-07-09",
      "last_active_date": "2026-02-18",
      "open_to_work_flag": false,
      "profile_views_received_30d": 88,
      "applications_submitted_30d": 11,
      "recruiter_response_rate": 0.16,
      "avg_response_time_hours": 154.6,
      "skill_assessment_scores": {},
      "connection_count": 284,
      "endorsements_received": 49,
      "notice_period_days": 120,
      "expected_salary_range_inr_lpa": {
        "min": 12.3,
        "max": 26.4
      },
      "preferred_work_mode": "onsite",
      "willing_to_relocate": false,
      "github_activity_score": -1,
      "search_appearance_30d": 41,
      "saved_by_recruiters_30d": 16,
      "interview_completion_rate": 0.7,
      "offer_acceptance_rate": 0.46,
      "verified_email": false,
      "verified_phone": false,
      "linkedin_connected": false
    }
  },
  {
    "candidate_id": "CAND_0000019",
    "profile": {
      "anonymized_name": "Myra Mishra",
      "headline": "Project Manager | 6.5+ yrs experience",
      "summary": "Professional with 6.5+ years of experience. I've spent my career in marketing manager roles, mostly focused on driving outcomes through process, people, and customer relationships. Lately I've been curious about how AI tools could augment my work \u2014 I've experimented with ChatGPT and a few other tools for productivity and content creation, and I think the space is exciting. Open to roles where I can apply my domain expertise alongside emerging AI capabilities.",
      "location": "Trivandrum, Kerala",
      "country": "India",
      "years_of_experience": 6.5,
      "current_title": "Project Manager",
      "current_company": "Wayne Enterprises",
      "current_company_size": "10001+",
      "current_industry": "Conglomerate"
    },
    "career_history": [
      {
        "company": "Wayne Enterprises",
        "title": "Project Manager",
        "start_date": "2022-10-15",
        "end_date": null,
        "duration_months": 44,
        "is_current": true,
        "industry": "Conglomerate",
        "company_size": "10001+",
        "description": "Business analyst at a consulting firm, working primarily with retail and CPG clients. Conducted business diagnostics, process re-engineering work, and digital transformation strategy projects. Strong on stakeholder management, structured problem-solving, and the typical consulting toolkit (slide-craft, Excel modeling, executive communication). Recent project work involved AI-strategy advisory but my own technical depth in AI is limited."
      },
      {
        "company": "Wayne Enterprises",
        "title": "Marketing Manager",
        "start_date": "2020-08-19",
        "end_date": "2022-10-08",
        "duration_months": 26,
        "is_current": false,
        "industry": "Conglomerate",
        "company_size": "10001+",
        "description": "Mechanical engineering design role at a hardware-product company. Led the design of two product subsystems through full lifecycle: concept, DFM/DFMA review, prototype, production tooling. Comfortable with CAD (SolidWorks, Creo), FEA (ANSYS), and the typical hardware-development cadence. Worked closely with manufacturing partners on production scale-up."
      },
      {
        "company": "Pied Piper",
        "title": "Business Analyst",
        "start_date": "2020-01-15",
        "end_date": "2020-08-12",
        "duration_months": 7,
        "is_current": false,
        "industry": "Software",
        "company_size": "11-50",
        "description": "Brand design and creative direction at a consumer-products company. Owned brand identity (logo, visual system, typography), packaging design, and digital creative across web and social. Led the recent rebrand and managed a small external agency for production work. Comfortable across the Adobe suite, Figma, and the production side of brand and packaging design."
      }
    ],
    "education": [
      {
        "institution": "IISc Bangalore",
        "degree": "M.Tech",
        "field_of_study": "Computer Science",
        "start_year": 2010,
        "end_year": 2014,
        "grade": "72%",
        "tier": "tier_1"
      },
      {
        "institution": "IIT Guwahati",
        "degree": "M.Tech",
        "field_of_study": "Machine Learning",
        "start_year": 2002,
        "end_year": 2006,
        "grade": "7.34 CGPA",
        "tier": "tier_1"
      }
    ],
    "skills": [
      {
        "name": "Figma",
        "proficiency": "intermediate",
        "endorsements": 13,
        "duration_months": 34
      },
      {
        "name": "GraphQL",
        "proficiency": "beginner",
        "endorsements": 4,
        "duration_months": 12
      },
      {
        "name": "Six Sigma",
        "proficiency": "beginner",
        "endorsements": 4,
        "duration_months": 10
      },
      {
        "name": "Scrum",
        "proficiency": "beginner",
        "endorsements": 15,
        "duration_months": 2
      },
      {
        "name": "YOLO",
        "proficiency": "intermediate",
        "endorsements": 10,
        "duration_months": 34
      },
      {
        "name": "gRPC",
        "proficiency": "beginner",
        "endorsements": 12,
        "duration_months": 9
      },
      {
        "name": "AWS",
        "proficiency": "intermediate",
        "endorsements": 11,
        "duration_months": 28
      },
      {
        "name": "Azure",
        "proficiency": "intermediate",
        "endorsements": 6,
        "duration_months": 21
      }
    ],
    "certifications": [],
    "languages": [
      {
        "language": "English",
        "proficiency": "professional"
      },
      {
        "language": "Hindi",
        "proficiency": "conversational"
      }
    ],
    "redrob_signals": {
      "profile_completeness_score": 38.6,
      "signup_date": "2025-07-20",
      "last_active_date": "2026-05-21",
      "open_to_work_flag": false,
      "profile_views_received_30d": 61,
      "applications_submitted_30d": 9,
      "recruiter_response_rate": 0.34,
      "avg_response_time_hours": 100.0,
      "skill_assessment_scores": {},
      "connection_count": 593,
      "endorsements_received": 25,
      "notice_period_days": 60,
      "expected_salary_range_inr_lpa": {
        "min": 12.5,
        "max": 7.7
      },
      "preferred_work_mode": "hybrid",
      "willing_to_relocate": false,
      "github_activity_score": -1,
      "search_appearance_30d": 141,
      "saved_by_recruiters_30d": 0,
      "interview_completion_rate": 0.31,
      "offer_acceptance_rate": -1,
      "verified_email": false,
      "verified_phone": true,
      "linkedin_connected": false
    }
  },
  {
    "candidate_id": "CAND_0000020",
    "profile": {
      "anonymized_name": "Aditya Iyengar",
      "headline": "Mechanical Engineer | 6.3+ yrs experience",
      "summary": "Professional with 6.3+ years of experience. I'm a marketing manager with substantial experience in cross-functional collaboration, stakeholder management, and execution. Lately I've been curious about how AI tools could augment my work \u2014 I've experimented with ChatGPT and a few other tools for productivity and content creation, and I think the space is exciting. Open to roles where I can apply my domain expertise alongside emerging AI capabilities.",
      "location": "Ahmedabad, Gujarat",
      "country": "India",
      "years_of_experience": 6.3,
      "current_title": "Mechanical Engineer",
      "current_company": "Wipro",
      "current_company_size": "10001+",
      "current_industry": "IT Services"
    },
    "career_history": [
      {
        "company": "Wipro",
        "title": "Mechanical Engineer",
        "start_date": "2023-06-12",
        "end_date": null,
        "duration_months": 36,
        "is_current": true,
        "industry": "IT Services",
        "company_size": "10001+",
        "description": "Marketing leadership role at a B2B SaaS company. Owned the demand-generation function \u2014 content marketing, paid acquisition, SEO, email nurture. Built and managed a team of 5 across content, performance marketing, and marketing operations. Worked closely with sales on lead-quality definitions and the SDR-handoff process. Recent focus has been on account-based marketing for our enterprise segment."
      },
      {
        "company": "Stark Industries",
        "title": "Graphic Designer",
        "start_date": "2020-07-27",
        "end_date": "2023-04-13",
        "duration_months": 33,
        "is_current": false,
        "industry": "Manufacturing",
        "company_size": "1001-5000",
        "description": "Marketing leadership role at a B2B SaaS company. Owned the demand-generation function \u2014 content marketing, paid acquisition, SEO, email nurture. Built and managed a team of 5 across content, performance marketing, and marketing operations. Worked closely with sales on lead-quality definitions and the SDR-handoff process. Recent focus has been on account-based marketing for our enterprise segment."
      },
      {
        "company": "Dunder Mifflin",
        "title": "Civil Engineer",
        "start_date": "2020-01-29",
        "end_date": "2020-07-27",
        "duration_months": 6,
        "is_current": false,
        "industry": "Paper Products",
        "company_size": "201-500",
        "description": "Content writing and SEO strategy for a tech-focused publication. Wrote longform articles on developer tools, cloud platforms, and AI/ML topics \u2014 including some that ranked on the first page of search for high-competition keywords. Managed a freelance writer pool and the editorial calendar. Recent work has been on AI-assisted content production, using LLM tools for research, drafting, and editing while maintaining editorial quality."
      }
    ],
    "education": [
      {
        "institution": "IIT Kharagpur",
        "degree": "B.E.",
        "field_of_study": "Computer Science",
        "start_year": 2018,
        "end_year": 2022,
        "grade": "6.67 CGPA",
        "tier": "tier_1"
      }
    ],
    "skills": [
      {
        "name": "GraphQL",
        "proficiency": "beginner",
        "endorsements": 15,
        "duration_months": 18
      },
      {
        "name": "TypeScript",
        "proficiency": "beginner",
        "endorsements": 7,
        "duration_months": 8
      },
      {
        "name": "Flask",
        "proficiency": "beginner",
        "endorsements": 11,
        "duration_months": 16
      },
      {
        "name": "Weights & Biases",
        "proficiency": "advanced",
        "endorsements": 50,
        "duration_months": 30
      },
      {
        "name": "GCP",
        "proficiency": "intermediate",
        "endorsements": 2,
        "duration_months": 34
      },
      {
        "name": "Salesforce CRM",
        "proficiency": "beginner",
        "endorsements": 8,
        "duration_months": 16
      },
      {
        "name": "HTML",
        "proficiency": "intermediate",
        "endorsements": 2,
        "duration_months": 35
      }
    ],
    "certifications": [
      {
        "name": "Scrum Master Certified",
        "issuer": "Scrum Alliance",
        "year": 2021
      }
    ],
    "languages": [
      {
        "language": "English",
        "proficiency": "professional"
      },
      {
        "language": "Hindi",
        "proficiency": "conversational"
      }
    ],
    "redrob_signals": {
      "profile_completeness_score": 73.0,
      "signup_date": "2023-01-26",
      "last_active_date": "2025-10-05",
      "open_to_work_flag": false,
      "profile_views_received_30d": 38,
      "applications_submitted_30d": 9,
      "recruiter_response_rate": 0.55,
      "avg_response_time_hours": 207.8,
      "skill_assessment_scores": {
        "Weights & Biases": 53.7
      },
      "connection_count": 479,
      "endorsements_received": 35,
      "notice_period_days": 30,
      "expected_salary_range_inr_lpa": {
        "min": 17.2,
        "max": 18.2
      },
      "preferred_work_mode": "remote",
      "willing_to_relocate": false,
      "github_activity_score": -1,
      "search_appearance_30d": 31,
      "saved_by_recruiters_30d": 11,
      "interview_completion_rate": 0.71,
      "offer_acceptance_rate": -1,
      "verified_email": true,
      "verified_phone": false,
      "linkedin_connected": true
    }
  },
  {
    "candidate_id": "CAND_0000021",
    "profile": {
      "anonymized_name": "Rahul Joshi",
      "headline": "Project Manager | AI enthusiast | Building with LLMs",
      "summary": "Project Manager with 14.5+ years of experience driving outcomes in my domain. I have built strong functional expertise in the typical responsibilities of the role, including team management, stakeholder communication, and project delivery. Recently I've been excited about how AI and GenAI tools can augment my work. I've been taking online courses on RAG and vector databases, experimenting with LangChain and the OpenAI API for side projects, and exploring how LLMs can streamline workflows in my current function. Open to roles that combine my existing domain experience with emerging AI technologies \u2014 I think the most interesting opportunities are at this intersection. Looking for positions where I can contribute both my functional expertise and grow my AI capabilities.",
      "location": "Bhubaneswar, Odisha",
      "country": "India",
      "years_of_experience": 14.5,
      "current_title": "Project Manager",
      "current_company": "Wipro",
      "current_company_size": "10001+",
      "current_industry": "IT Services"
    },
    "career_history": [
      {
        "company": "Wipro",
        "title": "Project Manager",
        "start_date": "2023-12-09",
        "end_date": null,
        "duration_months": 30,
        "is_current": true,
        "industry": "IT Services",
        "company_size": "10001+",
        "description": "Brand design and creative direction at a consumer-products company. Owned brand identity (logo, visual system, typography), packaging design, and digital creative across web and social. Led the recent rebrand and managed a small external agency for production work. Comfortable across the Adobe suite, Figma, and the production side of brand and packaging design."
      },
      {
        "company": "Infosys",
        "title": "Marketing Manager",
        "start_date": "2021-02-22",
        "end_date": "2023-10-10",
        "duration_months": 32,
        "is_current": false,
        "industry": "IT Services",
        "company_size": "10001+",
        "description": "Mechanical engineering design role at a hardware-product company. Led the design of two product subsystems through full lifecycle: concept, DFM/DFMA review, prototype, production tooling. Comfortable with CAD (SolidWorks, Creo), FEA (ANSYS), and the typical hardware-development cadence. Worked closely with manufacturing partners on production scale-up."
      },
      {
        "company": "Stark Industries",
        "title": "Sales Executive",
        "start_date": "2019-08-25",
        "end_date": "2021-02-15",
        "duration_months": 18,
        "is_current": false,
        "industry": "Manufacturing",
        "company_size": "1001-5000",
        "description": "Customer support team lead at a SaaS product. Managed a team of 8 support agents handling tier-1 and tier-2 tickets; owned the escalation process to engineering and the customer-feedback loop to product. Built out the support knowledge base and the agent training program. Strong on the people-management side and the process side; lighter on technical depth beyond product expertise."
      },
      {
        "company": "Dunder Mifflin",
        "title": "Customer Support",
        "start_date": "2015-06-17",
        "end_date": "2019-08-25",
        "duration_months": 51,
        "is_current": false,
        "industry": "Paper Products",
        "company_size": "201-500",
        "description": "Customer support team lead at a SaaS product. Managed a team of 8 support agents handling tier-1 and tier-2 tickets; owned the escalation process to engineering and the customer-feedback loop to product. Built out the support knowledge base and the agent training program. Strong on the people-management side and the process side; lighter on technical depth beyond product expertise."
      },
      {
        "company": "Wipro",
        "title": "Project Manager",
        "start_date": "2014-03-24",
        "end_date": "2015-06-17",
        "duration_months": 15,
        "is_current": false,
        "industry": "IT Services",
        "company_size": "10001+",
        "description": "Customer support team lead at a SaaS product. Managed a team of 8 support agents handling tier-1 and tier-2 tickets; owned the escalation process to engineering and the customer-feedback loop to product. Built out the support knowledge base and the agent training program. Strong on the people-management side and the process side; lighter on technical depth beyond product expertise."
      },
      {
        "company": "TCS",
        "title": "Customer Support",
        "start_date": "2011-12-05",
        "end_date": "2014-01-23",
        "duration_months": 26,
        "is_current": false,
        "industry": "IT Services",
        "company_size": "10001+",
        "description": "Business analyst at a consulting firm, working primarily with retail and CPG clients. Conducted business diagnostics, process re-engineering work, and digital transformation strategy projects. Strong on stakeholder management, structured problem-solving, and the typical consulting toolkit (slide-craft, Excel modeling, executive communication). Recent project work involved AI-strategy advisory but my own technical depth in AI is limited."
      }
    ],
    "education": [
      {
        "institution": "Tier-3 Engineering College",
        "degree": "B.Tech",
        "field_of_study": "Artificial Intelligence",
        "start_year": 2008,
        "end_year": 2011,
        "grade": "9.30 CGPA",
        "tier": "tier_4"
      }
    ],
    "skills": [
      {
        "name": "Hadoop",
        "proficiency": "beginner",
        "endorsements": 10,
        "duration_months": 5
      },
      {
        "name": "PostgreSQL",
        "proficiency": "beginner",
        "endorsements": 10,
        "duration_months": 4
      },
      {
        "name": "Kafka",
        "proficiency": "beginner",
        "endorsements": 6,
        "duration_months": 6
      },
      {
        "name": "Microservices",
        "proficiency": "intermediate",
        "endorsements": 0,
        "duration_months": 14
      },
      {
        "name": "AWS",
        "proficiency": "intermediate",
        "endorsements": 11,
        "duration_months": 26
      },
      {
        "name": "TypeScript",
        "proficiency": "beginner",
        "endorsements": 6,
        "duration_months": 6
      },
      {
        "name": "ETL",
        "proficiency": "beginner",
        "endorsements": 11,
        "duration_months": 3
      },
      {
        "name": "Spring Boot",
        "proficiency": "beginner",
        "endorsements": 1,
        "duration_months": 12
      },
      {
        "name": "Recommendation Systems",
        "proficiency": "advanced",
        "endorsements": 3,
        "duration_months": 13
      },
      {
        "name": "Fine-tuning LLMs",
        "proficiency": "advanced",
        "endorsements": 4,
        "duration_months": 4
      },
      {
        "name": "Prompt Engineering",
        "proficiency": "advanced",
        "endorsements": 4,
        "duration_months": 5
      },
      {
        "name": "LangChain",
        "proficiency": "intermediate",
        "endorsements": 1,
        "duration_months": 7
      },
      {
        "name": "Pinecone",
        "proficiency": "intermediate",
        "endorsements": 4,
        "duration_months": 16
      },
      {
        "name": "Vector Search",
        "proficiency": "intermediate",
        "endorsements": 3,
        "duration_months": 13
      },
      {
        "name": "Embeddings",
        "proficiency": "advanced",
        "endorsements": 4,
        "duration_months": 18
      },
      {
        "name": "FAISS",
        "proficiency": "intermediate",
        "endorsements": 2,
        "duration_months": 8
      }
    ],
    "certifications": [],
    "languages": [
      {
        "language": "English",
        "proficiency": "professional"
      },
      {
        "language": "Hindi",
        "proficiency": "professional"
      }
    ],
    "redrob_signals": {
      "profile_completeness_score": 58.5,
      "signup_date": "2026-02-10",
      "last_active_date": "2025-11-21",
      "open_to_work_flag": false,
      "profile_views_received_30d": 1,
      "applications_submitted_30d": 8,
      "recruiter_response_rate": 0.49,
      "avg_response_time_hours": 98.7,
      "skill_assessment_scores": {},
      "connection_count": 52,
      "endorsements_received": 3,
      "notice_period_days": 60,
      "expected_salary_range_inr_lpa": {
        "min": 10.9,
        "max": 24.4
      },
      "preferred_work_mode": "hybrid",
      "willing_to_relocate": true,
      "github_activity_score": 6.4,
      "search_appearance_30d": 8,
      "saved_by_recruiters_30d": 3,
      "interview_completion_rate": 0.53,
      "offer_acceptance_rate": -1,
      "verified_email": true,
      "verified_phone": true,
      "linkedin_connected": true
    }
  },
  {
    "candidate_id": "CAND_0000022",
    "profile": {
      "anonymized_name": "Shaurya Chatterjee",
      "headline": "Mechanical Engineer | Driving business outcomes",
      "summary": "Professional with 1.1+ years of experience. I'm a marketing manager with substantial experience in cross-functional collaboration, stakeholder management, and execution. Lately I've been curious about how AI tools could augment my work \u2014 I've experimented with ChatGPT and a few other tools for productivity and content creation, and I think the space is exciting. Open to roles where I can apply my domain expertise alongside emerging AI capabilities.",
      "location": "Sydney",
      "country": "Australia",
      "years_of_experience": 1.1,
      "current_title": "Mechanical Engineer",
      "current_company": "Hooli",
      "current_company_size": "1001-5000",
      "current_industry": "Software"
    },
    "career_history": [
      {
        "company": "Hooli",
        "title": "Mechanical Engineer",
        "start_date": "2025-05-02",
        "end_date": null,
        "duration_months": 13,
        "is_current": true,
        "industry": "Software",
        "company_size": "1001-5000",
        "description": "Content writing and SEO strategy for a tech-focused publication. Wrote longform articles on developer tools, cloud platforms, and AI/ML topics \u2014 including some that ranked on the first page of search for high-competition keywords. Managed a freelance writer pool and the editorial calendar. Recent work has been on AI-assisted content production, using LLM tools for research, drafting, and editing while maintaining editorial quality."
      }
    ],
    "education": [
      {
        "institution": "VJTI Mumbai",
        "degree": "M.E.",
        "field_of_study": "Information Technology",
        "start_year": 2002,
        "end_year": 2006,
        "grade": "8.23 CGPA",
        "tier": "tier_2"
      }
    ],
    "skills": [
      {
        "name": "OpenCV",
        "proficiency": "intermediate",
        "endorsements": 2,
        "duration_months": 11
      },
      {
        "name": "Django",
        "proficiency": "intermediate",
        "endorsements": 0,
        "duration_months": 23
      },
      {
        "name": "Terraform",
        "proficiency": "intermediate",
        "endorsements": 11,
        "duration_months": 14
      },
      {
        "name": "Scrum",
        "proficiency": "intermediate",
        "endorsements": 7,
        "duration_months": 18
      },
      {
        "name": "SQL",
        "proficiency": "intermediate",
        "endorsements": 3,
        "duration_months": 36
      },
      {
        "name": "Java",
        "proficiency": "beginner",
        "endorsements": 11,
        "duration_months": 15
      },
      {
        "name": "AWS",
        "proficiency": "intermediate",
        "endorsements": 5,
        "duration_months": 29
      },
      {
        "name": "Six Sigma",
        "proficiency": "beginner",
        "endorsements": 1,
        "duration_months": 4
      }
    ],
    "certifications": [
      {
        "name": "AWS Certified Cloud Practitioner",
        "issuer": "AWS",
        "year": 2024
      }
    ],
    "languages": [
      {
        "language": "English",
        "proficiency": "professional"
      },
      {
        "language": "Hindi",
        "proficiency": "conversational"
      }
    ],
    "redrob_signals": {
      "profile_completeness_score": 63.1,
      "signup_date": "2022-12-26",
      "last_active_date": "2026-05-03",
      "open_to_work_flag": true,
      "profile_views_received_30d": 39,
      "applications_submitted_30d": 2,
      "recruiter_response_rate": 0.27,
      "avg_response_time_hours": 30.2,
      "skill_assessment_scores": {},
      "connection_count": 538,
      "endorsements_received": 21,
      "notice_period_days": 150,
      "expected_salary_range_inr_lpa": {
        "min": 12.3,
        "max": 8.5
      },
      "preferred_work_mode": "remote",
      "willing_to_relocate": false,
      "github_activity_score": -1,
      "search_appearance_30d": 136,
      "saved_by_recruiters_30d": 5,
      "interview_completion_rate": 0.45,
      "offer_acceptance_rate": -1,
      "verified_email": true,
      "verified_phone": false,
      "linkedin_connected": false
    }
  },
  {
    "candidate_id": "CAND_0000023",
    "profile": {
      "anonymized_name": "Kavya Nair",
      "headline": "Software Engineer | Cloud & DevOps",
      "summary": "Software engineer with 3.7 years of experience across web, backend, and cloud systems. Strong fundamentals in software development and system design. Most of my work is conventional backend engineering \u2014 APIs, databases, queues, deployments. I've been keeping up with AI/ML at a self-learner level \u2014 taken some online courses, played with the OpenAI and Anthropic APIs, built a small RAG side project \u2014 but I haven't done it in a professional capacity yet. Open to roles where I can either deepen my software engineering work or, if the team is open to it, start contributing to ML-adjacent systems.",
      "location": "New York",
      "country": "USA",
      "years_of_experience": 3.7,
      "current_title": "Software Engineer",
      "current_company": "Acme Corp",
      "current_company_size": "201-500",
      "current_industry": "Manufacturing"
    },
    "career_history": [
      {
        "company": "Acme Corp",
        "title": "Software Engineer",
        "start_date": "2025-04-02",
        "end_date": null,
        "duration_months": 14,
        "is_current": true,
        "industry": "Manufacturing",
        "company_size": "201-500",
        "description": "Test automation and QA engineering for a fintech product. Built and maintained the end-to-end test suite using Selenium and pytest, plus the load-testing setup using Locust. Worked closely with developers on testability patterns and with product on acceptance criteria. Recent work has been on shifting test responsibility into the dev team \u2014 moving from QA-as-gate to QA-as-coach. Career has been entirely in QA/test engineering."
      },
      {
        "company": "Flipkart",
        "title": "Frontend Engineer",
        "start_date": "2022-10-15",
        "end_date": "2025-04-02",
        "duration_months": 30,
        "is_current": false,
        "industry": "E-commerce",
        "company_size": "10001+",
        "description": "Android mobile development using Java and (more recently) Kotlin at a consumer-app company. Built and maintained multiple production features including the main shopping flow, push notification system, and the offline-first sync layer. Comfortable with the Android framework, Jetpack components, and the typical patterns (MVVM, Hilt, Coroutines). My career has been entirely on mobile so far; interested in expanding into broader backend or platform engineering."
      }
    ],
    "education": [
      {
        "institution": "VJTI Mumbai",
        "degree": "B.E.",
        "field_of_study": "Data Science",
        "start_year": 2009,
        "end_year": 2013,
        "grade": "9.43 CGPA",
        "tier": "tier_2"
      }
    ],
    "skills": [
      {
        "name": "BigQuery",
        "proficiency": "beginner",
        "endorsements": 8,
        "duration_months": 6
      },
      {
        "name": "Marketing",
        "proficiency": "intermediate",
        "endorsements": 0,
        "duration_months": 15
      },
      {
        "name": "Node.js",
        "proficiency": "intermediate",
        "endorsements": 3,
        "duration_months": 28
      },
      {
        "name": "Django",
        "proficiency": "beginner",
        "endorsements": 13,
        "duration_months": 16
      },
      {
        "name": "Salesforce CRM",
        "proficiency": "beginner",
        "endorsements": 9,
        "duration_months": 18
      },
      {
        "name": "MongoDB",
        "proficiency": "beginner",
        "endorsements": 8,
        "duration_months": 10
      },
      {
        "name": "ETL",
        "proficiency": "beginner",
        "endorsements": 12,
        "duration_months": 16
      },
      {
        "name": "Redis",
        "proficiency": "beginner",
        "endorsements": 4,
        "duration_months": 10
      },
      {
        "name": "Illustrator",
        "proficiency": "beginner",
        "endorsements": 1,
        "duration_months": 2
      },
      {
        "name": "Rust",
        "proficiency": "intermediate",
        "endorsements": 2,
        "duration_months": 16
      }
    ],
    "certifications": [],
    "languages": [
      {
        "language": "English",
        "proficiency": "native"
      },
      {
        "language": "Hindi",
        "proficiency": "native"
      }
    ],
    "redrob_signals": {
      "profile_completeness_score": 50.7,
      "signup_date": "2025-09-13",
      "last_active_date": "2026-04-06",
      "open_to_work_flag": false,
      "profile_views_received_30d": 10,
      "applications_submitted_30d": 2,
      "recruiter_response_rate": 0.57,
      "avg_response_time_hours": 64.8,
      "skill_assessment_scores": {},
      "connection_count": 763,
      "endorsements_received": 39,
      "notice_period_days": 30,
      "expected_salary_range_inr_lpa": {
        "min": 17.4,
        "max": 20.5
      },
      "preferred_work_mode": "flexible",
      "willing_to_relocate": false,
      "github_activity_score": 48.5,
      "search_appearance_30d": 239,
      "saved_by_recruiters_30d": 14,
      "interview_completion_rate": 0.9,
      "offer_acceptance_rate": -1,
      "verified_email": true,
      "verified_phone": true,
      "linkedin_connected": false
    }
  },
  {
    "candidate_id": "CAND_0000024",
    "profile": {
      "anonymized_name": "Rajesh Arora",
      "headline": "HR Manager | 7.5+ yrs experience",
      "summary": "Professional with 7.5+ years of experience. I've spent my career in marketing manager roles, mostly focused on driving outcomes through process, people, and customer relationships. Lately I've been curious about how AI tools could augment my work \u2014 I've experimented with ChatGPT and a few other tools for productivity and content creation, and I think the space is exciting. Open to roles where I can apply my domain expertise alongside emerging AI capabilities.",
      "location": "Trivandrum, Kerala",
      "country": "India",
      "years_of_experience": 7.5,
      "current_title": "HR Manager",
      "current_company": "TCS",
      "current_company_size": "10001+",
      "current_industry": "IT Services"
    },
    "career_history": [
      {
        "company": "TCS",
        "title": "HR Manager",
        "start_date": "2023-04-13",
        "end_date": null,
        "duration_months": 38,
        "is_current": true,
        "industry": "IT Services",
        "company_size": "10001+",
        "description": "Marketing leadership role at a B2B SaaS company. Owned the demand-generation function \u2014 content marketing, paid acquisition, SEO, email nurture. Built and managed a team of 5 across content, performance marketing, and marketing operations. Worked closely with sales on lead-quality definitions and the SDR-handoff process. Recent focus has been on account-based marketing for our enterprise segment."
      },
      {
        "company": "Infosys",
        "title": "Accountant",
        "start_date": "2018-12-05",
        "end_date": "2023-02-12",
        "duration_months": 51,
        "is_current": false,
        "industry": "IT Services",
        "company_size": "10001+",
        "description": "Content writing and SEO strategy for a tech-focused publication. Wrote longform articles on developer tools, cloud platforms, and AI/ML topics \u2014 including some that ranked on the first page of search for high-competition keywords. Managed a freelance writer pool and the editorial calendar. Recent work has been on AI-assisted content production, using LLM tools for research, drafting, and editing while maintaining editorial quality."
      }
    ],
    "education": [
      {
        "institution": "Symbiosis International",
        "degree": "Ph.D",
        "field_of_study": "Computer Science",
        "start_year": 2008,
        "end_year": 2013,
        "grade": "7.65 CGPA",
        "tier": "tier_3"
      }
    ],
    "skills": [
      {
        "name": "Figma",
        "proficiency": "beginner",
        "endorsements": 14,
        "duration_months": 15
      },
      {
        "name": "Kubernetes",
        "proficiency": "beginner",
        "endorsements": 8,
        "duration_months": 17
      },
      {
        "name": "Forecasting",
        "proficiency": "advanced",
        "endorsements": 43,
        "duration_months": 30
      },
      {
        "name": "ETL",
        "proficiency": "intermediate",
        "endorsements": 11,
        "duration_months": 12
      },
      {
        "name": "Node.js",
        "proficiency": "intermediate",
        "endorsements": 3,
        "duration_months": 15
      },
      {
        "name": "Docker",
        "proficiency": "beginner",
        "endorsements": 5,
        "duration_months": 5
      }
    ],
    "certifications": [],
    "languages": [
      {
        "language": "English",
        "proficiency": "professional"
      },
      {
        "language": "Hindi",
        "proficiency": "native"
      }
    ],
    "redrob_signals": {
      "profile_completeness_score": 63.7,
      "signup_date": "2022-08-30",
      "last_active_date": "2026-01-20",
      "open_to_work_flag": false,
      "profile_views_received_30d": 71,
      "applications_submitted_30d": 4,
      "recruiter_response_rate": 0.78,
      "avg_response_time_hours": 238.2,
      "skill_assessment_scores": {
        "Forecasting": 65.1
      },
      "connection_count": 445,
      "endorsements_received": 41,
      "notice_period_days": 60,
      "expected_salary_range_inr_lpa": {
        "min": 9.9,
        "max": 22.1
      },
      "preferred_work_mode": "flexible",
      "willing_to_relocate": true,
      "github_activity_score": 46.3,
      "search_appearance_30d": 84,
      "saved_by_recruiters_30d": 7,
      "interview_completion_rate": 0.72,
      "offer_acceptance_rate": -1,
      "verified_email": false,
      "verified_phone": true,
      "linkedin_connected": false
    }
  },
  {
    "candidate_id": "CAND_0000025",
    "profile": {
      "anonymized_name": "Anika Kumar",
      "headline": "Frontend Engineer | Cloud & DevOps",
      "summary": "Software engineer with 7.3 years of experience across web, backend, and cloud systems. Strong fundamentals in software development and system design. I've worked across web frontends, REST APIs, and cloud deployments; comfortable in most parts of a typical SaaS stack. I've been keeping up with AI/ML at a self-learner level \u2014 taken some online courses, played with the OpenAI and Anthropic APIs, built a small RAG side project \u2014 but I haven't done it in a professional capacity yet. Open to roles where I can either deepen my software engineering work or, if the team is open to it, start contributing to ML-adjacent systems.",
      "location": "Vizag, Andhra Pradesh",
      "country": "India",
      "years_of_experience": 7.3,
      "current_title": "Frontend Engineer",
      "current_company": "Tech Mahindra",
      "current_company_size": "10001+",
      "current_industry": "IT Services"
    },
    "career_history": [
      {
        "company": "Tech Mahindra",
        "title": "Frontend Engineer",
        "start_date": "2023-09-10",
        "end_date": null,
        "duration_months": 33,
        "is_current": true,
        "industry": "IT Services",
        "company_size": "10001+",
        "description": "Android mobile development using Java and (more recently) Kotlin at a consumer-app company. Built and maintained multiple production features including the main shopping flow, push notification system, and the offline-first sync layer. Comfortable with the Android framework, Jetpack components, and the typical patterns (MVVM, Hilt, Coroutines). My career has been entirely on mobile so far; interested in expanding into broader backend or platform engineering."
      },
      {
        "company": "Mindtree",
        "title": "Frontend Engineer",
        "start_date": "2019-09-17",
        "end_date": "2023-08-27",
        "duration_months": 48,
        "is_current": false,
        "industry": "IT Services",
        "company_size": "10001+",
        "description": "Android mobile development using Java and (more recently) Kotlin at a consumer-app company. Built and maintained multiple production features including the main shopping flow, push notification system, and the offline-first sync layer. Comfortable with the Android framework, Jetpack components, and the typical patterns (MVVM, Hilt, Coroutines). My career has been entirely on mobile so far; interested in expanding into broader backend or platform engineering."
      },
      {
        "company": "Zomato",
        "title": "Frontend Engineer",
        "start_date": "2019-03-21",
        "end_date": "2019-09-17",
        "duration_months": 6,
        "is_current": false,
        "industry": "Food Delivery",
        "company_size": "5001-10000",
        "description": "Frontend engineering at a media company. React, TypeScript, and the typical surrounding tooling (Webpack, Jest, Cypress). Built the company's design system from scratch and led the migration from a legacy AngularJS app. Strong on the frontend craft \u2014 accessibility, performance, animations \u2014 but limited backend exposure."
      }
    ],
    "education": [
      {
        "institution": "Regional Technical Institute",
        "degree": "Ph.D",
        "field_of_study": "Mechanical Engineering",
        "start_year": 2006,
        "end_year": 2010,
        "grade": "8.18 CGPA",
        "tier": "tier_4"
      }
    ],
    "skills": [
      {
        "name": "JavaScript",
        "proficiency": "intermediate",
        "endorsements": 10,
        "duration_months": 26
      },
      {
        "name": "Spark",
        "proficiency": "intermediate",
        "endorsements": 0,
        "duration_months": 22
      },
      {
        "name": "GCP",
        "proficiency": "beginner",
        "endorsements": 7,
        "duration_months": 13
      },
      {
        "name": "TypeScript",
        "proficiency": "beginner",
        "endorsements": 2,
        "duration_months": 17
      },
      {
        "name": "LangChain",
        "proficiency": "advanced",
        "endorsements": 15,
        "duration_months": 34
      },
      {
        "name": "Apache Flink",
        "proficiency": "intermediate",
        "endorsements": 2,
        "duration_months": 19
      },
      {
        "name": "ETL",
        "proficiency": "beginner",
        "endorsements": 1,
        "duration_months": 18
      },
      {
        "name": "Redis",
        "proficiency": "beginner",
        "endorsements": 0,
        "duration_months": 10
      },
      {
        "name": "Data Pipelines",
        "proficiency": "intermediate",
        "endorsements": 9,
        "duration_months": 32
      }
    ],
    "certifications": [
      {
        "name": "Six Sigma Green Belt",
        "issuer": "ASQ",
        "year": 2018
      },
      {
        "name": "AWS Certified Cloud Practitioner",
        "issuer": "AWS",
        "year": 2025
      }
    ],
    "languages": [
      {
        "language": "English",
        "proficiency": "professional"
      },
      {
        "language": "Hindi",
        "proficiency": "conversational"
      }
    ],
    "redrob_signals": {
      "profile_completeness_score": 70.7,
      "signup_date": "2023-12-18",
      "last_active_date": "2026-03-30",
      "open_to_work_flag": true,
      "profile_views_received_30d": 107,
      "applications_submitted_30d": 11,
      "recruiter_response_rate": 0.74,
      "avg_response_time_hours": 128.0,
      "skill_assessment_scores": {
        "LangChain": 40.0
      },
      "connection_count": 276,
      "endorsements_received": 52,
      "notice_period_days": 120,
      "expected_salary_range_inr_lpa": {
        "min": 18.8,
        "max": 30.7
      },
      "preferred_work_mode": "hybrid",
      "willing_to_relocate": false,
      "github_activity_score": -1,
      "search_appearance_30d": 74,
      "saved_by_recruiters_30d": 9,
      "interview_completion_rate": 0.7,
      "offer_acceptance_rate": -1,
      "verified_email": true,
      "verified_phone": true,
      "linkedin_connected": true
    }
  },
  {
    "candidate_id": "CAND_0000026",
    "profile": {
      "anonymized_name": "Neha Rao",
      "headline": "Graphic Designer | 6.8+ yrs experience",
      "summary": "Professional with 6.8+ years of experience. My professional background is in marketing manager \u2014 I've built and led teams, owned KPIs, and driven business outcomes in this domain. Lately I've been curious about how AI tools could augment my work \u2014 I've experimented with ChatGPT and a few other tools for productivity and content creation, and I think the space is exciting. Open to roles where I can apply my domain expertise alongside emerging AI capabilities.",
      "location": "Kochi, Kerala",
      "country": "India",
      "years_of_experience": 6.8,
      "current_title": "Graphic Designer",
      "current_company": "Initech",
      "current_company_size": "51-200",
      "current_industry": "Software"
    },
    "career_history": [
      {
        "company": "Initech",
        "title": "Graphic Designer",
        "start_date": "2022-11-14",
        "end_date": null,
        "duration_months": 43,
        "is_current": true,
        "industry": "Software",
        "company_size": "51-200",
        "description": "Customer support team lead at a SaaS product. Managed a team of 8 support agents handling tier-1 and tier-2 tickets; owned the escalation process to engineering and the customer-feedback loop to product. Built out the support knowledge base and the agent training program. Strong on the people-management side and the process side; lighter on technical depth beyond product expertise."
      },
      {
        "company": "Acme Corp",
        "title": "Accountant",
        "start_date": "2021-03-24",
        "end_date": "2022-11-14",
        "duration_months": 20,
        "is_current": false,
        "industry": "Manufacturing",
        "company_size": "201-500",
        "description": "Content writing and SEO strategy for a tech-focused publication. Wrote longform articles on developer tools, cloud platforms, and AI/ML topics \u2014 including some that ranked on the first page of search for high-competition keywords. Managed a freelance writer pool and the editorial calendar. Recent work has been on AI-assisted content production, using LLM tools for research, drafting, and editing while maintaining editorial quality."
      },
      {
        "company": "Acme Corp",
        "title": "Project Manager",
        "start_date": "2019-10-31",
        "end_date": "2021-03-24",
        "duration_months": 17,
        "is_current": false,
        "industry": "Manufacturing",
        "company_size": "201-500",
        "description": "Enterprise sales of cloud software solutions into the mid-market segment. Carried a $1.8M ARR quota and consistently delivered against it across the last three years. Owned the full sales cycle: prospecting, discovery, technical evaluation (with SE support), commercial negotiation, and close. Strong on consultative selling for technical buyers; comfortable engaging with both engineering and finance stakeholders."
      }
    ],
    "education": [
      {
        "institution": "Generic State University",
        "degree": "M.Sc",
        "field_of_study": "Statistics",
        "start_year": 2008,
        "end_year": 2012,
        "grade": "81%",
        "tier": "tier_4"
      },
      {
        "institution": "Lovely Professional University",
        "degree": "B.E.",
        "field_of_study": "Machine Learning",
        "start_year": 2008,
        "end_year": 2013,
        "grade": "83%",
        "tier": "tier_3"
      }
    ],
    "skills": [
      {
        "name": "Apache Beam",
        "proficiency": "intermediate",
        "endorsements": 4,
        "duration_months": 18
      },
      {
        "name": "Kubeflow",
        "proficiency": "intermediate",
        "endorsements": 14,
        "duration_months": 27
      },
      {
        "name": "Scrum",
        "proficiency": "beginner",
        "endorsements": 12,
        "duration_months": 8
      },
      {
        "name": "ETL",
        "proficiency": "beginner",
        "endorsements": 15,
        "duration_months": 17
      },
      {
        "name": "Django",
        "proficiency": "beginner",
        "endorsements": 6,
        "duration_months": 11
      },
      {
        "name": "Docker",
        "proficiency": "beginner",
        "endorsements": 4,
        "duration_months": 9
      },
      {
        "name": "Airflow",
        "proficiency": "intermediate",
        "endorsements": 7,
        "duration_months": 21
      },
      {
        "name": "Kubernetes",
        "proficiency": "intermediate",
        "endorsements": 13,
        "duration_months": 22
      }
    ],
    "certifications": [],
    "languages": [
      {
        "language": "English",
        "proficiency": "native"
      },
      {
        "language": "Hindi",
        "proficiency": "native"
      }
    ],
    "redrob_signals": {
      "profile_completeness_score": 80.3,
      "signup_date": "2023-12-08",
      "last_active_date": "2025-10-03",
      "open_to_work_flag": false,
      "profile_views_received_30d": 75,
      "applications_submitted_30d": 7,
      "recruiter_response_rate": 0.59,
      "avg_response_time_hours": 45.4,
      "skill_assessment_scores": {},
      "connection_count": 321,
      "endorsements_received": 8,
      "notice_period_days": 30,
      "expected_salary_range_inr_lpa": {
        "min": 17.1,
        "max": 8.0
      },
      "preferred_work_mode": "hybrid",
      "willing_to_relocate": false,
      "github_activity_score": -1,
      "search_appearance_30d": 154,
      "saved_by_recruiters_30d": 11,
      "interview_completion_rate": 0.49,
      "offer_acceptance_rate": -1,
      "verified_email": true,
      "verified_phone": false,
      "linkedin_connected": true
    }
  },
  {
    "candidate_id": "CAND_0000027",
    "profile": {
      "anonymized_name": "Avni Pandey",
      "headline": "DevOps Engineer | Cloud & DevOps",
      "summary": "Software engineer with 3.9 years of experience across web, backend, and cloud systems. Strong fundamentals in software development and system design. I've worked across web frontends, REST APIs, and cloud deployments; comfortable in most parts of a typical SaaS stack. I've been keeping up with AI/ML at a self-learner level \u2014 taken some online courses, played with the OpenAI and Anthropic APIs, built a small RAG side project \u2014 but I haven't done it in a professional capacity yet. Open to roles where I can either deepen my software engineering work or, if the team is open to it, start contributing to ML-adjacent systems.",
      "location": "Kolkata, West Bengal",
      "country": "India",
      "years_of_experience": 3.9,
      "current_title": "DevOps Engineer",
      "current_company": "Infosys",
      "current_company_size": "10001+",
      "current_industry": "IT Services"
    },
    "career_history": [
      {
        "company": "Infosys",
        "title": "DevOps Engineer",
        "start_date": "2023-06-12",
        "end_date": null,
        "duration_months": 36,
        "is_current": true,
        "industry": "IT Services",
        "company_size": "10001+",
        "description": "Java backend development at a large enterprise \u2014 Spring Boot microservices, Kafka for inter-service messaging, Postgres + Redis for storage. Worked on the customer onboarding flow which involved orchestrating multiple downstream services. Solid on the Spring ecosystem, transaction handling, and the operational side of Java services. Looking to either go deeper on distributed systems or expand into modern application stacks."
      },
      {
        "company": "Wipro",
        "title": "DevOps Engineer",
        "start_date": "2022-08-02",
        "end_date": "2023-05-29",
        "duration_months": 10,
        "is_current": false,
        "industry": "IT Services",
        "company_size": "10001+",
        "description": "Full-stack web application development at a SaaS company. Built React-based admin interfaces and the Node.js REST API backing them. Worked across the stack: frontend components, REST endpoint design, PostgreSQL schema, deployment via Docker/Kubernetes. Comfortable in most parts of a typical web stack though my comfort zone is the backend and database side. Recent learning has been on the testing and CI/CD discipline."
      }
    ],
    "education": [
      {
        "institution": "IIT Bombay",
        "degree": "Ph.D",
        "field_of_study": "Information Technology",
        "start_year": 2006,
        "end_year": 2010,
        "grade": "6.55 CGPA",
        "tier": "tier_1"
      },
      {
        "institution": "VJTI Mumbai",
        "degree": "B.E.",
        "field_of_study": "Artificial Intelligence",
        "start_year": 2017,
        "end_year": 2020,
        "grade": "9.11 CGPA",
        "tier": "tier_2"
      }
    ],
    "skills": [
      {
        "name": "Docker",
        "proficiency": "intermediate",
        "endorsements": 1,
        "duration_months": 9
      },
      {
        "name": "YOLO",
        "proficiency": "advanced",
        "endorsements": 31,
        "duration_months": 20
      },
      {
        "name": "PEFT",
        "proficiency": "advanced",
        "endorsements": 39,
        "duration_months": 35
      },
      {
        "name": "Webpack",
        "proficiency": "intermediate",
        "endorsements": 0,
        "duration_months": 33
      },
      {
        "name": "Data Science",
        "proficiency": "advanced",
        "endorsements": 18,
        "duration_months": 24
      },
      {
        "name": "AWS",
        "proficiency": "beginner",
        "endorsements": 4,
        "duration_months": 16
      },
      {
        "name": "Java",
        "proficiency": "intermediate",
        "endorsements": 2,
        "duration_months": 15
      },
      {
        "name": "Angular",
        "proficiency": "intermediate",
        "endorsements": 4,
        "duration_months": 25
      },
      {
        "name": "Databricks",
        "proficiency": "intermediate",
        "endorsements": 3,
        "duration_months": 31
      },
      {
        "name": "ETL",
        "proficiency": "intermediate",
        "endorsements": 5,
        "duration_months": 11
      },
      {
        "name": "Marketing",
        "proficiency": "beginner",
        "endorsements": 15,
        "duration_months": 14
      },
      {
        "name": "Information Retrieval",
        "proficiency": "intermediate",
        "endorsements": 5,
        "duration_months": 32
      },
      {
        "name": "Weights & Biases",
        "proficiency": "advanced",
        "endorsements": 31,
        "duration_months": 44
      },
      {
        "name": "Terraform",
        "proficiency": "intermediate",
        "endorsements": 12,
        "duration_months": 19
      },
      {
        "name": "SAP",
        "proficiency": "intermediate",
        "endorsements": 8,
        "duration_months": 9
      },
      {
        "name": "Illustrator",
        "proficiency": "beginner",
        "endorsements": 1,
        "duration_months": 11
      }
    ],
    "certifications": [],
    "languages": [
      {
        "language": "English",
        "proficiency": "professional"
      },
      {
        "language": "Hindi",
        "proficiency": "conversational"
      }
    ],
    "redrob_signals": {
      "profile_completeness_score": 31.0,
      "signup_date": "2023-03-07",
      "last_active_date": "2026-05-07",
      "open_to_work_flag": true,
      "profile_views_received_30d": 89,
      "applications_submitted_30d": 5,
      "recruiter_response_rate": 0.58,
      "avg_response_time_hours": 112.3,
      "skill_assessment_scores": {
        "YOLO": 60.2,
        "PEFT": 50.5,
        "Data Science": 35.1
      },
      "connection_count": 282,
      "endorsements_received": 24,
      "notice_period_days": 90,
      "expected_salary_range_inr_lpa": {
        "min": 17.9,
        "max": 20.2
      },
      "preferred_work_mode": "hybrid",
      "willing_to_relocate": false,
      "github_activity_score": 38.6,
      "search_appearance_30d": 136,
      "saved_by_recruiters_30d": 6,
      "interview_completion_rate": 0.61,
      "offer_acceptance_rate": 0.65,
      "verified_email": false,
      "verified_phone": true,
      "linkedin_connected": true
    }
  },
  {
    "candidate_id": "CAND_0000028",
    "profile": {
      "anonymized_name": "Rohan Krishnan",
      "headline": "Operations Manager | Driving business outcomes",
      "summary": "Professional with 1.1+ years of experience. My professional background is in marketing manager \u2014 I've built and led teams, owned KPIs, and driven business outcomes in this domain. Lately I've been curious about how AI tools could augment my work \u2014 I've experimented with ChatGPT and a few other tools for productivity and content creation, and I think the space is exciting. Open to roles where I can apply my domain expertise alongside emerging AI capabilities.",
      "location": "Dubai",
      "country": "UAE",
      "years_of_experience": 1.1,
      "current_title": "Operations Manager",
      "current_company": "Wipro",
      "current_company_size": "10001+",
      "current_industry": "IT Services"
    },
    "career_history": [
      {
        "company": "Wipro",
        "title": "Operations Manager",
        "start_date": "2025-05-02",
        "end_date": null,
        "duration_months": 13,
        "is_current": true,
        "industry": "IT Services",
        "company_size": "10001+",
        "description": "Brand design and creative direction at a consumer-products company. Owned brand identity (logo, visual system, typography), packaging design, and digital creative across web and social. Led the recent rebrand and managed a small external agency for production work. Comfortable across the Adobe suite, Figma, and the production side of brand and packaging design."
      }
    ],
    "education": [
      {
        "institution": "Symbiosis International",
        "degree": "M.Tech",
        "field_of_study": "Mathematics",
        "start_year": 2014,
        "end_year": 2017,
        "grade": "7.85 CGPA",
        "tier": "tier_3"
      }
    ],
    "skills": [
      {
        "name": "Snowflake",
        "proficiency": "beginner",
        "endorsements": 6,
        "duration_months": 3
      },
      {
        "name": "React",
        "proficiency": "beginner",
        "endorsements": 11,
        "duration_months": 7
      },
      {
        "name": "JavaScript",
        "proficiency": "beginner",
        "endorsements": 6,
        "duration_months": 10
      },
      {
        "name": "Tailwind",
        "proficiency": "beginner",
        "endorsements": 13,
        "duration_months": 15
      },
      {
        "name": "REST APIs",
        "proficiency": "intermediate",
        "endorsements": 6,
        "duration_months": 21
      },
      {
        "name": "Photoshop",
        "proficiency": "intermediate",
        "endorsements": 9,
        "duration_months": 30
      },
      {
        "name": "Data Pipelines",
        "proficiency": "intermediate",
        "endorsements": 4,
        "duration_months": 23
      },
      {
        "name": "Terraform",
        "proficiency": "intermediate",
        "endorsements": 9,
        "duration_months": 18
      },
      {
        "name": "CNN",
        "proficiency": "intermediate",
        "endorsements": 8,
        "duration_months": 29
      },
      {
        "name": "Content Writing",
        "proficiency": "beginner",
        "endorsements": 9,
        "duration_months": 18
      }
    ],
    "certifications": [
      {
        "name": "AWS Certified Cloud Practitioner",
        "issuer": "AWS",
        "year": 2020
      }
    ],
    "languages": [
      {
        "language": "English",
        "proficiency": "professional"
      },
      {
        "language": "Hindi",
        "proficiency": "conversational"
      }
    ],
    "redrob_signals": {
      "profile_completeness_score": 51.2,
      "signup_date": "2025-09-23",
      "last_active_date": "2026-03-31",
      "open_to_work_flag": false,
      "profile_views_received_30d": 6,
      "applications_submitted_30d": 7,
      "recruiter_response_rate": 0.14,
      "avg_response_time_hours": 13.2,
      "skill_assessment_scores": {},
      "connection_count": 524,
      "endorsements_received": 1,
      "notice_period_days": 60,
      "expected_salary_range_inr_lpa": {
        "min": 12.9,
        "max": 17.2
      },
      "preferred_work_mode": "remote",
      "willing_to_relocate": true,
      "github_activity_score": -1,
      "search_appearance_30d": 68,
      "saved_by_recruiters_30d": 4,
      "interview_completion_rate": 0.86,
      "offer_acceptance_rate": 0.49,
      "verified_email": true,
      "verified_phone": true,
      "linkedin_connected": false
    }
  },
  {
    "candidate_id": "CAND_0000029",
    "profile": {
      "anonymized_name": "Priya Sethi",
      "headline": "Business Analyst | Driving business outcomes",
      "summary": "Professional with 7.2+ years of experience. I'm a marketing manager with substantial experience in cross-functional collaboration, stakeholder management, and execution. Lately I've been curious about how AI tools could augment my work \u2014 I've experimented with ChatGPT and a few other tools for productivity and content creation, and I think the space is exciting. Open to roles where I can apply my domain expertise alongside emerging AI capabilities.",
      "location": "Noida, Uttar Pradesh",
      "country": "India",
      "years_of_experience": 7.2,
      "current_title": "Business Analyst",
      "current_company": "Wipro",
      "current_company_size": "10001+",
      "current_industry": "IT Services"
    },
    "career_history": [
      {
        "company": "Wipro",
        "title": "Business Analyst",
        "start_date": "2025-02-01",
        "end_date": null,
        "duration_months": 16,
        "is_current": true,
        "industry": "IT Services",
        "company_size": "10001+",
        "description": "Marketing leadership role at a B2B SaaS company. Owned the demand-generation function \u2014 content marketing, paid acquisition, SEO, email nurture. Built and managed a team of 5 across content, performance marketing, and marketing operations. Worked closely with sales on lead-quality definitions and the SDR-handoff process. Recent focus has been on account-based marketing for our enterprise segment."
      },
      {
        "company": "Globex Inc",
        "title": "Mechanical Engineer",
        "start_date": "2023-12-09",
        "end_date": "2025-02-01",
        "duration_months": 14,
        "is_current": false,
        "industry": "Manufacturing",
        "company_size": "501-1000",
        "description": "Mechanical engineering design role at a hardware-product company. Led the design of two product subsystems through full lifecycle: concept, DFM/DFMA review, prototype, production tooling. Comfortable with CAD (SolidWorks, Creo), FEA (ANSYS), and the typical hardware-development cadence. Worked closely with manufacturing partners on production scale-up."
      },
      {
        "company": "TCS",
        "title": "Civil Engineer",
        "start_date": "2019-05-27",
        "end_date": "2023-12-02",
        "duration_months": 55,
        "is_current": false,
        "industry": "IT Services",
        "company_size": "10001+",
        "description": "Customer support team lead at a SaaS product. Managed a team of 8 support agents handling tier-1 and tier-2 tickets; owned the escalation process to engineering and the customer-feedback loop to product. Built out the support knowledge base and the agent training program. Strong on the people-management side and the process side; lighter on technical depth beyond product expertise."
      }
    ],
    "education": [
      {
        "institution": "Symbiosis International",
        "degree": "B.Tech",
        "field_of_study": "Artificial Intelligence",
        "start_year": 2007,
        "end_year": 2011,
        "grade": "6.59 CGPA",
        "tier": "tier_3"
      }
    ],
    "skills": [
      {
        "name": "Node.js",
        "proficiency": "beginner",
        "endorsements": 9,
        "duration_months": 18
      },
      {
        "name": "Scrum",
        "proficiency": "beginner",
        "endorsements": 2,
        "duration_months": 5
      },
      {
        "name": "Tailwind",
        "proficiency": "intermediate",
        "endorsements": 5,
        "duration_months": 21
      },
      {
        "name": "Hadoop",
        "proficiency": "beginner",
        "endorsements": 10,
        "duration_months": 4
      },
      {
        "name": "Spring Boot",
        "proficiency": "intermediate",
        "endorsements": 1,
        "duration_months": 10
      },
      {
        "name": "CI/CD",
        "proficiency": "beginner",
        "endorsements": 4,
        "duration_months": 18
      },
      {
        "name": "gRPC",
        "proficiency": "beginner",
        "endorsements": 15,
        "duration_months": 17
      },
      {
        "name": "Terraform",
        "proficiency": "beginner",
        "endorsements": 9,
        "duration_months": 10
      }
    ],
    "certifications": [],
    "languages": [
      {
        "language": "English",
        "proficiency": "native"
      },
      {
        "language": "Hindi",
        "proficiency": "native"
      }
    ],
    "redrob_signals": {
      "profile_completeness_score": 40.5,
      "signup_date": "2025-06-17",
      "last_active_date": "2025-09-29",
      "open_to_work_flag": false,
      "profile_views_received_30d": 51,
      "applications_submitted_30d": 8,
      "recruiter_response_rate": 0.12,
      "avg_response_time_hours": 48.4,
      "skill_assessment_scores": {},
      "connection_count": 297,
      "endorsements_received": 4,
      "notice_period_days": 60,
      "expected_salary_range_inr_lpa": {
        "min": 17.5,
        "max": 19.1
      },
      "preferred_work_mode": "onsite",
      "willing_to_relocate": false,
      "github_activity_score": 42.5,
      "search_appearance_30d": 150,
      "saved_by_recruiters_30d": 8,
      "interview_completion_rate": 0.67,
      "offer_acceptance_rate": -1,
      "verified_email": true,
      "verified_phone": false,
      "linkedin_connected": true
    }
  },
  {
    "candidate_id": "CAND_0000030",
    "profile": {
      "anonymized_name": "Ritu Pillai",
      "headline": "Marketing Manager | Driving business outcomes",
      "summary": "Professional with 10.0+ years of experience. I've spent my career in marketing manager roles, mostly focused on driving outcomes through process, people, and customer relationships. Lately I've been curious about how AI tools could augment my work \u2014 I've experimented with ChatGPT and a few other tools for productivity and content creation, and I think the space is exciting. Open to roles where I can apply my domain expertise alongside emerging AI capabilities.",
      "location": "Kochi, Kerala",
      "country": "India",
      "years_of_experience": 10.0,
      "current_title": "Marketing Manager",
      "current_company": "Dunder Mifflin",
      "current_company_size": "201-500",
      "current_industry": "Paper Products"
    },
    "career_history": [
      {
        "company": "Dunder Mifflin",
        "title": "Marketing Manager",
        "start_date": "2022-03-19",
        "end_date": null,
        "duration_months": 51,
        "is_current": true,
        "industry": "Paper Products",
        "company_size": "201-500",
        "description": "Senior accounting role at a mid-sized company \u2014 month-end close, financial reporting, statutory compliance (GAAP / Ind-AS), and tax filings. Owned the GL, fixed-asset register, and the audit-readiness function. Managed a team of 3 staff accountants. Built strong process discipline around the close cycle, reducing close time from 12 days to 7 over the last two years."
      },
      {
        "company": "Hooli",
        "title": "Sales Executive",
        "start_date": "2018-07-08",
        "end_date": "2022-03-19",
        "duration_months": 45,
        "is_current": false,
        "industry": "Software",
        "company_size": "1001-5000",
        "description": "Operations management role at a logistics company. Owned daily fulfillment operations across 3 warehouses, managing a team of 80 across receiving, picking, packing, and outbound. Built and tracked the operational KPIs (on-time fulfillment, accuracy, cost per order) and led the continuous improvement initiatives that drove a 22% productivity gain over 18 months."
      },
      {
        "company": "Hooli",
        "title": "Content Writer",
        "start_date": "2016-08-17",
        "end_date": "2018-06-08",
        "duration_months": 22,
        "is_current": false,
        "industry": "Software",
        "company_size": "1001-5000",
        "description": "Content writing and SEO strategy for a tech-focused publication. Wrote longform articles on developer tools, cloud platforms, and AI/ML topics \u2014 including some that ranked on the first page of search for high-competition keywords. Managed a freelance writer pool and the editorial calendar. Recent work has been on AI-assisted content production, using LLM tools for research, drafting, and editing while maintaining editorial quality."
      }
    ],
    "education": [
      {
        "institution": "Generic State University",
        "degree": "B.E.",
        "field_of_study": "Computer Engineering",
        "start_year": 2010,
        "end_year": 2014,
        "grade": "81%",
        "tier": "tier_4"
      },
      {
        "institution": "Generic State University",
        "degree": "Ph.D",
        "field_of_study": "Artificial Intelligence",
        "start_year": 2017,
        "end_year": 2020,
        "grade": "7.98 CGPA",
        "tier": "tier_4"
      }
    ],
    "skills": [
      {
        "name": "gRPC",
        "proficiency": "intermediate",
        "endorsements": 3,
        "duration_months": 36
      },
      {
        "name": "Apache Beam",
        "proficiency": "beginner",
        "endorsements": 5,
        "duration_months": 6
      },
      {
        "name": "GraphQL",
        "proficiency": "intermediate",
        "endorsements": 2,
        "duration_months": 22
      },
      {
        "name": "Java",
        "proficiency": "intermediate",
        "endorsements": 14,
        "duration_months": 11
      },
      {
        "name": "Spring Boot",
        "proficiency": "intermediate",
        "endorsements": 2,
        "duration_months": 22
      },
      {
        "name": "Microservices",
        "proficiency": "beginner",
        "endorsements": 11,
        "duration_months": 5
      },
      {
        "name": "Six Sigma",
        "proficiency": "beginner",
        "endorsements": 8,
        "duration_months": 8
      },
      {
        "name": "Accounting",
        "proficiency": "intermediate",
        "endorsements": 3,
        "duration_months": 30
      },
      {
        "name": "HTML",
        "proficiency": "intermediate",
        "endorsements": 12,
        "duration_months": 9
      }
    ],
    "certifications": [],
    "languages": [
      {
        "language": "English",
        "proficiency": "professional"
      },
      {
        "language": "Hindi",
        "proficiency": "conversational"
      }
    ],
    "redrob_signals": {
      "profile_completeness_score": 59.7,
      "signup_date": "2025-09-25",
      "last_active_date": "2025-10-27",
      "open_to_work_flag": false,
      "profile_views_received_30d": 58,
      "applications_submitted_30d": 0,
      "recruiter_response_rate": 0.66,
      "avg_response_time_hours": 131.1,
      "skill_assessment_scores": {},
      "connection_count": 552,
      "endorsements_received": 45,
      "notice_period_days": 60,
      "expected_salary_range_inr_lpa": {
        "min": 14.7,
        "max": 14.2
      },
      "preferred_work_mode": "flexible",
      "willing_to_relocate": false,
      "github_activity_score": 21.7,
      "search_appearance_30d": 54,
      "saved_by_recruiters_30d": 1,
      "interview_completion_rate": 0.73,
      "offer_acceptance_rate": -1,
      "verified_email": true,
      "verified_phone": true,
      "linkedin_connected": false
    }
  },
  {
    "candidate_id": "CAND_0000031",
    "profile": {
      "anonymized_name": "Ela Singh",
      "headline": "Recommendation Systems Engineer | Search, Ranking & Retrieval",
      "summary": "Machine learning engineer with 6.0 years of experience building ML-powered features in production. Strong background in NLP, recommendation systems, and applied AI; comfortable across the ML stack from feature engineering through deployment. Recently, I led the team that migrated our keyword-search-based product to embedding-based retrieval. I've learned that most retrieval problems are actually evaluation problems in disguise. My academic background is in CS/ML but my main learning has come from shipping real systems and seeing what holds up under production load. Open to senior IC roles in applied ML or AI engineering, ideally at product companies where I'd own a meaningful piece of the ML stack.",
      "location": "Hyderabad, Telangana",
      "country": "India",
      "years_of_experience": 6.0,
      "current_title": "Recommendation Systems Engineer",
      "current_company": "Swiggy",
      "current_company_size": "5001-10000",
      "current_industry": "Food Delivery"
    },
    "career_history": [
      {
        "company": "Swiggy",
        "title": "Recommendation Systems Engineer",
        "start_date": "2025-04-02",
        "end_date": null,
        "duration_months": 14,
        "is_current": true,
        "industry": "Food Delivery",
        "company_size": "5001-10000",
        "description": "Trained and shipped multiple ranking models for our product's discovery feed using XGBoost and LightGBM. Designed features across three families: content metadata, user behavior signals, and item engagement history. Owned the offline-online correlation analysis that determined which offline metrics actually predicted A/B test outcomes. Worked closely with PMs to define the optimization target (click-through vs. dwell time vs. downstream conversion) \u2014 that work was as important as the modeling itself."
      },
      {
        "company": "Mad Street Den",
        "title": "Search Engineer",
        "start_date": "2023-10-10",
        "end_date": "2025-02-01",
        "duration_months": 16,
        "is_current": false,
        "industry": "AI/ML",
        "company_size": "201-500",
        "description": "Trained and shipped multiple ranking models for our product's discovery feed using XGBoost and LightGBM. Designed features across three families: content metadata, user behavior signals, and item engagement history. Owned the offline-online correlation analysis that determined which offline metrics actually predicted A/B test outcomes. Worked closely with PMs to define the optimization target (click-through vs. dwell time vs. downstream conversion) \u2014 that work was as important as the modeling itself."
      },
      {
        "company": "Uber",
        "title": "NLP Engineer",
        "start_date": "2021-07-22",
        "end_date": "2023-10-10",
        "duration_months": 27,
        "is_current": false,
        "industry": "Transportation",
        "company_size": "10001+",
        "description": "Trained and shipped multiple ranking models for our product's discovery feed using XGBoost and LightGBM. Designed features across three families: content metadata, user behavior signals, and item engagement history. Owned the offline-online correlation analysis that determined which offline metrics actually predicted A/B test outcomes. Worked closely with PMs to define the optimization target (click-through vs. dwell time vs. downstream conversion) \u2014 that work was as important as the modeling itself."
      },
      {
        "company": "Zomato",
        "title": "Applied ML Engineer",
        "start_date": "2020-06-27",
        "end_date": "2021-07-22",
        "duration_months": 13,
        "is_current": false,
        "industry": "Food Delivery",
        "company_size": "5001-10000",
        "description": "Owned the ranking layer for an e-commerce search product, evolving it from a hand-tuned scoring function to a learning-to-rank model over 9 months. Designed the relevance labeling pipeline (mix of click-through data and explicit human judgments), the feature pipeline, and the training/eval workflow. Most of the work was infrastructure and data quality \u2014 the modeling part was almost the easy bit. Final model improved revenue-per-search by 12%."
      }
    ],
    "education": [
      {
        "institution": "SRM University",
        "degree": "M.Tech",
        "field_of_study": "Computer Engineering",
        "start_year": 2002,
        "end_year": 2006,
        "grade": "9.16 CGPA",
        "tier": "tier_2"
      }
    ],
    "skills": [
      {
        "name": "Go",
        "proficiency": "intermediate",
        "endorsements": 7,
        "duration_months": 19
      },
      {
        "name": "MLflow",
        "proficiency": "advanced",
        "endorsements": 59,
        "duration_months": 21
      },
      {
        "name": "FAISS",
        "proficiency": "advanced",
        "endorsements": 19,
        "duration_months": 35
      },
      {
        "name": "Pinecone",
        "proficiency": "expert",
        "endorsements": 34,
        "duration_months": 88
      },
      {
        "name": "Angular",
        "proficiency": "beginner",
        "endorsements": 4,
        "duration_months": 18
      },
      {
        "name": "Image Classification",
        "proficiency": "advanced",
        "endorsements": 56,
        "duration_months": 28
      },
      {
        "name": "Machine Learning",
        "proficiency": "advanced",
        "endorsements": 43,
        "duration_months": 23
      },
      {
        "name": "Speech Recognition",
        "proficiency": "intermediate",
        "endorsements": 14,
        "duration_months": 24
      },
      {
        "name": "BentoML",
        "proficiency": "intermediate",
        "endorsements": 6,
        "duration_months": 14
      },
      {
        "name": "MLOps",
        "proficiency": "intermediate",
        "endorsements": 15,
        "duration_months": 36
      },
      {
        "name": "Embeddings",
        "proficiency": "expert",
        "endorsements": 48,
        "duration_months": 60
      },
      {
        "name": "Information Retrieval",
        "proficiency": "expert",
        "endorsements": 2,
        "duration_months": 84
      },
      {
        "name": "Hugging Face Transformers",
        "proficiency": "expert",
        "endorsements": 18,
        "duration_months": 36
      },
      {
        "name": "Feature Engineering",
        "proficiency": "advanced",
        "endorsements": 38,
        "duration_months": 26
      },
      {
        "name": "Sentence Transformers",
        "proficiency": "expert",
        "endorsements": 16,
        "duration_months": 69
      },
      {
        "name": "scikit-learn",
        "proficiency": "advanced",
        "endorsements": 41,
        "duration_months": 60
      },
      {
        "name": "Marketing",
        "proficiency": "intermediate",
        "endorsements": 11,
        "duration_months": 36
      }
    ],
    "certifications": [],
    "languages": [
      {
        "language": "English",
        "proficiency": "native"
      },
      {
        "language": "Hindi",
        "proficiency": "native"
      }
    ],
    "redrob_signals": {
      "profile_completeness_score": 83.4,
      "signup_date": "2026-01-28",
      "last_active_date": "2026-05-24",
      "open_to_work_flag": true,
      "profile_views_received_30d": 194,
      "applications_submitted_30d": 2,
      "recruiter_response_rate": 0.91,
      "avg_response_time_hours": 76.1,
      "skill_assessment_scores": {
        "MLflow": 75.1,
        "FAISS": 68.4,
        "Pinecone": 53.6,
        "Image Classification": 57.1
      },
      "connection_count": 832,
      "endorsements_received": 177,
      "notice_period_days": 60,
      "expected_salary_range_inr_lpa": {
        "min": 27.3,
        "max": 60.2
      },
      "preferred_work_mode": "flexible",
      "willing_to_relocate": true,
      "github_activity_score": 32.6,
      "search_appearance_30d": 778,
      "saved_by_recruiters_30d": 13,
      "interview_completion_rate": 0.6,
      "offer_acceptance_rate": 0.38,
      "verified_email": false,
      "verified_phone": true,
      "linkedin_connected": false
    }
  },
  {
    "candidate_id": "CAND_0000032",
    "profile": {
      "anonymized_name": "Pranav Agarwal",
      "headline": ".NET Developer | Full-stack development",
      "summary": "Software engineer with 8.1 years of experience across web, backend, and cloud systems. Strong fundamentals in software development and system design. Most of my work is conventional backend engineering \u2014 APIs, databases, queues, deployments. I've been keeping up with AI/ML at a self-learner level \u2014 taken some online courses, played with the OpenAI and Anthropic APIs, built a small RAG side project \u2014 but I haven't done it in a professional capacity yet. Open to roles where I can either deepen my software engineering work or, if the team is open to it, start contributing to ML-adjacent systems.",
      "location": "Gurgaon, Haryana",
      "country": "India",
      "years_of_experience": 8.1,
      "current_title": ".NET Developer",
      "current_company": "Cognizant",
      "current_company_size": "10001+",
      "current_industry": "IT Services"
    },
    "career_history": [
      {
        "company": "Cognizant",
        "title": ".NET Developer",
        "start_date": "2024-02-07",
        "end_date": null,
        "duration_months": 28,
        "is_current": true,
        "industry": "IT Services",
        "company_size": "10001+",
        "description": "Java backend development at a large enterprise \u2014 Spring Boot microservices, Kafka for inter-service messaging, Postgres + Redis for storage. Worked on the customer onboarding flow which involved orchestrating multiple downstream services. Solid on the Spring ecosystem, transaction handling, and the operational side of Java services. Looking to either go deeper on distributed systems or expand into modern application stacks."
      },
      {
        "company": "HCL",
        "title": "Cloud Engineer",
        "start_date": "2021-11-19",
        "end_date": "2024-02-07",
        "duration_months": 27,
        "is_current": false,
        "industry": "IT Services",
        "company_size": "10001+",
        "description": "Test automation and QA engineering for a fintech product. Built and maintained the end-to-end test suite using Selenium and pytest, plus the load-testing setup using Locust. Worked closely with developers on testability patterns and with product on acceptance criteria. Recent work has been on shifting test responsibility into the dev team \u2014 moving from QA-as-gate to QA-as-coach. Career has been entirely in QA/test engineering."
      },
      {
        "company": "Globex Inc",
        "title": "Mobile Developer",
        "start_date": "2018-07-24",
        "end_date": "2021-11-05",
        "duration_months": 40,
        "is_current": false,
        "industry": "Manufacturing",
        "company_size": "501-1000",
        "description": "Full-stack web application development at a SaaS company. Built React-based admin interfaces and the Node.js REST API backing them. Worked across the stack: frontend components, REST endpoint design, PostgreSQL schema, deployment via Docker/Kubernetes. Comfortable in most parts of a typical web stack though my comfort zone is the backend and database side. Recent learning has been on the testing and CI/CD discipline."
      }
    ],
    "education": [
      {
        "institution": "VIT Chennai",
        "degree": "M.Sc",
        "field_of_study": "Machine Learning",
        "start_year": 2017,
        "end_year": 2020,
        "grade": "8.37 CGPA",
        "tier": "tier_3"
      },
      {
        "institution": "Amity University",
        "degree": "Ph.D",
        "field_of_study": "Physics",
        "start_year": 2011,
        "end_year": 2015,
        "grade": "7.95 CGPA",
        "tier": "tier_3"
      }
    ],
    "skills": [
      {
        "name": "Speech Recognition",
        "proficiency": "advanced",
        "endorsements": 36,
        "duration_months": 19
      },
      {
        "name": "Project Management",
        "proficiency": "beginner",
        "endorsements": 6,
        "duration_months": 17
      },
      {
        "name": "REST APIs",
        "proficiency": "beginner",
        "endorsements": 13,
        "duration_months": 6
      },
      {
        "name": "CSS",
        "proficiency": "intermediate",
        "endorsements": 15,
        "duration_months": 27
      },
      {
        "name": "Embeddings",
        "proficiency": "advanced",
        "endorsements": 30,
        "duration_months": 30
      },
      {
        "name": "Hadoop",
        "proficiency": "beginner",
        "endorsements": 0,
        "duration_months": 8
      },
      {
        "name": "Spark",
        "proficiency": "intermediate",
        "endorsements": 14,
        "duration_months": 30
      },
      {
        "name": "Python",
        "proficiency": "intermediate",
        "endorsements": 2,
        "duration_months": 13
      },
      {
        "name": "Data Pipelines",
        "proficiency": "beginner",
        "endorsements": 6,
        "duration_months": 8
      },
      {
        "name": "OpenCV",
        "proficiency": "advanced",
        "endorsements": 45,
        "duration_months": 54
      }
    ],
    "certifications": [],
    "languages": [
      {
        "language": "English",
        "proficiency": "native"
      },
      {
        "language": "Hindi",
        "proficiency": "native"
      }
    ],
    "redrob_signals": {
      "profile_completeness_score": 35.4,
      "signup_date": "2023-12-20",
      "last_active_date": "2025-12-29",
      "open_to_work_flag": false,
      "profile_views_received_30d": 80,
      "applications_submitted_30d": 3,
      "recruiter_response_rate": 0.69,
      "avg_response_time_hours": 58.6,
      "skill_assessment_scores": {},
      "connection_count": 404,
      "endorsements_received": 22,
      "notice_period_days": 150,
      "expected_salary_range_inr_lpa": {
        "min": 18.3,
        "max": 15.7
      },
      "preferred_work_mode": "flexible",
      "willing_to_relocate": true,
      "github_activity_score": -1,
      "search_appearance_30d": 56,
      "saved_by_recruiters_30d": 4,
      "interview_completion_rate": 0.78,
      "offer_acceptance_rate": 0.25,
      "verified_email": true,
      "verified_phone": false,
      "linkedin_connected": false
    }
  },
  {
    "candidate_id": "CAND_0000033",
    "profile": {
      "anonymized_name": "Shreya Nair",
      "headline": "Graphic Designer | Helping teams scale",
      "summary": "Professional with 8.6+ years of experience. I'm a marketing manager with substantial experience in cross-functional collaboration, stakeholder management, and execution. Lately I've been curious about how AI tools could augment my work \u2014 I've experimented with ChatGPT and a few other tools for productivity and content creation, and I think the space is exciting. Open to roles where I can apply my domain expertise alongside emerging AI capabilities.",
      "location": "Pune, Maharashtra",
      "country": "India",
      "years_of_experience": 8.6,
      "current_title": "Graphic Designer",
      "current_company": "Wipro",
      "current_company_size": "10001+",
      "current_industry": "IT Services"
    },
    "career_history": [
      {
        "company": "Wipro",
        "title": "Graphic Designer",
        "start_date": "2023-11-09",
        "end_date": null,
        "duration_months": 31,
        "is_current": true,
        "industry": "IT Services",
        "company_size": "10001+",
        "description": "Business analyst at a consulting firm, working primarily with retail and CPG clients. Conducted business diagnostics, process re-engineering work, and digital transformation strategy projects. Strong on stakeholder management, structured problem-solving, and the typical consulting toolkit (slide-craft, Excel modeling, executive communication). Recent project work involved AI-strategy advisory but my own technical depth in AI is limited."
      },
      {
        "company": "Dunder Mifflin",
        "title": "Project Manager",
        "start_date": "2020-07-20",
        "end_date": "2023-11-02",
        "duration_months": 40,
        "is_current": false,
        "industry": "Paper Products",
        "company_size": "201-500",
        "description": "Senior accounting role at a mid-sized company \u2014 month-end close, financial reporting, statutory compliance (GAAP / Ind-AS), and tax filings. Owned the GL, fixed-asset register, and the audit-readiness function. Managed a team of 3 staff accountants. Built strong process discipline around the close cycle, reducing close time from 12 days to 7 over the last two years."
      },
      {
        "company": "Acme Corp",
        "title": "Content Writer",
        "start_date": "2018-01-02",
        "end_date": "2020-07-20",
        "duration_months": 31,
        "is_current": false,
        "industry": "Manufacturing",
        "company_size": "201-500",
        "description": "Enterprise sales of cloud software solutions into the mid-market segment. Carried a $1.8M ARR quota and consistently delivered against it across the last three years. Owned the full sales cycle: prospecting, discovery, technical evaluation (with SE support), commercial negotiation, and close. Strong on consultative selling for technical buyers; comfortable engaging with both engineering and finance stakeholders."
      }
    ],
    "education": [
      {
        "institution": "Tier-3 Engineering College",
        "degree": "M.S.",
        "field_of_study": "Computer Science",
        "start_year": 2014,
        "end_year": 2017,
        "grade": "9.32 CGPA",
        "tier": "tier_4"
      }
    ],
    "skills": [
      {
        "name": "Kubernetes",
        "proficiency": "beginner",
        "endorsements": 0,
        "duration_months": 9
      },
      {
        "name": "Data Pipelines",
        "proficiency": "beginner",
        "endorsements": 4,
        "duration_months": 8
      },
      {
        "name": "Snowflake",
        "proficiency": "intermediate",
        "endorsements": 11,
        "duration_months": 12
      },
      {
        "name": "CI/CD",
        "proficiency": "intermediate",
        "endorsements": 11,
        "duration_months": 29
      },
      {
        "name": "SEO",
        "proficiency": "intermediate",
        "endorsements": 9,
        "duration_months": 36
      },
      {
        "name": "ETL",
        "proficiency": "intermediate",
        "endorsements": 0,
        "duration_months": 20
      },
      {
        "name": "Airflow",
        "proficiency": "intermediate",
        "endorsements": 11,
        "duration_months": 16
      },
      {
        "name": "TypeScript",
        "proficiency": "intermediate",
        "endorsements": 13,
        "duration_months": 15
      },
      {
        "name": "Content Writing",
        "proficiency": "intermediate",
        "endorsements": 13,
        "duration_months": 11
      },
      {
        "name": "Spring Boot",
        "proficiency": "intermediate",
        "endorsements": 5,
        "duration_months": 26
      }
    ],
    "certifications": [
      {
        "name": "Six Sigma Green Belt",
        "issuer": "ASQ",
        "year": 2019
      }
    ],
    "languages": [
      {
        "language": "English",
        "proficiency": "professional"
      },
      {
        "language": "Hindi",
        "proficiency": "conversational"
      }
    ],
    "redrob_signals": {
      "profile_completeness_score": 74.0,
      "signup_date": "2026-03-13",
      "last_active_date": "2026-03-27",
      "open_to_work_flag": true,
      "profile_views_received_30d": 42,
      "applications_submitted_30d": 9,
      "recruiter_response_rate": 0.08,
      "avg_response_time_hours": 210.9,
      "skill_assessment_scores": {},
      "connection_count": 410,
      "endorsements_received": 29,
      "notice_period_days": 30,
      "expected_salary_range_inr_lpa": {
        "min": 8.3,
        "max": 13.0
      },
      "preferred_work_mode": "remote",
      "willing_to_relocate": false,
      "github_activity_score": 38.3,
      "search_appearance_30d": 98,
      "saved_by_recruiters_30d": 2,
      "interview_completion_rate": 0.37,
      "offer_acceptance_rate": -1,
      "verified_email": false,
      "verified_phone": true,
      "linkedin_connected": false
    }
  },
  {
    "candidate_id": "CAND_0000034",
    "profile": {
      "anonymized_name": "Yash Chatterjee",
      "headline": "Business Analyst | Driving business outcomes",
      "summary": "Professional with 14.5+ years of experience. I'm a marketing manager with substantial experience in cross-functional collaboration, stakeholder management, and execution. Lately I've been curious about how AI tools could augment my work \u2014 I've experimented with ChatGPT and a few other tools for productivity and content creation, and I think the space is exciting. Open to roles where I can apply my domain expertise alongside emerging AI capabilities.",
      "location": "Ahmedabad, Gujarat",
      "country": "India",
      "years_of_experience": 14.5,
      "current_title": "Business Analyst",
      "current_company": "Wipro",
      "current_company_size": "10001+",
      "current_industry": "IT Services"
    },
    "career_history": [
      {
        "company": "Wipro",
        "title": "Business Analyst",
        "start_date": "2025-04-02",
        "end_date": null,
        "duration_months": 14,
        "is_current": true,
        "industry": "IT Services",
        "company_size": "10001+",
        "description": "Content writing and SEO strategy for a tech-focused publication. Wrote longform articles on developer tools, cloud platforms, and AI/ML topics \u2014 including some that ranked on the first page of search for high-competition keywords. Managed a freelance writer pool and the editorial calendar. Recent work has been on AI-assisted content production, using LLM tools for research, drafting, and editing while maintaining editorial quality."
      },
      {
        "company": "Hooli",
        "title": "Mechanical Engineer",
        "start_date": "2023-05-13",
        "end_date": "2025-02-01",
        "duration_months": 21,
        "is_current": false,
        "industry": "Software",
        "company_size": "1001-5000",
        "description": "Business analyst at a consulting firm, working primarily with retail and CPG clients. Conducted business diagnostics, process re-engineering work, and digital transformation strategy projects. Strong on stakeholder management, structured problem-solving, and the typical consulting toolkit (slide-craft, Excel modeling, executive communication). Recent project work involved AI-strategy advisory but my own technical depth in AI is limited."
      },
      {
        "company": "Infosys",
        "title": "Business Analyst",
        "start_date": "2021-02-22",
        "end_date": "2023-04-13",
        "duration_months": 26,
        "is_current": false,
        "industry": "IT Services",
        "company_size": "10001+",
        "description": "Business analyst at a consulting firm, working primarily with retail and CPG clients. Conducted business diagnostics, process re-engineering work, and digital transformation strategy projects. Strong on stakeholder management, structured problem-solving, and the typical consulting toolkit (slide-craft, Excel modeling, executive communication). Recent project work involved AI-strategy advisory but my own technical depth in AI is limited."
      },
      {
        "company": "TCS",
        "title": "Accountant",
        "start_date": "2017-11-10",
        "end_date": "2021-02-22",
        "duration_months": 40,
        "is_current": false,
        "industry": "IT Services",
        "company_size": "10001+",
        "description": "Business analyst at a consulting firm, working primarily with retail and CPG clients. Conducted business diagnostics, process re-engineering work, and digital transformation strategy projects. Strong on stakeholder management, structured problem-solving, and the typical consulting toolkit (slide-craft, Excel modeling, executive communication). Recent project work involved AI-strategy advisory but my own technical depth in AI is limited."
      },
      {
        "company": "Hooli",
        "title": "Accountant",
        "start_date": "2016-01-20",
        "end_date": "2017-11-10",
        "duration_months": 22,
        "is_current": false,
        "industry": "Software",
        "company_size": "1001-5000",
        "description": "Marketing leadership role at a B2B SaaS company. Owned the demand-generation function \u2014 content marketing, paid acquisition, SEO, email nurture. Built and managed a team of 5 across content, performance marketing, and marketing operations. Worked closely with sales on lead-quality definitions and the SDR-handoff process. Recent focus has been on account-based marketing for our enterprise segment."
      },
      {
        "company": "Pied Piper",
        "title": "Content Writer",
        "start_date": "2012-12-06",
        "end_date": "2016-01-20",
        "duration_months": 38,
        "is_current": false,
        "industry": "Software",
        "company_size": "11-50",
        "description": "Marketing leadership role at a B2B SaaS company. Owned the demand-generation function \u2014 content marketing, paid acquisition, SEO, email nurture. Built and managed a team of 5 across content, performance marketing, and marketing operations. Worked closely with sales on lead-quality definitions and the SDR-handoff process. Recent focus has been on account-based marketing for our enterprise segment."
      },
      {
        "company": "Stark Industries",
        "title": "Content Writer",
        "start_date": "2012-01-11",
        "end_date": "2012-10-07",
        "duration_months": 9,
        "is_current": false,
        "industry": "Manufacturing",
        "company_size": "1001-5000",
        "description": "Mechanical engineering design role at a hardware-product company. Led the design of two product subsystems through full lifecycle: concept, DFM/DFMA review, prototype, production tooling. Comfortable with CAD (SolidWorks, Creo), FEA (ANSYS), and the typical hardware-development cadence. Worked closely with manufacturing partners on production scale-up."
      }
    ],
    "education": [
      {
        "institution": "Tier-3 Engineering College",
        "degree": "B.E.",
        "field_of_study": "Computer Engineering",
        "start_year": 2005,
        "end_year": 2010,
        "grade": "8.97 CGPA",
        "tier": "tier_4"
      }
    ],
    "skills": [
      {
        "name": "GraphQL",
        "proficiency": "beginner",
        "endorsements": 2,
        "duration_months": 3
      },
      {
        "name": "Excel",
        "proficiency": "intermediate",
        "endorsements": 15,
        "duration_months": 19
      },
      {
        "name": "Node.js",
        "proficiency": "beginner",
        "endorsements": 6,
        "duration_months": 6
      },
      {
        "name": "Terraform",
        "proficiency": "intermediate",
        "endorsements": 6,
        "duration_months": 13
      },
      {
        "name": "Salesforce CRM",
        "proficiency": "intermediate",
        "endorsements": 6,
        "duration_months": 27
      },
      {
        "name": "Flask",
        "proficiency": "intermediate",
        "endorsements": 8,
        "duration_months": 28
      },
      {
        "name": "React",
        "proficiency": "beginner",
        "endorsements": 14,
        "duration_months": 2
      },
      {
        "name": "Azure",
        "proficiency": "beginner",
        "endorsements": 1,
        "duration_months": 5
      },
      {
        "name": "Redux",
        "proficiency": "beginner",
        "endorsements": 15,
        "duration_months": 3
      },
      {
        "name": "Next.js",
        "proficiency": "intermediate",
        "endorsements": 7,
        "duration_months": 21
      }
    ],
    "certifications": [],
    "languages": [
      {
        "language": "English",
        "proficiency": "native"
      },
      {
        "language": "Hindi",
        "proficiency": "native"
      }
    ],
    "redrob_signals": {
      "profile_completeness_score": 41.2,
      "signup_date": "2024-01-15",
      "last_active_date": "2026-01-03",
      "open_to_work_flag": false,
      "profile_views_received_30d": 25,
      "applications_submitted_30d": 7,
      "recruiter_response_rate": 0.15,
      "avg_response_time_hours": 253.5,
      "skill_assessment_scores": {},
      "connection_count": 226,
      "endorsements_received": 27,
      "notice_period_days": 90,
      "expected_salary_range_inr_lpa": {
        "min": 14.4,
        "max": 28.0
      },
      "preferred_work_mode": "hybrid",
      "willing_to_relocate": false,
      "github_activity_score": -1,
      "search_appearance_30d": 143,
      "saved_by_recruiters_30d": 2,
      "interview_completion_rate": 0.41,
      "offer_acceptance_rate": -1,
      "verified_email": true,
      "verified_phone": true,
      "linkedin_connected": false
    }
  },
  {
    "candidate_id": "CAND_0000035",
    "profile": {
      "anonymized_name": "Vikram Verma",
      "headline": "Full Stack Developer | Backend systems & APIs",
      "summary": "Software engineer with 4.3 years of experience across web, backend, and cloud systems. Strong fundamentals in software development and system design. My background is full-stack, but my comfort zone is the backend and the database. I've been keeping up with AI/ML at a self-learner level \u2014 taken some online courses, played with the OpenAI and Anthropic APIs, built a small RAG side project \u2014 but I haven't done it in a professional capacity yet. Open to roles where I can either deepen my software engineering work or, if the team is open to it, start contributing to ML-adjacent systems.",
      "location": "Hyderabad, Telangana",
      "country": "India",
      "years_of_experience": 4.3,
      "current_title": "Full Stack Developer",
      "current_company": "Globex Inc",
      "current_company_size": "501-1000",
      "current_industry": "Manufacturing"
    },
    "career_history": [
      {
        "company": "Globex Inc",
        "title": "Full Stack Developer",
        "start_date": "2023-09-10",
        "end_date": null,
        "duration_months": 33,
        "is_current": true,
        "industry": "Manufacturing",
        "company_size": "501-1000",
        "description": "Full-stack web application development at a SaaS company. Built React-based admin interfaces and the Node.js REST API backing them. Worked across the stack: frontend components, REST endpoint design, PostgreSQL schema, deployment via Docker/Kubernetes. Comfortable in most parts of a typical web stack though my comfort zone is the backend and database side. Recent learning has been on the testing and CI/CD discipline."
      },
      {
        "company": "Wipro",
        "title": "Software Engineer",
        "start_date": "2022-03-19",
        "end_date": "2023-09-10",
        "duration_months": 18,
        "is_current": false,
        "industry": "IT Services",
        "company_size": "10001+",
        "description": "Frontend engineering at a media company. React, TypeScript, and the typical surrounding tooling (Webpack, Jest, Cypress). Built the company's design system from scratch and led the migration from a legacy AngularJS app. Strong on the frontend craft \u2014 accessibility, performance, animations \u2014 but limited backend exposure."
      }
    ],
    "education": [
      {
        "institution": "Generic State University",
        "degree": "B.E.",
        "field_of_study": "Civil Engineering",
        "start_year": 2010,
        "end_year": 2013,
        "grade": "90%",
        "tier": "tier_4"
      },
      {
        "institution": "Bharati Vidyapeeth",
        "degree": "M.S.",
        "field_of_study": "Data Science",
        "start_year": 2010,
        "end_year": 2015,
        "grade": "7.08 CGPA",
        "tier": "tier_3"
      }
    ],
    "skills": [
      {
        "name": "Snowflake",
        "proficiency": "intermediate",
        "endorsements": 11,
        "duration_months": 27
      },
      {
        "name": "BigQuery",
        "proficiency": "beginner",
        "endorsements": 15,
        "duration_months": 6
      },
      {
        "name": "Recommendation Systems",
        "proficiency": "intermediate",
        "endorsements": 2,
        "duration_months": 34
      },
      {
        "name": "Data Pipelines",
        "proficiency": "beginner",
        "endorsements": 14,
        "duration_months": 18
      },
      {
        "name": "Docker",
        "proficiency": "beginner",
        "endorsements": 3,
        "duration_months": 11
      },
      {
        "name": "MongoDB",
        "proficiency": "intermediate",
        "endorsements": 11,
        "duration_months": 16
      },
      {
        "name": "PostgreSQL",
        "proficiency": "intermediate",
        "endorsements": 14,
        "duration_months": 13
      },
      {
        "name": "Sales",
        "proficiency": "intermediate",
        "endorsements": 0,
        "duration_months": 19
      },
      {
        "name": "Kafka",
        "proficiency": "intermediate",
        "endorsements": 14,
        "duration_months": 29
      },
      {
        "name": "Speech Recognition",
        "proficiency": "intermediate",
        "endorsements": 5,
        "duration_months": 33
      },
      {
        "name": "BentoML",
        "proficiency": "advanced",
        "endorsements": 40,
        "duration_months": 59
      },
      {
        "name": "Go",
        "proficiency": "beginner",
        "endorsements": 1,
        "duration_months": 11
      },
      {
        "name": "Next.js",
        "proficiency": "intermediate",
        "endorsements": 15,
        "duration_months": 22
      },
      {
        "name": "dbt",
        "proficiency": "beginner",
        "endorsements": 2,
        "duration_months": 9
      }
    ],
    "certifications": [
      {
        "name": "AWS Certified Cloud Practitioner",
        "issuer": "AWS",
        "year": 2022
      },
      {
        "name": "Six Sigma Green Belt",
        "issuer": "ASQ",
        "year": 2020
      }
    ],
    "languages": [
      {
        "language": "English",
        "proficiency": "professional"
      },
      {
        "language": "Hindi",
        "proficiency": "conversational"
      }
    ],
    "redrob_signals": {
      "profile_completeness_score": 56.2,
      "signup_date": "2024-08-13",
      "last_active_date": "2026-02-06",
      "open_to_work_flag": false,
      "profile_views_received_30d": 7,
      "applications_submitted_30d": 4,
      "recruiter_response_rate": 0.34,
      "avg_response_time_hours": 178.0,
      "skill_assessment_scores": {},
      "connection_count": 398,
      "endorsements_received": 45,
      "notice_period_days": 60,
      "expected_salary_range_inr_lpa": {
        "min": 11.2,
        "max": 22.8
      },
      "preferred_work_mode": "onsite",
      "willing_to_relocate": false,
      "github_activity_score": -1,
      "search_appearance_30d": 173,
      "saved_by_recruiters_30d": 1,
      "interview_completion_rate": 0.5,
      "offer_acceptance_rate": -1,
      "verified_email": true,
      "verified_phone": true,
      "linkedin_connected": false
    }
  },
  {
    "candidate_id": "CAND_0000036",
    "profile": {
      "anonymized_name": "Ananya Bose",
      "headline": "Project Manager | 11.3+ yrs experience",
      "summary": "Professional with 11.3+ years of experience. I'm a marketing manager with substantial experience in cross-functional collaboration, stakeholder management, and execution. Lately I've been curious about how AI tools could augment my work \u2014 I've experimented with ChatGPT and a few other tools for productivity and content creation, and I think the space is exciting. Open to roles where I can apply my domain expertise alongside emerging AI capabilities.",
      "location": "Trivandrum, Kerala",
      "country": "India",
      "years_of_experience": 11.3,
      "current_title": "Project Manager",
      "current_company": "Initech",
      "current_company_size": "51-200",
      "current_industry": "Software"
    },
    "career_history": [
      {
        "company": "Initech",
        "title": "Project Manager",
        "start_date": "2025-02-01",
        "end_date": null,
        "duration_months": 16,
        "is_current": true,
        "industry": "Software",
        "company_size": "51-200",
        "description": "Brand design and creative direction at a consumer-products company. Owned brand identity (logo, visual system, typography), packaging design, and digital creative across web and social. Led the recent rebrand and managed a small external agency for production work. Comfortable across the Adobe suite, Figma, and the production side of brand and packaging design."
      },
      {
        "company": "Hooli",
        "title": "Content Writer",
        "start_date": "2023-08-11",
        "end_date": "2025-02-01",
        "duration_months": 18,
        "is_current": false,
        "industry": "Software",
        "company_size": "1001-5000",
        "description": "Marketing leadership role at a B2B SaaS company. Owned the demand-generation function \u2014 content marketing, paid acquisition, SEO, email nurture. Built and managed a team of 5 across content, performance marketing, and marketing operations. Worked closely with sales on lead-quality definitions and the SDR-handoff process. Recent focus has been on account-based marketing for our enterprise segment."
      },
      {
        "company": "Dunder Mifflin",
        "title": "HR Manager",
        "start_date": "2019-11-30",
        "end_date": "2023-08-11",
        "duration_months": 45,
        "is_current": false,
        "industry": "Paper Products",
        "company_size": "201-500",
        "description": "Brand design and creative direction at a consumer-products company. Owned brand identity (logo, visual system, typography), packaging design, and digital creative across web and social. Led the recent rebrand and managed a small external agency for production work. Comfortable across the Adobe suite, Figma, and the production side of brand and packaging design."
      },
      {
        "company": "TCS",
        "title": "Civil Engineer",
        "start_date": "2017-12-10",
        "end_date": "2019-11-30",
        "duration_months": 24,
        "is_current": false,
        "industry": "IT Services",
        "company_size": "10001+",
        "description": "Senior accounting role at a mid-sized company \u2014 month-end close, financial reporting, statutory compliance (GAAP / Ind-AS), and tax filings. Owned the GL, fixed-asset register, and the audit-readiness function. Managed a team of 3 staff accountants. Built strong process discipline around the close cycle, reducing close time from 12 days to 7 over the last two years."
      },
      {
        "company": "Wayne Enterprises",
        "title": "Marketing Manager",
        "start_date": "2015-05-18",
        "end_date": "2017-12-03",
        "duration_months": 31,
        "is_current": false,
        "industry": "Conglomerate",
        "company_size": "10001+",
        "description": "Mechanical engineering design role at a hardware-product company. Led the design of two product subsystems through full lifecycle: concept, DFM/DFMA review, prototype, production tooling. Comfortable with CAD (SolidWorks, Creo), FEA (ANSYS), and the typical hardware-development cadence. Worked closely with manufacturing partners on production scale-up."
      }
    ],
    "education": [
      {
        "institution": "KIIT University",
        "degree": "M.S.",
        "field_of_study": "Commerce",
        "start_year": 2002,
        "end_year": 2006,
        "grade": "8.89 CGPA",
        "tier": "tier_3"
      }
    ],
    "skills": [
      {
        "name": "Figma",
        "proficiency": "beginner",
        "endorsements": 6,
        "duration_months": 13
      },
      {
        "name": "MongoDB",
        "proficiency": "beginner",
        "endorsements": 4,
        "duration_months": 8
      },
      {
        "name": "PowerPoint",
        "proficiency": "beginner",
        "endorsements": 5,
        "duration_months": 3
      },
      {
        "name": "CSS",
        "proficiency": "beginner",
        "endorsements": 5,
        "duration_months": 14
      },
      {
        "name": "Excel",
        "proficiency": "intermediate",
        "endorsements": 9,
        "duration_months": 28
      },
      {
        "name": "GraphQL",
        "proficiency": "beginner",
        "endorsements": 5,
        "duration_months": 18
      },
      {
        "name": "Object Detection",
        "proficiency": "advanced",
        "endorsements": 39,
        "duration_months": 37
      },
      {
        "name": "Vue.js",
        "proficiency": "beginner",
        "endorsements": 13,
        "duration_months": 4
      },
      {
        "name": "Sales",
        "proficiency": "beginner",
        "endorsements": 4,
        "duration_months": 9
      }
    ],
    "certifications": [],
    "languages": [
      {
        "language": "English",
        "proficiency": "native"
      },
      {
        "language": "Hindi",
        "proficiency": "conversational"
      }
    ],
    "redrob_signals": {
      "profile_completeness_score": 81.8,
      "signup_date": "2025-09-05",
      "last_active_date": "2025-12-12",
      "open_to_work_flag": true,
      "profile_views_received_30d": 70,
      "applications_submitted_30d": 4,
      "recruiter_response_rate": 0.4,
      "avg_response_time_hours": 236.6,
      "skill_assessment_scores": {},
      "connection_count": 324,
      "endorsements_received": 3,
      "notice_period_days": 60,
      "expected_salary_range_inr_lpa": {
        "min": 15.3,
        "max": 9.7
      },
      "preferred_work_mode": "remote",
      "willing_to_relocate": false,
      "github_activity_score": -1,
      "search_appearance_30d": 175,
      "saved_by_recruiters_30d": 0,
      "interview_completion_rate": 0.78,
      "offer_acceptance_rate": 0.46,
      "verified_email": true,
      "verified_phone": false,
      "linkedin_connected": false
    }
  },
  {
    "candidate_id": "CAND_0000037",
    "profile": {
      "anonymized_name": "Ved Sen",
      "headline": "Business Analyst | 14.3+ yrs experience",
      "summary": "Professional with 14.3+ years of experience. I've spent my career in marketing manager roles, mostly focused on driving outcomes through process, people, and customer relationships. Lately I've been curious about how AI tools could augment my work \u2014 I've experimented with ChatGPT and a few other tools for productivity and content creation, and I think the space is exciting. Open to roles where I can apply my domain expertise alongside emerging AI capabilities.",
      "location": "Dubai",
      "country": "UAE",
      "years_of_experience": 14.3,
      "current_title": "Business Analyst",
      "current_company": "Stark Industries",
      "current_company_size": "1001-5000",
      "current_industry": "Manufacturing"
    },
    "career_history": [
      {
        "company": "Stark Industries",
        "title": "Business Analyst",
        "start_date": "2022-03-19",
        "end_date": null,
        "duration_months": 51,
        "is_current": true,
        "industry": "Manufacturing",
        "company_size": "1001-5000",
        "description": "Business analyst at a consulting firm, working primarily with retail and CPG clients. Conducted business diagnostics, process re-engineering work, and digital transformation strategy projects. Strong on stakeholder management, structured problem-solving, and the typical consulting toolkit (slide-craft, Excel modeling, executive communication). Recent project work involved AI-strategy advisory but my own technical depth in AI is limited."
      },
      {
        "company": "Initech",
        "title": "Civil Engineer",
        "start_date": "2019-08-18",
        "end_date": "2022-03-05",
        "duration_months": 31,
        "is_current": false,
        "industry": "Software",
        "company_size": "51-200",
        "description": "Operations management role at a logistics company. Owned daily fulfillment operations across 3 warehouses, managing a team of 80 across receiving, picking, packing, and outbound. Built and tracked the operational KPIs (on-time fulfillment, accuracy, cost per order) and led the continuous improvement initiatives that drove a 22% productivity gain over 18 months."
      },
      {
        "company": "Stark Industries",
        "title": "Content Writer",
        "start_date": "2016-01-06",
        "end_date": "2019-08-18",
        "duration_months": 44,
        "is_current": false,
        "industry": "Manufacturing",
        "company_size": "1001-5000",
        "description": "Operations management role at a logistics company. Owned daily fulfillment operations across 3 warehouses, managing a team of 80 across receiving, picking, packing, and outbound. Built and tracked the operational KPIs (on-time fulfillment, accuracy, cost per order) and led the continuous improvement initiatives that drove a 22% productivity gain over 18 months."
      },
      {
        "company": "Hooli",
        "title": "Mechanical Engineer",
        "start_date": "2013-07-20",
        "end_date": "2016-01-06",
        "duration_months": 30,
        "is_current": false,
        "industry": "Software",
        "company_size": "1001-5000",
        "description": "Marketing leadership role at a B2B SaaS company. Owned the demand-generation function \u2014 content marketing, paid acquisition, SEO, email nurture. Built and managed a team of 5 across content, performance marketing, and marketing operations. Worked closely with sales on lead-quality definitions and the SDR-handoff process. Recent focus has been on account-based marketing for our enterprise segment."
      },
      {
        "company": "Acme Corp",
        "title": "HR Manager",
        "start_date": "2012-04-26",
        "end_date": "2013-06-20",
        "duration_months": 14,
        "is_current": false,
        "industry": "Manufacturing",
        "company_size": "201-500",
        "description": "Brand design and creative direction at a consumer-products company. Owned brand identity (logo, visual system, typography), packaging design, and digital creative across web and social. Led the recent rebrand and managed a small external agency for production work. Comfortable across the Adobe suite, Figma, and the production side of brand and packaging design."
      }
    ],
    "education": [
      {
        "institution": "Tier-3 Engineering College",
        "degree": "B.Tech",
        "field_of_study": "Machine Learning",
        "start_year": 2001,
        "end_year": 2005,
        "grade": "69%",
        "tier": "tier_4"
      },
      {
        "institution": "Lovely Professional University",
        "degree": "M.Tech",
        "field_of_study": "Statistics",
        "start_year": 2013,
        "end_year": 2016,
        "grade": "6.81 CGPA",
        "tier": "tier_3"
      }
    ],
    "skills": [
      {
        "name": "Databricks",
        "proficiency": "intermediate",
        "endorsements": 12,
        "duration_months": 32
      },
      {
        "name": "Docker",
        "proficiency": "intermediate",
        "endorsements": 6,
        "duration_months": 35
      },
      {
        "name": "Flask",
        "proficiency": "beginner",
        "endorsements": 5,
        "duration_months": 14
      },
      {
        "name": "AWS",
        "proficiency": "intermediate",
        "endorsements": 2,
        "duration_months": 33
      },
      {
        "name": "Terraform",
        "proficiency": "beginner",
        "endorsements": 3,
        "duration_months": 18
      },
      {
        "name": "Tally",
        "proficiency": "intermediate",
        "endorsements": 14,
        "duration_months": 24
      },
      {
        "name": "TTS",
        "proficiency": "intermediate",
        "endorsements": 3,
        "duration_months": 13
      },
      {
        "name": "Apache Beam",
        "proficiency": "intermediate",
        "endorsements": 15,
        "duration_months": 23
      }
    ],
    "certifications": [
      {
        "name": "AWS Certified Cloud Practitioner",
        "issuer": "AWS",
        "year": 2024
      }
    ],
    "languages": [
      {
        "language": "English",
        "proficiency": "native"
      },
      {
        "language": "Hindi",
        "proficiency": "professional"
      }
    ],
    "redrob_signals": {
      "profile_completeness_score": 25.6,
      "signup_date": "2025-04-11",
      "last_active_date": "2025-12-11",
      "open_to_work_flag": false,
      "profile_views_received_30d": 80,
      "applications_submitted_30d": 6,
      "recruiter_response_rate": 0.78,
      "avg_response_time_hours": 107.1,
      "skill_assessment_scores": {},
      "connection_count": 503,
      "endorsements_received": 13,
      "notice_period_days": 30,
      "expected_salary_range_inr_lpa": {
        "min": 8.8,
        "max": 14.2
      },
      "preferred_work_mode": "onsite",
      "willing_to_relocate": false,
      "github_activity_score": -1,
      "search_appearance_30d": 178,
      "saved_by_recruiters_30d": 1,
      "interview_completion_rate": 0.82,
      "offer_acceptance_rate": 0.19,
      "verified_email": true,
      "verified_phone": true,
      "linkedin_connected": true
    }
  },
  {
    "candidate_id": "CAND_0000038",
    "profile": {
      "anonymized_name": "Myra Trivedi",
      "headline": "Java Developer | Cloud & DevOps",
      "summary": "Software engineer with 6.7 years of experience across web, backend, and cloud systems. Strong fundamentals in software development and system design. My background is full-stack, but my comfort zone is the backend and the database. I've been keeping up with AI/ML at a self-learner level \u2014 taken some online courses, played with the OpenAI and Anthropic APIs, built a small RAG side project \u2014 but I haven't done it in a professional capacity yet. Open to roles where I can either deepen my software engineering work or, if the team is open to it, start contributing to ML-adjacent systems.",
      "location": "Coimbatore, Tamil Nadu",
      "country": "India",
      "years_of_experience": 6.7,
      "current_title": "Java Developer",
      "current_company": "Swiggy",
      "current_company_size": "5001-10000",
      "current_industry": "Food Delivery"
    },
    "career_history": [
      {
        "company": "Swiggy",
        "title": "Java Developer",
        "start_date": "2023-11-09",
        "end_date": null,
        "duration_months": 31,
        "is_current": true,
        "industry": "Food Delivery",
        "company_size": "5001-10000",
        "description": "Frontend engineering at a media company. React, TypeScript, and the typical surrounding tooling (Webpack, Jest, Cypress). Built the company's design system from scratch and led the migration from a legacy AngularJS app. Strong on the frontend craft \u2014 accessibility, performance, animations \u2014 but limited backend exposure."
      },
      {
        "company": "Globex Inc",
        "title": ".NET Developer",
        "start_date": "2022-09-15",
        "end_date": "2023-11-09",
        "duration_months": 14,
        "is_current": false,
        "industry": "Manufacturing",
        "company_size": "501-1000",
        "description": "Java backend development at a large enterprise \u2014 Spring Boot microservices, Kafka for inter-service messaging, Postgres + Redis for storage. Worked on the customer onboarding flow which involved orchestrating multiple downstream services. Solid on the Spring ecosystem, transaction handling, and the operational side of Java services. Looking to either go deeper on distributed systems or expand into modern application stacks."
      },
      {
        "company": "Hooli",
        "title": "DevOps Engineer",
        "start_date": "2019-11-30",
        "end_date": "2022-09-15",
        "duration_months": 34,
        "is_current": false,
        "industry": "Software",
        "company_size": "1001-5000",
        "description": "Frontend engineering at a media company. React, TypeScript, and the typical surrounding tooling (Webpack, Jest, Cypress). Built the company's design system from scratch and led the migration from a legacy AngularJS app. Strong on the frontend craft \u2014 accessibility, performance, animations \u2014 but limited backend exposure."
      }
    ],
    "education": [
      {
        "institution": "VIT Chennai",
        "degree": "B.Sc",
        "field_of_study": "Computer Engineering",
        "start_year": 2015,
        "end_year": 2020,
        "grade": "70%",
        "tier": "tier_3"
      }
    ],
    "skills": [
      {
        "name": "Kubeflow",
        "proficiency": "intermediate",
        "endorsements": 3,
        "duration_months": 26
      },
      {
        "name": "Django",
        "proficiency": "beginner",
        "endorsements": 2,
        "duration_months": 18
      },
      {
        "name": "Redux",
        "proficiency": "intermediate",
        "endorsements": 3,
        "duration_months": 13
      },
      {
        "name": "Weaviate",
        "proficiency": "advanced",
        "endorsements": 37,
        "duration_months": 27
      },
      {
        "name": "PowerPoint",
        "proficiency": "beginner",
        "endorsements": 15,
        "duration_months": 9
      },
      {
        "name": "Figma",
        "proficiency": "beginner",
        "endorsements": 9,
        "duration_months": 8
      },
      {
        "name": "Docker",
        "proficiency": "beginner",
        "endorsements": 12,
        "duration_months": 3
      },
      {
        "name": "GraphQL",
        "proficiency": "intermediate",
        "endorsements": 13,
        "duration_months": 27
      },
      {
        "name": "Agile",
        "proficiency": "intermediate",
        "endorsements": 14,
        "duration_months": 24
      },
      {
        "name": "MLOps",
        "proficiency": "intermediate",
        "endorsements": 13,
        "duration_months": 26
      },
      {
        "name": "Azure",
        "proficiency": "intermediate",
        "endorsements": 14,
        "duration_months": 27
      }
    ],
    "certifications": [],
    "languages": [
      {
        "language": "English",
        "proficiency": "professional"
      },
      {
        "language": "Hindi",
        "proficiency": "conversational"
      }
    ],
    "redrob_signals": {
      "profile_completeness_score": 35.8,
      "signup_date": "2026-03-25",
      "last_active_date": "2026-03-31",
      "open_to_work_flag": true,
      "profile_views_received_30d": 102,
      "applications_submitted_30d": 9,
      "recruiter_response_rate": 0.2,
      "avg_response_time_hours": 61.0,
      "skill_assessment_scores": {},
      "connection_count": 316,
      "endorsements_received": 51,
      "notice_period_days": 90,
      "expected_salary_range_inr_lpa": {
        "min": 9.2,
        "max": 15.9
      },
      "preferred_work_mode": "flexible",
      "willing_to_relocate": true,
      "github_activity_score": 37.8,
      "search_appearance_30d": 300,
      "saved_by_recruiters_30d": 18,
      "interview_completion_rate": 0.75,
      "offer_acceptance_rate": -1,
      "verified_email": true,
      "verified_phone": true,
      "linkedin_connected": false
    }
  },
  {
    "candidate_id": "CAND_0000039",
    "profile": {
      "anonymized_name": "Sai Saxena",
      "headline": "Marketing Manager | Helping teams scale",
      "summary": "Professional with 3.9+ years of experience. My professional background is in marketing manager \u2014 I've built and led teams, owned KPIs, and driven business outcomes in this domain. Lately I've been curious about how AI tools could augment my work \u2014 I've experimented with ChatGPT and a few other tools for productivity and content creation, and I think the space is exciting. Open to roles where I can apply my domain expertise alongside emerging AI capabilities.",
      "location": "Bhubaneswar, Odisha",
      "country": "India",
      "years_of_experience": 3.9,
      "current_title": "Marketing Manager",
      "current_company": "Acme Corp",
      "current_company_size": "201-500",
      "current_industry": "Manufacturing"
    },
    "career_history": [
      {
        "company": "Acme Corp",
        "title": "Marketing Manager",
        "start_date": "2024-08-05",
        "end_date": null,
        "duration_months": 22,
        "is_current": true,
        "industry": "Manufacturing",
        "company_size": "201-500",
        "description": "Mechanical engineering design role at a hardware-product company. Led the design of two product subsystems through full lifecycle: concept, DFM/DFMA review, prototype, production tooling. Comfortable with CAD (SolidWorks, Creo), FEA (ANSYS), and the typical hardware-development cadence. Worked closely with manufacturing partners on production scale-up."
      },
      {
        "company": "Stark Industries",
        "title": "Mechanical Engineer",
        "start_date": "2022-08-16",
        "end_date": "2024-08-05",
        "duration_months": 24,
        "is_current": false,
        "industry": "Manufacturing",
        "company_size": "1001-5000",
        "description": "Brand design and creative direction at a consumer-products company. Owned brand identity (logo, visual system, typography), packaging design, and digital creative across web and social. Led the recent rebrand and managed a small external agency for production work. Comfortable across the Adobe suite, Figma, and the production side of brand and packaging design."
      }
    ],
    "education": [
      {
        "institution": "Chandigarh University",
        "degree": "B.Tech",
        "field_of_study": "Electrical Engineering",
        "start_year": 2009,
        "end_year": 2014,
        "grade": "66%",
        "tier": "tier_3"
      },
      {
        "institution": "Chandigarh University",
        "degree": "M.E.",
        "field_of_study": "Artificial Intelligence",
        "start_year": 2007,
        "end_year": 2011,
        "grade": "82%",
        "tier": "tier_3"
      }
    ],
    "skills": [
      {
        "name": "Spark",
        "proficiency": "intermediate",
        "endorsements": 9,
        "duration_months": 15
      },
      {
        "name": "Tailwind",
        "proficiency": "intermediate",
        "endorsements": 9,
        "duration_months": 8
      },
      {
        "name": "Sales",
        "proficiency": "beginner",
        "endorsements": 6,
        "duration_months": 8
      },
      {
        "name": "CI/CD",
        "proficiency": "intermediate",
        "endorsements": 11,
        "duration_months": 35
      },
      {
        "name": "Illustrator",
        "proficiency": "intermediate",
        "endorsements": 9,
        "duration_months": 21
      },
      {
        "name": "Hadoop",
        "proficiency": "intermediate",
        "endorsements": 15,
        "duration_months": 26
      },
      {
        "name": "Microservices",
        "proficiency": "intermediate",
        "endorsements": 13,
        "duration_months": 19
      },
      {
        "name": "REST APIs",
        "proficiency": "beginner",
        "endorsements": 15,
        "duration_months": 12
      },
      {
        "name": "AWS",
        "proficiency": "intermediate",
        "endorsements": 7,
        "duration_months": 18
      }
    ],
    "certifications": [],
    "languages": [
      {
        "language": "English",
        "proficiency": "native"
      },
      {
        "language": "Hindi",
        "proficiency": "conversational"
      }
    ],
    "redrob_signals": {
      "profile_completeness_score": 28.3,
      "signup_date": "2025-04-14",
      "last_active_date": "2026-04-18",
      "open_to_work_flag": false,
      "profile_views_received_30d": 22,
      "applications_submitted_30d": 0,
      "recruiter_response_rate": 0.52,
      "avg_response_time_hours": 40.1,
      "skill_assessment_scores": {},
      "connection_count": 102,
      "endorsements_received": 9,
      "notice_period_days": 30,
      "expected_salary_range_inr_lpa": {
        "min": 17.4,
        "max": 16.0
      },
      "preferred_work_mode": "hybrid",
      "willing_to_relocate": true,
      "github_activity_score": -1,
      "search_appearance_30d": 146,
      "saved_by_recruiters_30d": 6,
      "interview_completion_rate": 0.74,
      "offer_acceptance_rate": -1,
      "verified_email": false,
      "verified_phone": true,
      "linkedin_connected": true
    }
  },
  {
    "candidate_id": "CAND_0000040",
    "profile": {
      "anonymized_name": "Dev Vora",
      "headline": "Customer Support | Helping teams scale",
      "summary": "Professional with 1.6+ years of experience. I've spent my career in marketing manager roles, mostly focused on driving outcomes through process, people, and customer relationships. Lately I've been curious about how AI tools could augment my work \u2014 I've experimented with ChatGPT and a few other tools for productivity and content creation, and I think the space is exciting. Open to roles where I can apply my domain expertise alongside emerging AI capabilities.",
      "location": "Kochi, Kerala",
      "country": "India",
      "years_of_experience": 1.6,
      "current_title": "Customer Support",
      "current_company": "Globex Inc",
      "current_company_size": "501-1000",
      "current_industry": "Manufacturing"
    },
    "career_history": [
      {
        "company": "Globex Inc",
        "title": "Customer Support",
        "start_date": "2024-11-03",
        "end_date": null,
        "duration_months": 19,
        "is_current": true,
        "industry": "Manufacturing",
        "company_size": "501-1000",
        "description": "Enterprise sales of cloud software solutions into the mid-market segment. Carried a $1.8M ARR quota and consistently delivered against it across the last three years. Owned the full sales cycle: prospecting, discovery, technical evaluation (with SE support), commercial negotiation, and close. Strong on consultative selling for technical buyers; comfortable engaging with both engineering and finance stakeholders."
      }
    ],
    "education": [
      {
        "institution": "Local Engineering College",
        "degree": "B.Tech",
        "field_of_study": "MBA",
        "start_year": 2010,
        "end_year": 2013,
        "grade": "7.45 CGPA",
        "tier": "tier_4"
      }
    ],
    "skills": [
      {
        "name": "SQL",
        "proficiency": "beginner",
        "endorsements": 12,
        "duration_months": 4
      },
      {
        "name": "Spring Boot",
        "proficiency": "intermediate",
        "endorsements": 15,
        "duration_months": 27
      },
      {
        "name": "Accounting",
        "proficiency": "intermediate",
        "endorsements": 15,
        "duration_months": 16
      },
      {
        "name": "Rust",
        "proficiency": "beginner",
        "endorsements": 12,
        "duration_months": 15
      },
      {
        "name": "Redux",
        "proficiency": "intermediate",
        "endorsements": 2,
        "duration_months": 19
      },
      {
        "name": "SAP",
        "proficiency": "intermediate",
        "endorsements": 14,
        "duration_months": 25
      },
      {
        "name": "Weights & Biases",
        "proficiency": "intermediate",
        "endorsements": 10,
        "duration_months": 21
      },
      {
        "name": "REST APIs",
        "proficiency": "intermediate",
        "endorsements": 6,
        "duration_months": 18
      },
      {
        "name": "Spark",
        "proficiency": "beginner",
        "endorsements": 4,
        "duration_months": 14
      }
    ],
    "certifications": [],
    "languages": [
      {
        "language": "English",
        "proficiency": "professional"
      },
      {
        "language": "Hindi",
        "proficiency": "conversational"
      }
    ],
    "redrob_signals": {
      "profile_completeness_score": 39.5,
      "signup_date": "2024-10-03",
      "last_active_date": "2026-01-31",
      "open_to_work_flag": false,
      "profile_views_received_30d": 52,
      "applications_submitted_30d": 6,
      "recruiter_response_rate": 0.46,
      "avg_response_time_hours": 268.3,
      "skill_assessment_scores": {},
      "connection_count": 176,
      "endorsements_received": 35,
      "notice_period_days": 90,
      "expected_salary_range_inr_lpa": {
        "min": 7.6,
        "max": 8.2
      },
      "preferred_work_mode": "remote",
      "willing_to_relocate": false,
      "github_activity_score": -1,
      "search_appearance_30d": 95,
      "saved_by_recruiters_30d": 1,
      "interview_completion_rate": 0.8,
      "offer_acceptance_rate": 0.44,
      "verified_email": true,
      "verified_phone": false,
      "linkedin_connected": false
    }
  },
  {
    "candidate_id": "CAND_0000041",
    "profile": {
      "anonymized_name": "Anjali Khanna",
      "headline": "Operations Manager | Helping teams scale",
      "summary": "Professional with 13.7+ years of experience. I've spent my career in marketing manager roles, mostly focused on driving outcomes through process, people, and customer relationships. Lately I've been curious about how AI tools could augment my work \u2014 I've experimented with ChatGPT and a few other tools for productivity and content creation, and I think the space is exciting. Open to roles where I can apply my domain expertise alongside emerging AI capabilities.",
      "location": "Delhi, Delhi",
      "country": "India",
      "years_of_experience": 13.7,
      "current_title": "Operations Manager",
      "current_company": "Hooli",
      "current_company_size": "1001-5000",
      "current_industry": "Software"
    },
    "career_history": [
      {
        "company": "Hooli",
        "title": "Operations Manager",
        "start_date": "2022-12-14",
        "end_date": null,
        "duration_months": 42,
        "is_current": true,
        "industry": "Software",
        "company_size": "1001-5000",
        "description": "Customer support team lead at a SaaS product. Managed a team of 8 support agents handling tier-1 and tier-2 tickets; owned the escalation process to engineering and the customer-feedback loop to product. Built out the support knowledge base and the agent training program. Strong on the people-management side and the process side; lighter on technical depth beyond product expertise."
      },
      {
        "company": "Acme Corp",
        "title": "Business Analyst",
        "start_date": "2021-03-24",
        "end_date": "2022-12-14",
        "duration_months": 21,
        "is_current": false,
        "industry": "Manufacturing",
        "company_size": "201-500",
        "description": "Mechanical engineering design role at a hardware-product company. Led the design of two product subsystems through full lifecycle: concept, DFM/DFMA review, prototype, production tooling. Comfortable with CAD (SolidWorks, Creo), FEA (ANSYS), and the typical hardware-development cadence. Worked closely with manufacturing partners on production scale-up."
      },
      {
        "company": "Wayne Enterprises",
        "title": "Content Writer",
        "start_date": "2019-03-21",
        "end_date": "2021-03-10",
        "duration_months": 24,
        "is_current": false,
        "industry": "Conglomerate",
        "company_size": "10001+",
        "description": "Operations management role at a logistics company. Owned daily fulfillment operations across 3 warehouses, managing a team of 80 across receiving, picking, packing, and outbound. Built and tracked the operational KPIs (on-time fulfillment, accuracy, cost per order) and led the continuous improvement initiatives that drove a 22% productivity gain over 18 months."
      },
      {
        "company": "Infosys",
        "title": "Content Writer",
        "start_date": "2014-12-05",
        "end_date": "2019-03-14",
        "duration_months": 52,
        "is_current": false,
        "industry": "IT Services",
        "company_size": "10001+",
        "description": "Customer support team lead at a SaaS product. Managed a team of 8 support agents handling tier-1 and tier-2 tickets; owned the escalation process to engineering and the customer-feedback loop to product. Built out the support knowledge base and the agent training program. Strong on the people-management side and the process side; lighter on technical depth beyond product expertise."
      },
      {
        "company": "Dunder Mifflin",
        "title": "Mechanical Engineer",
        "start_date": "2013-01-07",
        "end_date": "2014-11-28",
        "duration_months": 23,
        "is_current": false,
        "industry": "Paper Products",
        "company_size": "201-500",
        "description": "Marketing leadership role at a B2B SaaS company. Owned the demand-generation function \u2014 content marketing, paid acquisition, SEO, email nurture. Built and managed a team of 5 across content, performance marketing, and marketing operations. Worked closely with sales on lead-quality definitions and the SDR-handoff process. Recent focus has been on account-based marketing for our enterprise segment."
      }
    ],
    "education": [
      {
        "institution": "Local Engineering College",
        "degree": "M.S.",
        "field_of_study": "Machine Learning",
        "start_year": 2013,
        "end_year": 2018,
        "grade": "8.73 CGPA",
        "tier": "tier_4"
      }
    ],
    "skills": [
      {
        "name": "Airflow",
        "proficiency": "intermediate",
        "endorsements": 13,
        "duration_months": 20
      },
      {
        "name": "SQL",
        "proficiency": "intermediate",
        "endorsements": 12,
        "duration_months": 12
      },
      {
        "name": "Go",
        "proficiency": "beginner",
        "endorsements": 13,
        "duration_months": 8
      },
      {
        "name": "GCP",
        "proficiency": "beginner",
        "endorsements": 0,
        "duration_months": 3
      },
      {
        "name": "Figma",
        "proficiency": "beginner",
        "endorsements": 14,
        "duration_months": 16
      },
      {
        "name": "React",
        "proficiency": "intermediate",
        "endorsements": 15,
        "duration_months": 22
      },
      {
        "name": "Webpack",
        "proficiency": "beginner",
        "endorsements": 3,
        "duration_months": 9
      },
      {
        "name": "Kubernetes",
        "proficiency": "beginner",
        "endorsements": 5,
        "duration_months": 3
      },
      {
        "name": "Angular",
        "proficiency": "intermediate",
        "endorsements": 2,
        "duration_months": 31
      }
    ],
    "certifications": [],
    "languages": [
      {
        "language": "English",
        "proficiency": "professional"
      },
      {
        "language": "Hindi",
        "proficiency": "professional"
      }
    ],
    "redrob_signals": {
      "profile_completeness_score": 75.9,
      "signup_date": "2025-08-18",
      "last_active_date": "2026-03-16",
      "open_to_work_flag": true,
      "profile_views_received_30d": 3,
      "applications_submitted_30d": 9,
      "recruiter_response_rate": 0.07,
      "avg_response_time_hours": 135.3,
      "skill_assessment_scores": {},
      "connection_count": 143,
      "endorsements_received": 19,
      "notice_period_days": 90,
      "expected_salary_range_inr_lpa": {
        "min": 5.4,
        "max": 9.3
      },
      "preferred_work_mode": "remote",
      "willing_to_relocate": true,
      "github_activity_score": -1,
      "search_appearance_30d": 150,
      "saved_by_recruiters_30d": 6,
      "interview_completion_rate": 0.85,
      "offer_acceptance_rate": 0.16,
      "verified_email": true,
      "verified_phone": true,
      "linkedin_connected": true
    }
  },
  {
    "candidate_id": "CAND_0000042",
    "profile": {
      "anonymized_name": "Riya Mukherjee",
      "headline": "HR Manager | 5.0+ yrs experience",
      "summary": "Professional with 5.0+ years of experience. I'm a marketing manager with substantial experience in cross-functional collaboration, stakeholder management, and execution. Lately I've been curious about how AI tools could augment my work \u2014 I've experimented with ChatGPT and a few other tools for productivity and content creation, and I think the space is exciting. Open to roles where I can apply my domain expertise alongside emerging AI capabilities.",
      "location": "Berlin",
      "country": "Germany",
      "years_of_experience": 5.0,
      "current_title": "HR Manager",
      "current_company": "Wayne Enterprises",
      "current_company_size": "10001+",
      "current_industry": "Conglomerate"
    },
    "career_history": [
      {
        "company": "Wayne Enterprises",
        "title": "HR Manager",
        "start_date": "2024-04-07",
        "end_date": null,
        "duration_months": 26,
        "is_current": true,
        "industry": "Conglomerate",
        "company_size": "10001+",
        "description": "Content writing and SEO strategy for a tech-focused publication. Wrote longform articles on developer tools, cloud platforms, and AI/ML topics \u2014 including some that ranked on the first page of search for high-competition keywords. Managed a freelance writer pool and the editorial calendar. Recent work has been on AI-assisted content production, using LLM tools for research, drafting, and editing while maintaining editorial quality."
      },
      {
        "company": "Wipro",
        "title": "Business Analyst",
        "start_date": "2023-03-14",
        "end_date": "2024-03-08",
        "duration_months": 12,
        "is_current": false,
        "industry": "IT Services",
        "company_size": "10001+",
        "description": "Enterprise sales of cloud software solutions into the mid-market segment. Carried a $1.8M ARR quota and consistently delivered against it across the last three years. Owned the full sales cycle: prospecting, discovery, technical evaluation (with SE support), commercial negotiation, and close. Strong on consultative selling for technical buyers; comfortable engaging with both engineering and finance stakeholders."
      },
      {
        "company": "Infosys",
        "title": "Customer Support",
        "start_date": "2021-04-23",
        "end_date": "2023-01-13",
        "duration_months": 21,
        "is_current": false,
        "industry": "IT Services",
        "company_size": "10001+",
        "description": "Marketing leadership role at a B2B SaaS company. Owned the demand-generation function \u2014 content marketing, paid acquisition, SEO, email nurture. Built and managed a team of 5 across content, performance marketing, and marketing operations. Worked closely with sales on lead-quality definitions and the SDR-handoff process. Recent focus has been on account-based marketing for our enterprise segment."
      }
    ],
    "education": [
      {
        "institution": "SRM Chennai",
        "degree": "B.Tech",
        "field_of_study": "Civil Engineering",
        "start_year": 2014,
        "end_year": 2019,
        "grade": "7.40 CGPA",
        "tier": "tier_3"
      },
      {
        "institution": "Symbiosis International",
        "degree": "B.E.",
        "field_of_study": "Mathematics",
        "start_year": 2017,
        "end_year": 2021,
        "grade": "8.07 CGPA",
        "tier": "tier_3"
      }
    ],
    "skills": [
      {
        "name": "Project Management",
        "proficiency": "intermediate",
        "endorsements": 3,
        "duration_months": 33
      },
      {
        "name": "gRPC",
        "proficiency": "beginner",
        "endorsements": 3,
        "duration_months": 10
      },
      {
        "name": "Marketing",
        "proficiency": "intermediate",
        "endorsements": 10,
        "duration_months": 24
      },
      {
        "name": "SAP",
        "proficiency": "beginner",
        "endorsements": 12,
        "duration_months": 18
      },
      {
        "name": "Illustrator",
        "proficiency": "beginner",
        "endorsements": 11,
        "duration_months": 14
      },
      {
        "name": "Node.js",
        "proficiency": "intermediate",
        "endorsements": 3,
        "duration_months": 14
      },
      {
        "name": "YOLO",
        "proficiency": "intermediate",
        "endorsements": 5,
        "duration_months": 22
      },
      {
        "name": "Tailwind",
        "proficiency": "beginner",
        "endorsements": 8,
        "duration_months": 16
      },
      {
        "name": "CSS",
        "proficiency": "beginner",
        "endorsements": 11,
        "duration_months": 12
      },
      {
        "name": "Figma",
        "proficiency": "beginner",
        "endorsements": 10,
        "duration_months": 7
      }
    ],
    "certifications": [
      {
        "name": "Six Sigma Green Belt",
        "issuer": "ASQ",
        "year": 2022
      },
      {
        "name": "Scrum Master Certified",
        "issuer": "Scrum Alliance",
        "year": 2021
      }
    ],
    "languages": [
      {
        "language": "English",
        "proficiency": "professional"
      },
      {
        "language": "Hindi",
        "proficiency": "conversational"
      }
    ],
    "redrob_signals": {
      "profile_completeness_score": 58.6,
      "signup_date": "2025-05-02",
      "last_active_date": "2025-10-23",
      "open_to_work_flag": false,
      "profile_views_received_30d": 57,
      "applications_submitted_30d": 8,
      "recruiter_response_rate": 0.58,
      "avg_response_time_hours": 24.8,
      "skill_assessment_scores": {},
      "connection_count": 591,
      "endorsements_received": 29,
      "notice_period_days": 30,
      "expected_salary_range_inr_lpa": {
        "min": 10.5,
        "max": 18.8
      },
      "preferred_work_mode": "flexible",
      "willing_to_relocate": true,
      "github_activity_score": -1,
      "search_appearance_30d": 34,
      "saved_by_recruiters_30d": 9,
      "interview_completion_rate": 0.35,
      "offer_acceptance_rate": -1,
      "verified_email": true,
      "verified_phone": true,
      "linkedin_connected": false
    }
  },
  {
    "candidate_id": "CAND_0000043",
    "profile": {
      "anonymized_name": "Aarav Sen",
      "headline": "Cloud Engineer | Full-stack development",
      "summary": "Software engineer with 8.3 years of experience across web, backend, and cloud systems. Strong fundamentals in software development and system design. I've worked across web frontends, REST APIs, and cloud deployments; comfortable in most parts of a typical SaaS stack. I've been keeping up with AI/ML at a self-learner level \u2014 taken some online courses, played with the OpenAI and Anthropic APIs, built a small RAG side project \u2014 but I haven't done it in a professional capacity yet. Open to roles where I can either deepen my software engineering work or, if the team is open to it, start contributing to ML-adjacent systems.",
      "location": "Chandigarh, Chandigarh",
      "country": "India",
      "years_of_experience": 8.3,
      "current_title": "Cloud Engineer",
      "current_company": "Swiggy",
      "current_company_size": "5001-10000",
      "current_industry": "Food Delivery"
    },
    "career_history": [
      {
        "company": "Swiggy",
        "title": "Cloud Engineer",
        "start_date": "2023-12-09",
        "end_date": null,
        "duration_months": 30,
        "is_current": true,
        "industry": "Food Delivery",
        "company_size": "5001-10000",
        "description": "Test automation and QA engineering for a fintech product. Built and maintained the end-to-end test suite using Selenium and pytest, plus the load-testing setup using Locust. Worked closely with developers on testability patterns and with product on acceptance criteria. Recent work has been on shifting test responsibility into the dev team \u2014 moving from QA-as-gate to QA-as-coach. Career has been entirely in QA/test engineering."
      },
      {
        "company": "HCL",
        "title": ".NET Developer",
        "start_date": "2021-12-19",
        "end_date": "2023-12-09",
        "duration_months": 24,
        "is_current": false,
        "industry": "IT Services",
        "company_size": "10001+",
        "description": "Cloud infrastructure and DevOps work at an enterprise SaaS company. Owned the AWS account architecture (VPC, IAM, networking), the Terraform modules for our service deployments, and the Kubernetes cluster operations. Designed the CI/CD pipelines (GitLab CI + ArgoCD) and the monitoring stack (Prometheus, Grafana, Loki). Strong on the infra and ops side; haven't done much application development."
      },
      {
        "company": "Mindtree",
        "title": "Frontend Engineer",
        "start_date": "2019-11-30",
        "end_date": "2021-12-19",
        "duration_months": 25,
        "is_current": false,
        "industry": "IT Services",
        "company_size": "10001+",
        "description": "Android mobile development using Java and (more recently) Kotlin at a consumer-app company. Built and maintained multiple production features including the main shopping flow, push notification system, and the offline-first sync layer. Comfortable with the Android framework, Jetpack components, and the typical patterns (MVVM, Hilt, Coroutines). My career has been entirely on mobile so far; interested in expanding into broader backend or platform engineering."
      },
      {
        "company": "Initech",
        "title": "DevOps Engineer",
        "start_date": "2018-04-09",
        "end_date": "2019-11-30",
        "duration_months": 20,
        "is_current": false,
        "industry": "Software",
        "company_size": "51-200",
        "description": "Android mobile development using Java and (more recently) Kotlin at a consumer-app company. Built and maintained multiple production features including the main shopping flow, push notification system, and the offline-first sync layer. Comfortable with the Android framework, Jetpack components, and the typical patterns (MVVM, Hilt, Coroutines). My career has been entirely on mobile so far; interested in expanding into broader backend or platform engineering."
      }
    ],
    "education": [
      {
        "institution": "Regional Technical Institute",
        "degree": "M.E.",
        "field_of_study": "Electronics",
        "start_year": 2012,
        "end_year": 2017,
        "grade": "8.77 CGPA",
        "tier": "tier_4"
      },
      {
        "institution": "Regional Technical Institute",
        "degree": "B.Sc",
        "field_of_study": "Physics",
        "start_year": 2001,
        "end_year": 2005,
        "grade": "9.29 CGPA",
        "tier": "tier_4"
      }
    ],
    "skills": [
      {
        "name": "Elasticsearch",
        "proficiency": "advanced",
        "endorsements": 54,
        "duration_months": 44
      },
      {
        "name": "OpenSearch",
        "proficiency": "intermediate",
        "endorsements": 21,
        "duration_months": 34
      },
      {
        "name": "Airflow",
        "proficiency": "beginner",
        "endorsements": 5,
        "duration_months": 2
      },
      {
        "name": "Kubeflow",
        "proficiency": "advanced",
        "endorsements": 12,
        "duration_months": 55
      },
      {
        "name": "Fine-tuning LLMs",
        "proficiency": "intermediate",
        "endorsements": 8,
        "duration_months": 14
      },
      {
        "name": "Haystack",
        "proficiency": "advanced",
        "endorsements": 11,
        "duration_months": 27
      },
      {
        "name": "OpenCV",
        "proficiency": "advanced",
        "endorsements": 54,
        "duration_months": 33
      },
      {
        "name": "TensorFlow",
        "proficiency": "intermediate",
        "endorsements": 0,
        "duration_months": 19
      },
      {
        "name": "PEFT",
        "proficiency": "intermediate",
        "endorsements": 5,
        "duration_months": 21
      },
      {
        "name": "LangChain",
        "proficiency": "intermediate",
        "endorsements": 37,
        "duration_months": 25
      },
      {
        "name": "Weights & Biases",
        "proficiency": "intermediate",
        "endorsements": 4,
        "duration_months": 26
      },
      {
        "name": "Reinforcement Learning",
        "proficiency": "advanced",
        "endorsements": 0,
        "duration_months": 30
      },
      {
        "name": "Deep Learning",
        "proficiency": "intermediate",
        "endorsements": 0,
        "duration_months": 16
      },
      {
        "name": "Image Classification",
        "proficiency": "advanced",
        "endorsements": 24,
        "duration_months": 50
      },
      {
        "name": "Node.js",
        "proficiency": "intermediate",
        "endorsements": 12,
        "duration_months": 24
      },
      {
        "name": "Project Management",
        "proficiency": "intermediate",
        "endorsements": 0,
        "duration_months": 36
      },
      {
        "name": "Feature Engineering",
        "proficiency": "intermediate",
        "endorsements": 0,
        "duration_months": 20
      }
    ],
    "certifications": [],
    "languages": [
      {
        "language": "English",
        "proficiency": "native"
      },
      {
        "language": "Hindi",
        "proficiency": "professional"
      }
    ],
    "redrob_signals": {
      "profile_completeness_score": 57.0,
      "signup_date": "2024-09-30",
      "last_active_date": "2026-01-01",
      "open_to_work_flag": false,
      "profile_views_received_30d": 38,
      "applications_submitted_30d": 8,
      "recruiter_response_rate": 0.04,
      "avg_response_time_hours": 223.5,
      "skill_assessment_scores": {},
      "connection_count": 102,
      "endorsements_received": 39,
      "notice_period_days": 120,
      "expected_salary_range_inr_lpa": {
        "min": 6.3,
        "max": 21.2
      },
      "preferred_work_mode": "remote",
      "willing_to_relocate": true,
      "github_activity_score": -1,
      "search_appearance_30d": 167,
      "saved_by_recruiters_30d": 2,
      "interview_completion_rate": 0.72,
      "offer_acceptance_rate": -1,
      "verified_email": true,
      "verified_phone": true,
      "linkedin_connected": false
    }
  },
  {
    "candidate_id": "CAND_0000044",
    "profile": {
      "anonymized_name": "Vihaan Naidu",
      "headline": "Frontend Engineer | Backend systems & APIs",
      "summary": "Software engineer with 5.7 years of experience across web, backend, and cloud systems. Strong fundamentals in software development and system design. I've spent most of my career on web and API development \u2014 Python/Django and Node.js mostly. I've been keeping up with AI/ML at a self-learner level \u2014 taken some online courses, played with the OpenAI and Anthropic APIs, built a small RAG side project \u2014 but I haven't done it in a professional capacity yet. Open to roles where I can either deepen my software engineering work or, if the team is open to it, start contributing to ML-adjacent systems.",
      "location": "Indore, Madhya Pradesh",
      "country": "India",
      "years_of_experience": 5.7,
      "current_title": "Frontend Engineer",
      "current_company": "Tech Mahindra",
      "current_company_size": "10001+",
      "current_industry": "IT Services"
    },
    "career_history": [
      {
        "company": "Tech Mahindra",
        "title": "Frontend Engineer",
        "start_date": "2022-04-18",
        "end_date": null,
        "duration_months": 50,
        "is_current": true,
        "industry": "IT Services",
        "company_size": "10001+",
        "description": "Java backend development at a large enterprise \u2014 Spring Boot microservices, Kafka for inter-service messaging, Postgres + Redis for storage. Worked on the customer onboarding flow which involved orchestrating multiple downstream services. Solid on the Spring ecosystem, transaction handling, and the operational side of Java services. Looking to either go deeper on distributed systems or expand into modern application stacks."
      },
      {
        "company": "Hooli",
        "title": "DevOps Engineer",
        "start_date": "2020-10-25",
        "end_date": "2022-04-18",
        "duration_months": 18,
        "is_current": false,
        "industry": "Software",
        "company_size": "1001-5000",
        "description": "Frontend engineering at a media company. React, TypeScript, and the typical surrounding tooling (Webpack, Jest, Cypress). Built the company's design system from scratch and led the migration from a legacy AngularJS app. Strong on the frontend craft \u2014 accessibility, performance, animations \u2014 but limited backend exposure."
      }
    ],
    "education": [
      {
        "institution": "Chandigarh University",
        "degree": "M.Sc",
        "field_of_study": "Information Technology",
        "start_year": 2016,
        "end_year": 2021,
        "grade": "84%",
        "tier": "tier_3"
      },
      {
        "institution": "Amity University",
        "degree": "B.E.",
        "field_of_study": "Civil Engineering",
        "start_year": 2002,
        "end_year": 2006,
        "grade": "9.33 CGPA",
        "tier": "tier_3"
      }
    ],
    "skills": [
      {
        "name": "Hadoop",
        "proficiency": "beginner",
        "endorsements": 3,
        "duration_months": 3
      },
      {
        "name": "JavaScript",
        "proficiency": "beginner",
        "endorsements": 5,
        "duration_months": 18
      },
      {
        "name": "Databricks",
        "proficiency": "beginner",
        "endorsements": 13,
        "duration_months": 5
      },
      {
        "name": "Python",
        "proficiency": "intermediate",
        "endorsements": 2,
        "duration_months": 26
      },
      {
        "name": "dbt",
        "proficiency": "intermediate",
        "endorsements": 3,
        "duration_months": 16
      },
      {
        "name": "CI/CD",
        "proficiency": "beginner",
        "endorsements": 11,
        "duration_months": 13
      }
    ],
    "certifications": [],
    "languages": [
      {
        "language": "English",
        "proficiency": "professional"
      },
      {
        "language": "Hindi",
        "proficiency": "professional"
      }
    ],
    "redrob_signals": {
      "profile_completeness_score": 30.6,
      "signup_date": "2023-02-16",
      "last_active_date": "2025-12-11",
      "open_to_work_flag": false,
      "profile_views_received_30d": 78,
      "applications_submitted_30d": 7,
      "recruiter_response_rate": 0.66,
      "avg_response_time_hours": 179.1,
      "skill_assessment_scores": {},
      "connection_count": 58,
      "endorsements_received": 38,
      "notice_period_days": 90,
      "expected_salary_range_inr_lpa": {
        "min": 8.8,
        "max": 12.3
      },
      "preferred_work_mode": "remote",
      "willing_to_relocate": false,
      "github_activity_score": 6.5,
      "search_appearance_30d": 288,
      "saved_by_recruiters_30d": 18,
      "interview_completion_rate": 0.66,
      "offer_acceptance_rate": -1,
      "verified_email": true,
      "verified_phone": false,
      "linkedin_connected": false
    }
  },
  {
    "candidate_id": "CAND_0000045",
    "profile": {
      "anonymized_name": "Vikram Mittal",
      "headline": "Project Manager | 12.2+ yrs experience",
      "summary": "Professional with 12.2+ years of experience. My professional background is in marketing manager \u2014 I've built and led teams, owned KPIs, and driven business outcomes in this domain. Lately I've been curious about how AI tools could augment my work \u2014 I've experimented with ChatGPT and a few other tools for productivity and content creation, and I think the space is exciting. Open to roles where I can apply my domain expertise alongside emerging AI capabilities.",
      "location": "Indore, Madhya Pradesh",
      "country": "India",
      "years_of_experience": 12.2,
      "current_title": "Project Manager",
      "current_company": "Initech",
      "current_company_size": "51-200",
      "current_industry": "Software"
    },
    "career_history": [
      {
        "company": "Initech",
        "title": "Project Manager",
        "start_date": "2025-03-03",
        "end_date": null,
        "duration_months": 15,
        "is_current": true,
        "industry": "Software",
        "company_size": "51-200",
        "description": "Enterprise sales of cloud software solutions into the mid-market segment. Carried a $1.8M ARR quota and consistently delivered against it across the last three years. Owned the full sales cycle: prospecting, discovery, technical evaluation (with SE support), commercial negotiation, and close. Strong on consultative selling for technical buyers; comfortable engaging with both engineering and finance stakeholders."
      },
      {
        "company": "Pied Piper",
        "title": "Civil Engineer",
        "start_date": "2024-02-07",
        "end_date": "2025-03-03",
        "duration_months": 13,
        "is_current": false,
        "industry": "Software",
        "company_size": "11-50",
        "description": "Mechanical engineering design role at a hardware-product company. Led the design of two product subsystems through full lifecycle: concept, DFM/DFMA review, prototype, production tooling. Comfortable with CAD (SolidWorks, Creo), FEA (ANSYS), and the typical hardware-development cadence. Worked closely with manufacturing partners on production scale-up."
      },
      {
        "company": "Infosys",
        "title": "Marketing Manager",
        "start_date": "2023-01-29",
        "end_date": "2024-01-24",
        "duration_months": 12,
        "is_current": false,
        "industry": "IT Services",
        "company_size": "10001+",
        "description": "Customer support team lead at a SaaS product. Managed a team of 8 support agents handling tier-1 and tier-2 tickets; owned the escalation process to engineering and the customer-feedback loop to product. Built out the support knowledge base and the agent training program. Strong on the people-management side and the process side; lighter on technical depth beyond product expertise."
      },
      {
        "company": "Wayne Enterprises",
        "title": "Content Writer",
        "start_date": "2021-10-06",
        "end_date": "2023-01-29",
        "duration_months": 16,
        "is_current": false,
        "industry": "Conglomerate",
        "company_size": "10001+",
        "description": "Marketing leadership role at a B2B SaaS company. Owned the demand-generation function \u2014 content marketing, paid acquisition, SEO, email nurture. Built and managed a team of 5 across content, performance marketing, and marketing operations. Worked closely with sales on lead-quality definitions and the SDR-handoff process. Recent focus has been on account-based marketing for our enterprise segment."
      },
      {
        "company": "TCS",
        "title": "Graphic Designer",
        "start_date": "2018-08-09",
        "end_date": "2021-09-22",
        "duration_months": 38,
        "is_current": false,
        "industry": "IT Services",
        "company_size": "10001+",
        "description": "Customer support team lead at a SaaS product. Managed a team of 8 support agents handling tier-1 and tier-2 tickets; owned the escalation process to engineering and the customer-feedback loop to product. Built out the support knowledge base and the agent training program. Strong on the people-management side and the process side; lighter on technical depth beyond product expertise."
      },
      {
        "company": "Wayne Enterprises",
        "title": "Graphic Designer",
        "start_date": "2016-04-21",
        "end_date": "2018-08-09",
        "duration_months": 28,
        "is_current": false,
        "industry": "Conglomerate",
        "company_size": "10001+",
        "description": "Enterprise sales of cloud software solutions into the mid-market segment. Carried a $1.8M ARR quota and consistently delivered against it across the last three years. Owned the full sales cycle: prospecting, discovery, technical evaluation (with SE support), commercial negotiation, and close. Strong on consultative selling for technical buyers; comfortable engaging with both engineering and finance stakeholders."
      },
      {
        "company": "Stark Industries",
        "title": "Operations Manager",
        "start_date": "2014-07-17",
        "end_date": "2016-04-07",
        "duration_months": 21,
        "is_current": false,
        "industry": "Manufacturing",
        "company_size": "1001-5000",
        "description": "Enterprise sales of cloud software solutions into the mid-market segment. Carried a $1.8M ARR quota and consistently delivered against it across the last three years. Owned the full sales cycle: prospecting, discovery, technical evaluation (with SE support), commercial negotiation, and close. Strong on consultative selling for technical buyers; comfortable engaging with both engineering and finance stakeholders."
      }
    ],
    "education": [
      {
        "institution": "Generic State University",
        "degree": "M.E.",
        "field_of_study": "Statistics",
        "start_year": 2002,
        "end_year": 2005,
        "grade": "7.91 CGPA",
        "tier": "tier_4"
      },
      {
        "institution": "KIIT University",
        "degree": "B.Tech",
        "field_of_study": "Machine Learning",
        "start_year": 2016,
        "end_year": 2020,
        "grade": "86%",
        "tier": "tier_3"
      }
    ],
    "skills": [
      {
        "name": "GCP",
        "proficiency": "beginner",
        "endorsements": 1,
        "duration_months": 3
      },
      {
        "name": "Sales",
        "proficiency": "intermediate",
        "endorsements": 2,
        "duration_months": 12
      },
      {
        "name": "Redux",
        "proficiency": "intermediate",
        "endorsements": 14,
        "duration_months": 25
      },
      {
        "name": "PostgreSQL",
        "proficiency": "intermediate",
        "endorsements": 1,
        "duration_months": 36
      },
      {
        "name": "Airflow",
        "proficiency": "intermediate",
        "endorsements": 5,
        "duration_months": 18
      },
      {
        "name": "SAP",
        "proficiency": "intermediate",
        "endorsements": 13,
        "duration_months": 27
      }
    ],
    "certifications": [
      {
        "name": "Scrum Master Certified",
        "issuer": "Scrum Alliance",
        "year": 2024
      },
      {
        "name": "Six Sigma Green Belt",
        "issuer": "ASQ",
        "year": 2024
      }
    ],
    "languages": [
      {
        "language": "English",
        "proficiency": "professional"
      },
      {
        "language": "Hindi",
        "proficiency": "professional"
      }
    ],
    "redrob_signals": {
      "profile_completeness_score": 25.4,
      "signup_date": "2023-04-15",
      "last_active_date": "2026-04-24",
      "open_to_work_flag": true,
      "profile_views_received_30d": 37,
      "applications_submitted_30d": 0,
      "recruiter_response_rate": 0.62,
      "avg_response_time_hours": 20.8,
      "skill_assessment_scores": {},
      "connection_count": 490,
      "endorsements_received": 2,
      "notice_period_days": 60,
      "expected_salary_range_inr_lpa": {
        "min": 9.7,
        "max": 12.3
      },
      "preferred_work_mode": "hybrid",
      "willing_to_relocate": true,
      "github_activity_score": -1,
      "search_appearance_30d": 23,
      "saved_by_recruiters_30d": 6,
      "interview_completion_rate": 0.33,
      "offer_acceptance_rate": -1,
      "verified_email": true,
      "verified_phone": true,
      "linkedin_connected": true
    }
  },
  {
    "candidate_id": "CAND_0000046",
    "profile": {
      "anonymized_name": "Dev Nair",
      "headline": "Mechanical Engineer | 7.8+ yrs experience",
      "summary": "Professional with 7.8+ years of experience. I've spent my career in marketing manager roles, mostly focused on driving outcomes through process, people, and customer relationships. Lately I've been curious about how AI tools could augment my work \u2014 I've experimented with ChatGPT and a few other tools for productivity and content creation, and I think the space is exciting. Open to roles where I can apply my domain expertise alongside emerging AI capabilities.",
      "location": "London",
      "country": "UK",
      "years_of_experience": 7.8,
      "current_title": "Mechanical Engineer",
      "current_company": "Hooli",
      "current_company_size": "1001-5000",
      "current_industry": "Software"
    },
    "career_history": [
      {
        "company": "Hooli",
        "title": "Mechanical Engineer",
        "start_date": "2023-12-09",
        "end_date": null,
        "duration_months": 30,
        "is_current": true,
        "industry": "Software",
        "company_size": "1001-5000",
        "description": "Customer support team lead at a SaaS product. Managed a team of 8 support agents handling tier-1 and tier-2 tickets; owned the escalation process to engineering and the customer-feedback loop to product. Built out the support knowledge base and the agent training program. Strong on the people-management side and the process side; lighter on technical depth beyond product expertise."
      },
      {
        "company": "Pied Piper",
        "title": "HR Manager",
        "start_date": "2021-02-22",
        "end_date": "2023-11-09",
        "duration_months": 33,
        "is_current": false,
        "industry": "Software",
        "company_size": "11-50",
        "description": "Marketing leadership role at a B2B SaaS company. Owned the demand-generation function \u2014 content marketing, paid acquisition, SEO, email nurture. Built and managed a team of 5 across content, performance marketing, and marketing operations. Worked closely with sales on lead-quality definitions and the SDR-handoff process. Recent focus has been on account-based marketing for our enterprise segment."
      },
      {
        "company": "Hooli",
        "title": "Marketing Manager",
        "start_date": "2018-08-23",
        "end_date": "2021-02-08",
        "duration_months": 30,
        "is_current": false,
        "industry": "Software",
        "company_size": "1001-5000",
        "description": "Operations management role at a logistics company. Owned daily fulfillment operations across 3 warehouses, managing a team of 80 across receiving, picking, packing, and outbound. Built and tracked the operational KPIs (on-time fulfillment, accuracy, cost per order) and led the continuous improvement initiatives that drove a 22% productivity gain over 18 months."
      }
    ],
    "education": [
      {
        "institution": "Christ University",
        "degree": "M.Tech",
        "field_of_study": "Electrical Engineering",
        "start_year": 2014,
        "end_year": 2017,
        "grade": "9.49 CGPA",
        "tier": "tier_3"
      },
      {
        "institution": "Chandigarh University",
        "degree": "B.E.",
        "field_of_study": "Statistics",
        "start_year": 2004,
        "end_year": 2009,
        "grade": "7.94 CGPA",
        "tier": "tier_3"
      }
    ],
    "skills": [
      {
        "name": "Agile",
        "proficiency": "beginner",
        "endorsements": 1,
        "duration_months": 9
      },
      {
        "name": "Scrum",
        "proficiency": "intermediate",
        "endorsements": 11,
        "duration_months": 28
      },
      {
        "name": "SAP",
        "proficiency": "intermediate",
        "endorsements": 6,
        "duration_months": 35
      },
      {
        "name": "React",
        "proficiency": "beginner",
        "endorsements": 7,
        "duration_months": 7
      },
      {
        "name": "Azure",
        "proficiency": "intermediate",
        "endorsements": 3,
        "duration_months": 36
      },
      {
        "name": "ETL",
        "proficiency": "beginner",
        "endorsements": 2,
        "duration_months": 8
      }
    ],
    "certifications": [],
    "languages": [
      {
        "language": "English",
        "proficiency": "professional"
      },
      {
        "language": "Hindi",
        "proficiency": "conversational"
      }
    ],
    "redrob_signals": {
      "profile_completeness_score": 74.8,
      "signup_date": "2023-02-12",
      "last_active_date": "2026-02-20",
      "open_to_work_flag": false,
      "profile_views_received_30d": 80,
      "applications_submitted_30d": 6,
      "recruiter_response_rate": 0.41,
      "avg_response_time_hours": 209.8,
      "skill_assessment_scores": {},
      "connection_count": 301,
      "endorsements_received": 29,
      "notice_period_days": 30,
      "expected_salary_range_inr_lpa": {
        "min": 7.6,
        "max": 28.0
      },
      "preferred_work_mode": "remote",
      "willing_to_relocate": true,
      "github_activity_score": 20.9,
      "search_appearance_30d": 166,
      "saved_by_recruiters_30d": 7,
      "interview_completion_rate": 0.4,
      "offer_acceptance_rate": 0.34,
      "verified_email": true,
      "verified_phone": true,
      "linkedin_connected": true
    }
  },
  {
    "candidate_id": "CAND_0000047",
    "profile": {
      "anonymized_name": "Avni Bansal",
      "headline": "Project Manager | Helping teams scale",
      "summary": "Professional with 2.4+ years of experience. My professional background is in marketing manager \u2014 I've built and led teams, owned KPIs, and driven business outcomes in this domain. Lately I've been curious about how AI tools could augment my work \u2014 I've experimented with ChatGPT and a few other tools for productivity and content creation, and I think the space is exciting. Open to roles where I can apply my domain expertise alongside emerging AI capabilities.",
      "location": "Kochi, Kerala",
      "country": "India",
      "years_of_experience": 2.4,
      "current_title": "Project Manager",
      "current_company": "TCS",
      "current_company_size": "10001+",
      "current_industry": "IT Services"
    },
    "career_history": [
      {
        "company": "TCS",
        "title": "Project Manager",
        "start_date": "2024-02-07",
        "end_date": null,
        "duration_months": 28,
        "is_current": true,
        "industry": "IT Services",
        "company_size": "10001+",
        "description": "Operations management role at a logistics company. Owned daily fulfillment operations across 3 warehouses, managing a team of 80 across receiving, picking, packing, and outbound. Built and tracked the operational KPIs (on-time fulfillment, accuracy, cost per order) and led the continuous improvement initiatives that drove a 22% productivity gain over 18 months."
      }
    ],
    "education": [
      {
        "institution": "Amity University",
        "degree": "B.Sc",
        "field_of_study": "Mechanical Engineering",
        "start_year": 2011,
        "end_year": 2016,
        "grade": "7.18 CGPA",
        "tier": "tier_3"
      }
    ],
    "skills": [
      {
        "name": "FastAPI",
        "proficiency": "beginner",
        "endorsements": 8,
        "duration_months": 4
      },
      {
        "name": "Java",
        "proficiency": "intermediate",
        "endorsements": 5,
        "duration_months": 28
      },
      {
        "name": "Excel",
        "proficiency": "intermediate",
        "endorsements": 10,
        "duration_months": 9
      },
      {
        "name": "Tally",
        "proficiency": "intermediate",
        "endorsements": 10,
        "duration_months": 9
      },
      {
        "name": "SQL",
        "proficiency": "intermediate",
        "endorsements": 11,
        "duration_months": 15
      },
      {
        "name": "Scrum",
        "proficiency": "intermediate",
        "endorsements": 1,
        "duration_months": 13
      },
      {
        "name": "Hadoop",
        "proficiency": "beginner",
        "endorsements": 10,
        "duration_months": 17
      }
    ],
    "certifications": [],
    "languages": [
      {
        "language": "English",
        "proficiency": "native"
      },
      {
        "language": "Hindi",
        "proficiency": "conversational"
      }
    ],
    "redrob_signals": {
      "profile_completeness_score": 79.7,
      "signup_date": "2025-06-07",
      "last_active_date": "2026-03-22",
      "open_to_work_flag": false,
      "profile_views_received_30d": 34,
      "applications_submitted_30d": 5,
      "recruiter_response_rate": 0.39,
      "avg_response_time_hours": 109.1,
      "skill_assessment_scores": {},
      "connection_count": 444,
      "endorsements_received": 48,
      "notice_period_days": 90,
      "expected_salary_range_inr_lpa": {
        "min": 13.3,
        "max": 19.2
      },
      "preferred_work_mode": "remote",
      "willing_to_relocate": false,
      "github_activity_score": -1,
      "search_appearance_30d": 14,
      "saved_by_recruiters_30d": 4,
      "interview_completion_rate": 0.54,
      "offer_acceptance_rate": 0.21,
      "verified_email": true,
      "verified_phone": true,
      "linkedin_connected": false
    }
  },
  {
    "candidate_id": "CAND_0000048",
    "profile": {
      "anonymized_name": "Vihaan Saxena",
      "headline": "Mobile Developer | Full-stack development",
      "summary": "Software engineer with 9.7 years of experience across web, backend, and cloud systems. Strong fundamentals in software development and system design. I've worked across web frontends, REST APIs, and cloud deployments; comfortable in most parts of a typical SaaS stack. I've been keeping up with AI/ML at a self-learner level \u2014 taken some online courses, played with the OpenAI and Anthropic APIs, built a small RAG side project \u2014 but I haven't done it in a professional capacity yet. Open to roles where I can either deepen my software engineering work or, if the team is open to it, start contributing to ML-adjacent systems.",
      "location": "Hyderabad, Telangana",
      "country": "India",
      "years_of_experience": 9.7,
      "current_title": "Mobile Developer",
      "current_company": "CRED",
      "current_company_size": "1001-5000",
      "current_industry": "Fintech"
    },
    "career_history": [
      {
        "company": "CRED",
        "title": "Mobile Developer",
        "start_date": "2024-02-07",
        "end_date": null,
        "duration_months": 28,
        "is_current": true,
        "industry": "Fintech",
        "company_size": "1001-5000",
        "description": "Frontend engineering at a media company. React, TypeScript, and the typical surrounding tooling (Webpack, Jest, Cypress). Built the company's design system from scratch and led the migration from a legacy AngularJS app. Strong on the frontend craft \u2014 accessibility, performance, animations \u2014 but limited backend exposure."
      },
      {
        "company": "Cognizant",
        "title": "Full Stack Developer",
        "start_date": "2022-06-17",
        "end_date": "2024-02-07",
        "duration_months": 20,
        "is_current": false,
        "industry": "IT Services",
        "company_size": "10001+",
        "description": "Android mobile development using Java and (more recently) Kotlin at a consumer-app company. Built and maintained multiple production features including the main shopping flow, push notification system, and the offline-first sync layer. Comfortable with the Android framework, Jetpack components, and the typical patterns (MVVM, Hilt, Coroutines). My career has been entirely on mobile so far; interested in expanding into broader backend or platform engineering."
      },
      {
        "company": "Stark Industries",
        "title": "Mobile Developer",
        "start_date": "2018-02-08",
        "end_date": "2022-04-18",
        "duration_months": 51,
        "is_current": false,
        "industry": "Manufacturing",
        "company_size": "1001-5000",
        "description": "Java backend development at a large enterprise \u2014 Spring Boot microservices, Kafka for inter-service messaging, Postgres + Redis for storage. Worked on the customer onboarding flow which involved orchestrating multiple downstream services. Solid on the Spring ecosystem, transaction handling, and the operational side of Java services. Looking to either go deeper on distributed systems or expand into modern application stacks."
      },
      {
        "company": "Initech",
        "title": "Full Stack Developer",
        "start_date": "2016-11-15",
        "end_date": "2018-02-08",
        "duration_months": 15,
        "is_current": false,
        "industry": "Software",
        "company_size": "51-200",
        "description": "Android mobile development using Java and (more recently) Kotlin at a consumer-app company. Built and maintained multiple production features including the main shopping flow, push notification system, and the offline-first sync layer. Comfortable with the Android framework, Jetpack components, and the typical patterns (MVVM, Hilt, Coroutines). My career has been entirely on mobile so far; interested in expanding into broader backend or platform engineering."
      }
    ],
    "education": [
      {
        "institution": "VIT Chennai",
        "degree": "B.E.",
        "field_of_study": "Data Science",
        "start_year": 2005,
        "end_year": 2009,
        "grade": "84%",
        "tier": "tier_3"
      },
      {
        "institution": "Anna University",
        "degree": "B.Tech",
        "field_of_study": "Statistics",
        "start_year": 2011,
        "end_year": 2016,
        "grade": "6.96 CGPA",
        "tier": "tier_2"
      }
    ],
    "skills": [
      {
        "name": "Hadoop",
        "proficiency": "beginner",
        "endorsements": 2,
        "duration_months": 15
      },
      {
        "name": "Terraform",
        "proficiency": "intermediate",
        "endorsements": 13,
        "duration_months": 26
      },
      {
        "name": "Vue.js",
        "proficiency": "intermediate",
        "endorsements": 2,
        "duration_months": 24
      },
      {
        "name": "Content Writing",
        "proficiency": "intermediate",
        "endorsements": 11,
        "duration_months": 26
      },
      {
        "name": "AWS",
        "proficiency": "intermediate",
        "endorsements": 1,
        "duration_months": 31
      }
    ],
    "certifications": [
      {
        "name": "Scrum Master Certified",
        "issuer": "Scrum Alliance",
        "year": 2023
      },
      {
        "name": "Six Sigma Green Belt",
        "issuer": "ASQ",
        "year": 2024
      }
    ],
    "languages": [
      {
        "language": "English",
        "proficiency": "native"
      },
      {
        "language": "Hindi",
        "proficiency": "conversational"
      }
    ],
    "redrob_signals": {
      "profile_completeness_score": 62.2,
      "signup_date": "2026-03-28",
      "last_active_date": "2026-04-06",
      "open_to_work_flag": true,
      "profile_views_received_30d": 58,
      "applications_submitted_30d": 3,
      "recruiter_response_rate": 0.65,
      "avg_response_time_hours": 97.9,
      "skill_assessment_scores": {},
      "connection_count": 225,
      "endorsements_received": 48,
      "notice_period_days": 120,
      "expected_salary_range_inr_lpa": {
        "min": 12.6,
        "max": 26.6
      },
      "preferred_work_mode": "flexible",
      "willing_to_relocate": false,
      "github_activity_score": -1,
      "search_appearance_30d": 131,
      "saved_by_recruiters_30d": 5,
      "interview_completion_rate": 0.42,
      "offer_acceptance_rate": 0.4,
      "verified_email": false,
      "verified_phone": true,
      "linkedin_connected": false
    }
  },
  {
    "candidate_id": "CAND_0000049",
    "profile": {
      "anonymized_name": "Tanya Chowdary",
      "headline": "Mechanical Engineer | Helping teams scale",
      "summary": "Professional with 11.8+ years of experience. I've spent my career in marketing manager roles, mostly focused on driving outcomes through process, people, and customer relationships. Lately I've been curious about how AI tools could augment my work \u2014 I've experimented with ChatGPT and a few other tools for productivity and content creation, and I think the space is exciting. Open to roles where I can apply my domain expertise alongside emerging AI capabilities.",
      "location": "Berlin",
      "country": "Germany",
      "years_of_experience": 11.8,
      "current_title": "Mechanical Engineer",
      "current_company": "Wayne Enterprises",
      "current_company_size": "10001+",
      "current_industry": "Conglomerate"
    },
    "career_history": [
      {
        "company": "Wayne Enterprises",
        "title": "Mechanical Engineer",
        "start_date": "2022-05-18",
        "end_date": null,
        "duration_months": 49,
        "is_current": true,
        "industry": "Conglomerate",
        "company_size": "10001+",
        "description": "Enterprise sales of cloud software solutions into the mid-market segment. Carried a $1.8M ARR quota and consistently delivered against it across the last three years. Owned the full sales cycle: prospecting, discovery, technical evaluation (with SE support), commercial negotiation, and close. Strong on consultative selling for technical buyers; comfortable engaging with both engineering and finance stakeholders."
      },
      {
        "company": "Hooli",
        "title": "Business Analyst",
        "start_date": "2017-11-10",
        "end_date": "2022-04-18",
        "duration_months": 54,
        "is_current": false,
        "industry": "Software",
        "company_size": "1001-5000",
        "description": "Senior accounting role at a mid-sized company \u2014 month-end close, financial reporting, statutory compliance (GAAP / Ind-AS), and tax filings. Owned the GL, fixed-asset register, and the audit-readiness function. Managed a team of 3 staff accountants. Built strong process discipline around the close cycle, reducing close time from 12 days to 7 over the last two years."
      },
      {
        "company": "Wayne Enterprises",
        "title": "Sales Executive",
        "start_date": "2014-09-27",
        "end_date": "2017-11-10",
        "duration_months": 38,
        "is_current": false,
        "industry": "Conglomerate",
        "company_size": "10001+",
        "description": "Operations management role at a logistics company. Owned daily fulfillment operations across 3 warehouses, managing a team of 80 across receiving, picking, packing, and outbound. Built and tracked the operational KPIs (on-time fulfillment, accuracy, cost per order) and led the continuous improvement initiatives that drove a 22% productivity gain over 18 months."
      }
    ],
    "education": [
      {
        "institution": "Symbiosis International",
        "degree": "M.Tech",
        "field_of_study": "Mathematics",
        "start_year": 2016,
        "end_year": 2019,
        "grade": "7.32 CGPA",
        "tier": "tier_3"
      }
    ],
    "skills": [
      {
        "name": "TypeScript",
        "proficiency": "beginner",
        "endorsements": 0,
        "duration_months": 8
      },
      {
        "name": "Rust",
        "proficiency": "intermediate",
        "endorsements": 1,
        "duration_months": 16
      },
      {
        "name": "Data Pipelines",
        "proficiency": "beginner",
        "endorsements": 6,
        "duration_months": 11
      },
      {
        "name": "Apache Beam",
        "proficiency": "beginner",
        "endorsements": 10,
        "duration_months": 4
      },
      {
        "name": "GraphQL",
        "proficiency": "intermediate",
        "endorsements": 14,
        "duration_months": 16
      },
      {
        "name": "Kubernetes",
        "proficiency": "intermediate",
        "endorsements": 5,
        "duration_months": 13
      }
    ],
    "certifications": [],
    "languages": [
      {
        "language": "English",
        "proficiency": "native"
      },
      {
        "language": "Hindi",
        "proficiency": "professional"
      }
    ],
    "redrob_signals": {
      "profile_completeness_score": 64.0,
      "signup_date": "2025-09-12",
      "last_active_date": "2026-05-13",
      "open_to_work_flag": false,
      "profile_views_received_30d": 6,
      "applications_submitted_30d": 2,
      "recruiter_response_rate": 0.59,
      "avg_response_time_hours": 109.3,
      "skill_assessment_scores": {},
      "connection_count": 514,
      "endorsements_received": 12,
      "notice_period_days": 120,
      "expected_salary_range_inr_lpa": {
        "min": 12.9,
        "max": 18.8
      },
      "preferred_work_mode": "onsite",
      "willing_to_relocate": false,
      "github_activity_score": -1,
      "search_appearance_30d": 100,
      "saved_by_recruiters_30d": 6,
      "interview_completion_rate": 0.56,
      "offer_acceptance_rate": -1,
      "verified_email": true,
      "verified_phone": false,
      "linkedin_connected": true
    }
  },
  {
    "candidate_id": "CAND_0000050",
    "profile": {
      "anonymized_name": "Naina Bose",
      "headline": "Business Analyst | Helping teams scale",
      "summary": "Professional with 13.5+ years of experience. I'm a marketing manager with substantial experience in cross-functional collaboration, stakeholder management, and execution. Lately I've been curious about how AI tools could augment my work \u2014 I've experimented with ChatGPT and a few other tools for productivity and content creation, and I think the space is exciting. Open to roles where I can apply my domain expertise alongside emerging AI capabilities.",
      "location": "Gurgaon, Haryana",
      "country": "India",
      "years_of_experience": 13.5,
      "current_title": "Business Analyst",
      "current_company": "Infosys",
      "current_company_size": "10001+",
      "current_industry": "IT Services"
    },
    "career_history": [
      {
        "company": "Infosys",
        "title": "Business Analyst",
        "start_date": "2022-09-15",
        "end_date": null,
        "duration_months": 45,
        "is_current": true,
        "industry": "IT Services",
        "company_size": "10001+",
        "description": "Mechanical engineering design role at a hardware-product company. Led the design of two product subsystems through full lifecycle: concept, DFM/DFMA review, prototype, production tooling. Comfortable with CAD (SolidWorks, Creo), FEA (ANSYS), and the typical hardware-development cadence. Worked closely with manufacturing partners on production scale-up."
      },
      {
        "company": "TCS",
        "title": "Marketing Manager",
        "start_date": "2019-08-02",
        "end_date": "2022-07-17",
        "duration_months": 36,
        "is_current": false,
        "industry": "IT Services",
        "company_size": "10001+",
        "description": "Operations management role at a logistics company. Owned daily fulfillment operations across 3 warehouses, managing a team of 80 across receiving, picking, packing, and outbound. Built and tracked the operational KPIs (on-time fulfillment, accuracy, cost per order) and led the continuous improvement initiatives that drove a 22% productivity gain over 18 months."
      },
      {
        "company": "Hooli",
        "title": "HR Manager",
        "start_date": "2017-03-15",
        "end_date": "2019-07-03",
        "duration_months": 28,
        "is_current": false,
        "industry": "Software",
        "company_size": "1001-5000",
        "description": "Senior accounting role at a mid-sized company \u2014 month-end close, financial reporting, statutory compliance (GAAP / Ind-AS), and tax filings. Owned the GL, fixed-asset register, and the audit-readiness function. Managed a team of 3 staff accountants. Built strong process discipline around the close cycle, reducing close time from 12 days to 7 over the last two years."
      },
      {
        "company": "Acme Corp",
        "title": "Content Writer",
        "start_date": "2012-12-22",
        "end_date": "2017-03-01",
        "duration_months": 51,
        "is_current": false,
        "industry": "Manufacturing",
        "company_size": "201-500",
        "description": "Senior accounting role at a mid-sized company \u2014 month-end close, financial reporting, statutory compliance (GAAP / Ind-AS), and tax filings. Owned the GL, fixed-asset register, and the audit-readiness function. Managed a team of 3 staff accountants. Built strong process discipline around the close cycle, reducing close time from 12 days to 7 over the last two years."
      }
    ],
    "education": [
      {
        "institution": "VIT Chennai",
        "degree": "Ph.D",
        "field_of_study": "Artificial Intelligence",
        "start_year": 2001,
        "end_year": 2005,
        "grade": "6.66 CGPA",
        "tier": "tier_3"
      },
      {
        "institution": "Georgia Tech",
        "degree": "B.Sc",
        "field_of_study": "Artificial Intelligence",
        "start_year": 2013,
        "end_year": 2017,
        "grade": "7.81 CGPA",
        "tier": "tier_1"
      }
    ],
    "skills": [
      {
        "name": "gRPC",
        "proficiency": "intermediate",
        "endorsements": 9,
        "duration_months": 11
      },
      {
        "name": "SEO",
        "proficiency": "beginner",
        "endorsements": 8,
        "duration_months": 18
      },
      {
        "name": "Feature Engineering",
        "proficiency": "advanced",
        "endorsements": 4,
        "duration_months": 42
      },
      {
        "name": "Marketing",
        "proficiency": "beginner",
        "endorsements": 0,
        "duration_months": 15
      },
      {
        "name": "Data Pipelines",
        "proficiency": "beginner",
        "endorsements": 0,
        "duration_months": 15
      },
      {
        "name": "Kafka",
        "proficiency": "beginner",
        "endorsements": 7,
        "duration_months": 13
      },
      {
        "name": "Excel",
        "proficiency": "intermediate",
        "endorsements": 10,
        "duration_months": 23
      }
    ],
    "certifications": [],
    "languages": [
      {
        "language": "English",
        "proficiency": "professional"
      },
      {
        "language": "Hindi",
        "proficiency": "conversational"
      }
    ],
    "redrob_signals": {
      "profile_completeness_score": 42.5,
      "signup_date": "2023-01-23",
      "last_active_date": "2025-10-22",
      "open_to_work_flag": false,
      "profile_views_received_30d": 34,
      "applications_submitted_30d": 2,
      "recruiter_response_rate": 0.42,
      "avg_response_time_hours": 108.7,
      "skill_assessment_scores": {
        "Feature Engineering": 60.8
      },
      "connection_count": 245,
      "endorsements_received": 22,
      "notice_period_days": 90,
      "expected_salary_range_inr_lpa": {
        "min": 7.6,
        "max": 22.9
      },
      "preferred_work_mode": "onsite",
      "willing_to_relocate": false,
      "github_activity_score": 44.7,
      "search_appearance_30d": 87,
      "saved_by_recruiters_30d": 2,
      "interview_completion_rate": 0.58,
      "offer_acceptance_rate": -1,
      "verified_email": true,
      "verified_phone": true,
      "linkedin_connected": false
    }
  }
]


## Step 3 - Run the ranker
One command. The `--sample` flag reads a JSON array; the built-in TF-IDF semantic layer needs no installs.


In [ ]:
!python rank.py --candidates sample_candidates.json --out submission.csv --sample


## Step 4 - View the ranked output


In [ ]:
import csv
with open('submission.csv', encoding='utf-8') as f:
    rows = list(csv.DictReader(f))
print(f'{len(rows)} candidates ranked. Top 10:')
print('-'*100)
for r in rows[:10]:
    print(f"#{r['rank']:>2}  {r['candidate_id']}  score={r['score']}  {r['reasoning'][:70]}")


## Optional - Run on the FULL 100,000-candidate pool
Upload `candidates.jsonl` from the hackathon bundle, then run the cell below. Produces exactly 100 rows and
passes `validate_submission.py`. Takes ~40s on Colab CPU.



In [ ]:
# from google.colab import files
# up = files.upload()                      # choose candidates.jsonl
# fname = next(iter(up))
# !python rank.py --candidates "$fname" --out submission_full.csv
# !python validate_submission.py submission_full.csv
# from google.colab import files as _f; _f.download('submission_full.csv')


## Download your submission CSV


In [ ]:
try:
    from google.colab import files
    files.download('submission.csv')
except Exception as e:
    print('Download available in Colab. File saved as submission.csv')
